In [ ]:
import pandas as pd

# Load validation dataset
val_df = pd.read_csv("validation.csv")

print("Validation records:", len(val_df))
print("Columns:", val_df.columns.tolist())

print("\nMissing transcriptions:",
      val_df["transcription"].isna().sum())

print("Duplicate transcriptions:",
      val_df.duplicated(subset=["transcription"]).sum())

# Display first 3 records
display(
    val_df[
        ["case_id", "medical_specialty",
         "sample_name", "word_count"]
    ].head(3)
)


In [ ]:
!pip install -q --upgrade openai

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Read API key securely from Colab Secrets
api_key = userdata.get("OPENAI_API_KEY")

# Create API client
client = OpenAI(api_key=api_key)

# Choose the model for the baseline experiment
MODEL = "gpt-3.5-turbo"

print("API client initialized.")


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: API connection successful"
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)


In [ ]:
# Select the first record from the validation dataset
case = val_df.iloc[0]

case_id = case["case_id"]
specialty = case["medical_specialty"]
sample_name = case["sample_name"]
transcription = case["transcription"]

print("Case ID:", case_id)
print("Specialty:", specialty)
print("Sample:", sample_name)

print("\nOriginal transcription:\n")
print(transcription)


In [ ]:
# Simple baseline prompt
baseline_prompt = """Write a clinical note from this transcription.

Transcription:
{transcription}"""

print("Baseline prompt ready.")

In [ ]:
# Insert the selected transcription into the prompt
prompt = baseline_prompt.format(
    transcription=transcription
)

# Request clinical note generation
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0,
    max_tokens=1200
)

# Extract the generated note
baseline_note = response.choices[0].message.content

print("=== BASELINE GENERATED CLINICAL NOTE ===\n")
print(baseline_note)

In [ ]:
import pandas as pd

# Save the source transcription and generated note
baseline_result = pd.DataFrame([{
    "case_id": case_id,
    "medical_specialty": specialty,
    "sample_name": sample_name,
    "transcription": transcription,
    "baseline_note": baseline_note,
    "model": MODEL
}])

baseline_result.to_csv(
    "baseline_test_1_case.csv",
    index=False
)

print("Baseline result saved successfully!")
print("Number of saved cases:", len(baseline_result))

In [ ]:
print("Model:", response.model)

print("\nToken usage:")
print("Input tokens:", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

print("\nFinish reason:")
print(response.choices[0].finish_reason)

In [ ]:
# Add generation metadata
baseline_result["input_tokens"] = (
    response.usage.prompt_tokens
)

baseline_result["output_tokens"] = (
    response.usage.completion_tokens
)

baseline_result["total_tokens"] = (
    response.usage.total_tokens
)

baseline_result["finish_reason"] = (
    response.choices[0].finish_reason
)

baseline_result["temperature"] = 0
baseline_result["max_tokens"] = 1200

# Save updated results
baseline_result.to_csv(
    "baseline_test_1_case.csv",
    index=False
)

print("Updated baseline CSV saved successfully.")

display(
    baseline_result[
        [
            "case_id",
            "model",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "finish_reason"
        ]
    ]
)

In [ ]:
structured_prompt = """
You are a clinical documentation assistant.

Step 1: Extract key clinical facts from the transcription:

- Symptoms
- Negations
- Duration
- Medical history
- Medications
- Tests
- Diagnosis
- Treatment plan

Step 2: Write a structured clinical note using only these facts.

Use sections:

- Chief Complaint
- History
- Findings
- Assessment
- Plan

Do not invent information.

Transcription:
{transcription}
"""

print("Structured prompt ready.")

In [ ]:
# Use the same transcription as the baseline
prompt = structured_prompt.format(
    transcription=transcription
)

structured_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0,
    max_tokens=1200
)

structured_note = (
    structured_response.choices[0].message.content
)

print("=== STRUCTURED CLINICAL NOTE ===\n")
print(structured_note)

print("\nFinish reason:",
      structured_response.choices[0].finish_reason)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "case_id": case_id,
        "method": "Simple Baseline",
        "model": MODEL,
        "transcription": transcription,
        "generated_note": baseline_note
    },
    {
        "case_id": case_id,
        "method": "Structured Prompt",
        "model": MODEL,
        "transcription": transcription,
        "generated_note": structured_note
    }
])

comparison_df.to_csv(
    "baseline_vs_structured_1_case.csv",
    index=False
)

print("Comparison CSV saved successfully!")

display(
    comparison_df[
        ["case_id", "method", "model"]
    ]
)

In [ ]:
audit_prompt = """
You are a clinical documentation auditor.

Compare the original transcription and generated clinical note.

Identify:

1. Missing clinically relevant information
2. Hallucinations or unsupported statements
3. Redundant information
4. Potential clinical safety issues

Pay particular attention to:
- Negated symptoms and diseases
- Medication names, doses, and frequencies
- Allergies and reactions
- Vital signs and test results
- Diagnoses and treatment plans

Return only valid JSON using this structure:

{{
    "missing_information": [],
    "hallucinations": [],
    "redundancy": [],
    "safety_issues": [],
    "summary": ""
}}

Original transcription:
{transcription}

Generated note:
{note}
"""

print("Audit prompt ready.")

In [ ]:
import json
import pandas as pd

audit_results = []

for _, row in comparison_df.iterrows():

    print("Auditing:", row["method"])

    prompt = audit_prompt.format(
        transcription=row["transcription"],
        note=row["generated_note"]
    )

    audit_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=1500,
        response_format={"type": "json_object"}
    )

    audit = json.loads(
        audit_response.choices[0].message.content
    )

    audit_results.append({
        "case_id": row["case_id"],
        "method": row["method"],
        "model": MODEL,
        "audit": audit,
        "finish_reason": audit_response.choices[0].finish_reason
    })

    print("Finish reason:",
          audit_response.choices[0].finish_reason)

print("\nBoth audits completed.")

In [ ]:
for result in audit_results:

    print("\n" + "=" * 50)
    print("METHOD:", result["method"])
    print("=" * 50)

    print(json.dumps(
        result["audit"],
        indent=2,
        ensure_ascii=False
    ))

# Save full audit results
with open(
    "baseline_vs_structured_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nAudit JSON saved successfully.")

In [ ]:
improved_audit_prompt = """
You are a clinical documentation quality auditor.

Your task is to compare a generated clinical note
against its original transcription.

IMPORTANT RULES:
1. Use the transcription as the only source of truth.
2. Do not assume that a fluent note is complete.
3. Identify missing clinically relevant facts explicitly.
4. Pay special attention to negated symptoms and diseases.
5. Check medication doses, frequencies, allergies,
   vital signs, diagnoses, tests and treatment plans.
6. Do not invent clinical findings or recommendations.
7. Report unsupported statements separately.
8. Provide exact supporting excerpts from the source.
9. If no problem is found, return an empty list.

Return valid JSON using this structure:

{{
  "missing_information": [
    {{
      "fact": "Specific missing clinical fact",
      "source_evidence": "Exact excerpt from transcription"
    }}
  ],
  "hallucinations": [
    {{
      "statement": "Unsupported statement from generated note",
      "reason": "Why it is unsupported"
    }}
  ],
  "negation_errors": [
    {{
      "fact": "Incorrectly represented or omitted negation",
      "source_evidence": "Exact excerpt"
    }}
  ],
  "safety_issues": [
    {{
      "issue": "Potential safety concern",
      "source_evidence": "Relevant excerpt, if available"
    }}
  ],
  "summary": "Brief summary of the audit"
}}

Original transcription:
{transcription}

Generated clinical note:
{note}
"""

print("Improved audit prompt ready.")

In [ ]:
import json

improved_audit_results = []

for _, row in comparison_df.iterrows():

    print("Auditing:", row["method"])

    prompt = improved_audit_prompt.format(
        transcription=row["transcription"],
        note=row["generated_note"]
    )

    audit_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=2000,
        response_format={"type": "json_object"}
    )

    audit = json.loads(
        audit_response.choices[0].message.content
    )

    improved_audit_results.append({
        "case_id": row["case_id"],
        "method": row["method"],
        "audit": audit,
        "finish_reason": audit_response.choices[0].finish_reason
    })

    print("Finish reason:",
          audit_response.choices[0].finish_reason)

print("\nImproved audits completed.")

In [ ]:
with open(
    "improved_clinical_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        improved_audit_results,
        f,
        indent=2,
        ensure_ascii=False
    )

for result in improved_audit_results:

    audit = result["audit"]

    print("\nMethod:", result["method"])
    print("Missing facts:",
          len(audit.get("missing_information", [])))
    print("Hallucinations:",
          len(audit.get("hallucinations", [])))
    print("Negation errors:",
          len(audit.get("negation_errors", [])))
    print("Safety issues:",
          len(audit.get("safety_issues", [])))
    print("Finish reason:", result["finish_reason"])

print("\nImproved audit JSON saved.")

In [ ]:
import json

fact_extraction_prompt = """
Extract clinically relevant atomic facts from the transcription.

An atomic fact should describe one clinical finding,
condition, medication detail, negation, test result,
or treatment instruction.

Rules:
1. Use only information explicitly in the transcription.
2. Preserve important numbers, medication doses and units.
3. Preserve negations.
4. Separate distinct clinical facts.
5. Do not add recommendations.
6. Include an exact source excerpt for every fact.

Return JSON:
{
  "facts": [
    {
      "category": "history",
      "fact": "One clinical fact",
      "source_evidence": "Exact source excerpt"
    }
  ]
}

Transcription:
""" + transcription

fact_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": fact_extraction_prompt}
    ],
    temperature=0,
    max_tokens=3000,
    response_format={"type": "json_object"}
)

print("Finish reason:",
      fact_response.choices[0].finish_reason)

if fact_response.choices[0].finish_reason != "stop":
    raise RuntimeError("Fact extraction may be truncated.")

extracted_facts = json.loads(
    fact_response.choices[0].message.content
)

print("Facts extracted:",
      len(extracted_facts["facts"]))

print(json.dumps(
    extracted_facts["facts"][:5],
    indent=2
))

In [ ]:
with open(
    "case_0014_extracted_facts.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "model": MODEL,
            "extraction": extracted_facts
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Extracted facts saved successfully.")

In [ ]:
import json

categories = [
    "Medical history, including past surgeries and denied conditions",
    "Current medications, exact dosages, frequencies, allergies and reactions",
    "Vital signs and physical examination findings"
]

category_results = {}

for category in categories:

    print("\nExtracting:", category)

    prompt = f"""
You are a clinical information extraction assistant.

Extract EVERY explicitly documented fact related to:

{category}

Rules:
1. Use the transcription as the only source.
2. Extract one clinical fact per item.
3. Preserve exact numerical values and units.
4. Preserve important negations.
5. Do not infer missing details.
6. Include an exact supporting excerpt for each fact.
7. If a fact is not documented, do not invent it.

Return valid JSON:

{{
  "facts": [
    {{
      "fact": "One clinical fact",
      "source_evidence": "Exact excerpt from transcription"
    }}
  ]
}}

Transcription:
{transcription}
"""

    result = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2500,
        response_format={"type": "json_object"}
    )

    finish_reason = result.choices[0].finish_reason

    if finish_reason != "stop":
        raise RuntimeError(
            f"Incomplete extraction for {category}: {finish_reason}"
        )

    extracted = json.loads(
        result.choices[0].message.content
    )

    category_results[category] = extracted["facts"]

    print("Facts extracted:", len(extracted["facts"]))

print("\nCategory extraction completed.")

In [ ]:
with open(
    "case_0014_category_facts.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "model": MODEL,
            "category_results": category_results
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Category extraction results saved.")

for category, facts in category_results.items():
    print("\nCategory:", category)
    print("Number of facts:", len(facts))

In [ ]:
# Display the complete transcription
print("Case ID:", case_id)
print("Sample:", sample_name)

print("\n" + "=" * 60)
print("ORIGINAL TRANSCRIPTION")
print("=" * 60)

print(transcription)

print("\n" + "=" * 60)
print("Total words:", len(transcription.split()))
print("Total characters:", len(transcription))

In [ ]:
import re

# Section headings found in CASE_0014
section_headings = [
    "PAST MEDICAL HISTORY",
    "PAST SURGICAL HISTORY",
    "MEDICATIONS",
    "ALLERGIES TO MEDICATIONS",
    "SOCIAL HISTORY",
    "FAMILY HISTORY",
    "REVIEW OF SYSTEMS",
    "PHYSICAL EXAMINATION",
    "ASSESSMENT",
    "PLAN"
]

# Match longer headings first to avoid partial matches
ordered_headings = sorted(
    section_headings,
    key=len,
    reverse=True
)

pattern = re.compile(
    r"(?<![A-Za-z])("
    + "|".join(re.escape(h) for h in ordered_headings)
    + r")\s*:\s*,?",
    flags=re.IGNORECASE
)

matches = list(pattern.finditer(transcription))

sections = {}

# Preserve any text preceding the first heading
if matches and transcription[:matches[0].start()].strip(" ,"):
    sections["UNLABELED"] = (
        transcription[:matches[0].start()].strip(" ,")
    )

for i, match in enumerate(matches):
    heading = match.group(1).upper()

    start = match.end()

    end = (
        matches[i + 1].start()
        if i + 1 < len(matches)
        else len(transcription)
    )

    content = transcription[start:end].strip(" ,\n")

    sections[heading] = content

print("Sections detected:", len(sections))

for heading, content in sections.items():
    print("\n" + "=" * 55)
    print(heading)
    print("=" * 55)
    print(content)

In [ ]:
def normalize_for_check(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip(" ,\n")

# Remove section headings from original transcription
original_body = pattern.sub(" ", transcription)

# Join extracted section contents
reconstructed_body = " ".join(sections.values())

print(
    "Original body characters:",
    len(normalize_for_check(original_body))
)

print(
    "Extracted body characters:",
    len(normalize_for_check(reconstructed_body))
)

print(
    "Exact normalized text match:",
    normalize_for_check(original_body)
    == normalize_for_check(reconstructed_body)
)

In [ ]:
# Check whether each extracted section matches
# its corresponding original text segment

all_sections_match = True

for i, match in enumerate(matches):

    heading = match.group(1).upper()

    start = match.end()

    end = (
        matches[i + 1].start()
        if i + 1 < len(matches)
        else len(transcription)
    )

    original_section = transcription[start:end].strip(" ,\n")
    extracted_section = sections[heading]

    is_match = original_section == extracted_section

    print(f"{heading}: {is_match}")

    if not is_match:
        all_sections_match = False

print("\nAll sections match:", all_sections_match)

In [ ]:
import json

section_facts = {}

for heading, section_text in sections.items():

    print("Processing:", heading)

    prompt = f"""
You are a clinical information extraction assistant.

Extract all clinically relevant facts from the
following medical transcription section.

SECTION: {heading}

Rules:
1. Extract one clinical fact per item.
2. Preserve all medication names, doses, and frequencies.
3. Preserve exact numerical findings and units.
4. Preserve negative findings and denied conditions.
5. Do not add information not present in the section.
6. Include exact source evidence for each fact.
7. Do not combine unrelated facts.
8. If no clinical facts exist, return an empty list.

Return valid JSON:

{{
    "facts": [
        {{
            "fact": "One clinical fact",
            "source_evidence": "Exact excerpt from section"
        }}
    ]
}}

Section text:
{section_text}
"""

    response_section = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2000,
        response_format={"type": "json_object"}
    )

    finish_reason = response_section.choices[0].finish_reason

    if finish_reason != "stop":
        raise RuntimeError(
            f"Incomplete response for {heading}: {finish_reason}"
        )

    result = json.loads(
        response_section.choices[0].message.content
    )

    section_facts[heading] = result["facts"]

    print("Facts extracted:", len(result["facts"]))
    print("-" * 40)

print("\nAll sections processed.")

In [ ]:
# Save extracted facts from all sections
with open(
    "case_0014_section_facts.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "model": MODEL,
            "section_facts": section_facts
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Section facts saved successfully!")

print("\n=== EXTRACTION SUMMARY ===")

total_facts = 0

for heading, facts in section_facts.items():
    count = len(facts)
    total_facts += count
    print(f"{heading}: {count}")

print("\nTotal extracted facts:", total_facts)

In [ ]:
import json

heading = "PLAN"

original_section = sections[heading]
existing_facts = section_facts[heading]

refinement_prompt = f"""
You are checking clinical fact extraction for omissions.

ORIGINAL PLAN SECTION:
{original_section}

ALREADY EXTRACTED FACTS:
{json.dumps(existing_facts, indent=2)}

Task:
Identify clinically relevant facts from the original
Plan section that are NOT captured by the existing facts.

Rules:
1. Extract only additional missing facts.
2. Do not repeat existing facts.
3. Preserve all tests, consultations, treatments,
   durations, and numerical values.
4. Do not invent information.
5. Include an exact source excerpt for each new fact.
6. If nothing is missing, return an empty list.

Return valid JSON:
{{
    "missing_facts": [
        {{
            "fact": "One missing clinical fact",
            "source_evidence": "Exact source excerpt"
        }}
    ]
}}
"""

refinement_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": refinement_prompt}
    ],
    temperature=0,
    max_tokens=1200,
    response_format={"type": "json_object"}
)

finish_reason = refinement_response.choices[0].finish_reason
print("Finish reason:", finish_reason)

if finish_reason != "stop":
    raise RuntimeError("Refinement response may be incomplete.")

plan_refinement = json.loads(
    refinement_response.choices[0].message.content
)

missing_facts = plan_refinement["missing_facts"]

print("\nAdditional Plan facts found:", len(missing_facts))

for i, item in enumerate(missing_facts, 1):
    print(f"\n{i}. {item['fact']}")
    print("Evidence:", item["source_evidence"])

In [ ]:
# Verify each source excerpt occurs in the original Plan
for item in missing_facts:
    evidence = item["source_evidence"]

    item["evidence_verified"] = (
        isinstance(evidence, str)
        and bool(evidence.strip())
        and evidence in original_section
    )

print("=== EVIDENCE CHECK ===")

for item in missing_facts:
    print(
        item["fact"],
        "| Evidence found:",
        item["evidence_verified"]
    )

# Save the proposed corrections for review
with open(
    "case_0014_plan_refinement.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "section": "PLAN",
            "existing_facts": existing_facts,
            "proposed_missing_facts": missing_facts
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nPlan refinement saved.")

In [ ]:
# Correct the two evidence excerpts using the original text
missing_facts[1]["source_evidence"] = (
    "an upper GI series"
)

missing_facts[2]["source_evidence"] = (
    "consultations with the dietician and the "
    "psychologist preoperatively"
)

# Verify all excerpts
for item in missing_facts:
    item["evidence_verified"] = (
        item["source_evidence"] in sections["PLAN"]
    )

    print("Fact:", item["fact"])
    print("Evidence verified:", item["evidence_verified"])
    print()

In [ ]:
# Keep the original extracted Plan fact
refined_plan_facts = list(section_facts["PLAN"])

# Add the three verified missing facts
for item in missing_facts:
    if item["evidence_verified"]:
        refined_plan_facts.append({
            "fact": item["fact"],
            "source_evidence": item["source_evidence"]
        })

print("Original Plan facts:", len(section_facts["PLAN"]))
print("Refined Plan facts:", len(refined_plan_facts))

for i, fact in enumerate(refined_plan_facts, 1):
    print(f"\n{i}. {fact['fact']}")

In [ ]:
import json

heading = "PHYSICAL EXAMINATION"

original_exam = sections[heading]
existing_exam_facts = section_facts[heading]

exam_refinement_prompt = f"""
You are a clinical information extraction auditor.

Compare the original physical examination section
against the already extracted facts.

ORIGINAL PHYSICAL EXAMINATION:
{original_exam}

ALREADY EXTRACTED FACTS:
{json.dumps(existing_exam_facts, indent=2)}

TASK:
Identify examination findings that have NOT been
captured in the existing facts.

Rules:
1. Extract only missing findings.
2. Do not repeat already extracted facts.
3. Preserve exact measurements and numerical values.
4. Preserve important negative findings.
5. Do not add findings not present in the source.
6. Include an exact source excerpt for each missing fact.
7. Separate distinct clinical findings.

Return valid JSON:

{{
    "missing_facts": [
        {{
            "fact": "One missing examination finding",
            "source_evidence": "Exact source excerpt"
        }}
    ]
}}
"""

exam_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": exam_refinement_prompt
        }
    ],
    temperature=0,
    max_tokens=1800,
    response_format={"type": "json_object"}
)

exam_finish_reason = exam_response.choices[0].finish_reason

print("Finish reason:", exam_finish_reason)

if exam_finish_reason != "stop":
    raise RuntimeError("Examination refinement may be truncated.")

exam_refinement = json.loads(
    exam_response.choices[0].message.content
)

exam_missing_facts = exam_refinement["missing_facts"]

print("\nAdditional examination facts found:",
      len(exam_missing_facts))

for i, item in enumerate(exam_missing_facts, 1):
    print(f"\n{i}. {item['fact']}")
    print("Evidence:", item["source_evidence"])

In [ ]:
# Check whether each quoted excerpt appears
# in the original physical examination section
for item in exam_missing_facts:

    evidence = item.get("source_evidence", "")

    item["evidence_verified"] = (
        isinstance(evidence, str)
        and bool(evidence.strip())
        and evidence in original_exam
    )

print("=== EXAMINATION EVIDENCE CHECK ===")

for item in exam_missing_facts:
    print("\nFact:", item["fact"])
    print("Evidence verified:", item["evidence_verified"])

# Save without changing the original facts
with open(
    "case_0014_exam_refinement.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "section": heading,
            "existing_facts": existing_exam_facts,
            "proposed_missing_facts": exam_missing_facts
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nExamination refinement saved.")

In [ ]:
# Begin with the original 14 examination facts
refined_exam_facts = [
    item.copy() for item in section_facts["PHYSICAL EXAMINATION"]
]

# Add the 5 verified refinement facts
for item in exam_missing_facts:
    if item["evidence_verified"]:
        refined_exam_facts.append({
            "fact": item["fact"],
            "source_evidence": item["source_evidence"]
        })

# Add findings missed by both extraction attempts
manual_exam_facts = [
    {
        "fact": "Thyroid nonpalpable and trachea midline",
        "source_evidence": "Neck exam nonpalpable thyroid, midline trachea"
    },
    {
        "fact": "No carotid bruit",
        "source_evidence": "no carotid bruit"
    },
    {
        "fact": "Abdomen soft and nontender, with no palpable tenderness",
        "source_evidence": "soft, nontender, obese, nondistended abdomen. No palpable tenderness."
    }
]

# Verify source excerpts
for item in manual_exam_facts:
    assert item["source_evidence"] in sections["PHYSICAL EXAMINATION"]

refined_exam_facts.extend(manual_exam_facts)

print("Original examination facts:",
      len(section_facts["PHYSICAL EXAMINATION"]))

print("Verified refinement facts:",
      len(exam_missing_facts))

print("Manual additions:",
      len(manual_exam_facts))

print("Total refined examination facts:",
      len(refined_exam_facts))

In [ ]:
import json

# Create a separate copy of the extracted facts
final_section_facts = {
    heading: [item.copy() for item in facts]
    for heading, facts in section_facts.items()
}

# Replace the two refined sections
final_section_facts["PLAN"] = refined_plan_facts
final_section_facts["PHYSICAL EXAMINATION"] = refined_exam_facts

# Count facts
total_facts = sum(
    len(facts) for facts in final_section_facts.values()
)

print("=== REFINED FACT EXTRACTION SUMMARY ===")

for heading, facts in final_section_facts.items():
    print(f"{heading}: {len(facts)}")

print("\nTotal facts:", total_facts)

# Save the combined fact list
with open(
    "case_0014_refined_facts.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "model": MODEL,
            "section_facts": final_section_facts
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nRefined facts saved successfully.")

In [ ]:
import pandas as pd

fact_rows = []
fact_number = 1

for section, facts in final_section_facts.items():
    for item in facts:
        fact_rows.append({
            "fact_id": f"F{fact_number:03d}",
            "section": section,
            "fact": item["fact"],
            "source_evidence": item["source_evidence"]
        })
        fact_number += 1

fact_checklist = pd.DataFrame(fact_rows)

print("Total checklist entries:", len(fact_checklist))
print("Unique fact IDs:",
      fact_checklist["fact_id"].nunique())

print("\nFacts per section:")
print(fact_checklist["section"].value_counts())

display(fact_checklist.head(10))

fact_checklist.to_csv(
    "case_0014_fact_checklist.csv",
    index=False
)

print("\nFact checklist saved successfully.")

In [ ]:
import json

# Select first 10 facts for testing
test_facts = fact_checklist.head(10).to_dict(
    orient="records"
)

print("Facts selected:", len(test_facts))

for item in test_facts:
    print(
        item["fact_id"],
        "|",
        item["fact"]
    )

In [ ]:
fact_comparison_results = []

for _, row in comparison_df.iterrows():

    print("\nEvaluating:", row["method"])

    prompt = f"""
You are evaluating clinical fact preservation.

Compare each reference fact with the generated note.

Classification rules:

present:
The complete clinical meaning is accurately documented.

partial:
The note contains part of the fact but omits
an important detail.

missing:
The fact is absent from the note.

contradicted:
The note explicitly contradicts the reference fact.

Important:
- Preserve negation.
- Check medication dosage and frequency.
- Check numerical values.
- Do not count a vague statement as full preservation.
- Use only the provided fact checklist and note.
- Evaluate every fact ID exactly once.

Return valid JSON:

{{
  "results": [
    {{
      "fact_id": "F001",
      "status": "present",
      "note_evidence": "Exact excerpt from generated note",
      "reason": "Brief explanation"
    }}
  ]
}}

REFERENCE FACTS:
{json.dumps(test_facts, ensure_ascii=False)}

GENERATED NOTE:
{row["generated_note"]}
"""

    evaluation_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2500,
        response_format={"type": "json_object"}
    )

    finish_reason = (
        evaluation_response.choices[0].finish_reason
    )

    if finish_reason != "stop":
        raise RuntimeError(
            f"Evaluation incomplete: {finish_reason}"
        )

    evaluation = json.loads(
        evaluation_response.choices[0].message.content
    )

    # Validate that all 10 fact IDs were evaluated
    expected_ids = {
        item["fact_id"] for item in test_facts
    }

    returned_ids = [
        item["fact_id"]
        for item in evaluation["results"]
    ]

    assert len(returned_ids) == len(expected_ids)
    assert set(returned_ids) == expected_ids
    assert all(
        item["status"] in
        {"present", "partial", "missing", "contradicted"}
        for item in evaluation["results"]
    )

    fact_comparison_results.append({
        "case_id": row["case_id"],
        "method": row["method"],
        "results": evaluation["results"],
        "finish_reason": finish_reason
    })

    print("Facts evaluated:", len(returned_ids))
    print("Finish reason:", finish_reason)

print("\nFact comparison completed.")

In [ ]:
with open(
    "case_0014_fact_comparison_pilot.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        fact_comparison_results,
        f,
        indent=2,
        ensure_ascii=False
    )

for result in fact_comparison_results:

    statuses = [
        item["status"]
        for item in result["results"]
    ]

    print("\nMethod:", result["method"])

    for status in [
        "present",
        "partial",
        "missing",
        "contradicted"
    ]:
        print(
            f"{status}:",
            statuses.count(status)
        )

print("\nComparison saved successfully.")

In [ ]:
strict_comparison_rules = """
Evaluate each reference clinical fact independently.

CLASSIFICATION RULES:

present:
The generated note preserves the complete meaning
of the reference fact, including all essential details.

partial:
The generated note includes the core fact but omits
important details such as dosage, frequency, duration,
laterality, numerical value, or negation context.

missing:
The reference fact is not represented in the note.

contradicted:
The note explicitly conflicts with the reference fact.

IMPORTANT:
1. Do not penalize a fact because the note contains
   other unrelated information.
2. Additional medications or findings do not make
   an otherwise correct fact partial.
3. A medication name without its documented dose
   and frequency is partial.
4. An incorrect dose or reversed negation is contradicted.
5. A synonym is acceptable if the medical meaning is preserved.
6. Do not infer absent details.
7. Every fact must receive exactly one classification.
8. The note_evidence field must quote text that actually
   occurs in the generated note.
9. If a fact is missing, return an empty note_evidence string.
10. Provide a specific reason for partial or contradicted facts.
"""

print("Strict comparison rules ready.")

In [ ]:
for result in fact_comparison_results:

    method = result["method"]

    note = comparison_df.loc[
        comparison_df["method"] == method,
        "generated_note"
    ].iloc[0]

    print("\nMETHOD:", method)

    valid_count = 0
    invalid_count = 0

    for item in result["results"]:

        evidence = item.get("note_evidence", "")

        valid = (
            evidence in note
            if evidence
            else item["status"] == "missing"
        )

        if valid:
            valid_count += 1
        else:
            invalid_count += 1
            print(
                "Evidence mismatch:",
                item["fact_id"]
            )

    print("Valid evidence entries:", valid_count)
    print("Invalid evidence entries:", invalid_count)

In [ ]:
# Display the reference fact
reference_fact = fact_checklist[
    fact_checklist["fact_id"] == "F005"
].iloc[0]

print("REFERENCE FACT:")
print(reference_fact["fact"])

print("\nSOURCE EVIDENCE:")
print(reference_fact["source_evidence"])

# Inspect F005 for both generated notes
for result in fact_comparison_results:

    method = result["method"]

    note = comparison_df.loc[
        comparison_df["method"] == method,
        "generated_note"
    ].iloc[0]

    f005_result = next(
        item for item in result["results"]
        if item["fact_id"] == "F005"
    )

    print("\n" + "=" * 50)
    print("METHOD:", method)

    print("\nEvaluator classification:")
    print(f005_result["status"])

    print("\nEvaluator evidence:")
    print(f005_result["note_evidence"])

    print("\nEvidence occurs exactly in note:")
    print(f005_result["note_evidence"] in note)

    print("\nACTUAL GENERATED NOTE:")
    print(note)

In [ ]:
import copy

# Preserve original results
corrected_comparison = copy.deepcopy(
    fact_comparison_results
)

# Correct F005 in both evaluations
for result in corrected_comparison:
    for item in result["results"]:

        if item["fact_id"] == "F005":
            item["status"] = "missing"
            item["note_evidence"] = ""
            item["reason"] = (
                "The generated note does not document "
                "the patient's denial of cardiac disease "
                "or prior history of cancer."
            )

print("F005 classifications corrected.")

for result in corrected_comparison:
    f005 = next(
        item for item in result["results"]
        if item["fact_id"] == "F005"
    )

    print(result["method"], ":", f005["status"])

In [ ]:
import json

with open(
    "case_0014_fact_comparison_corrected.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        corrected_comparison,
        f,
        indent=2,
        ensure_ascii=False
    )

for result in corrected_comparison:

    statuses = [
        item["status"]
        for item in result["results"]
    ]

    print("\nMethod:", result["method"])

    for status in [
        "present",
        "partial",
        "missing",
        "contradicted"
    ]:
        print(status, ":", statuses.count(status))

print("\nCorrected comparison saved successfully.")

In [ ]:
# Select facts F011 through F020
next_facts = fact_checklist.iloc[10:20].to_dict(
    orient="records"
)

print("Facts selected:", len(next_facts))

for item in next_facts:
    print(item["fact_id"], "|", item["fact"])

In [ ]:
import json

next_comparison_results = []

for _, row in comparison_df.iterrows():

    print("\nEvaluating:", row["method"])

    prompt = f"""
{strict_comparison_rules}

Return valid JSON in this format:

{{
  "results": [
    {{
      "fact_id": "F011",
      "status": "present",
      "note_evidence": "Exact excerpt from generated note",
      "reason": "Explanation of classification"
    }}
  ]
}}

REFERENCE FACTS:
{json.dumps(next_facts, ensure_ascii=False)}

GENERATED NOTE:
{row["generated_note"]}
"""

    response_eval = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2500,
        response_format={"type": "json_object"}
    )

    finish_reason = response_eval.choices[0].finish_reason

    if finish_reason != "stop":
        raise RuntimeError(
            f"Incomplete evaluation: {finish_reason}"
        )

    evaluation = json.loads(
        response_eval.choices[0].message.content
    )

    expected_ids = {f["fact_id"] for f in next_facts}
    returned_ids = [r["fact_id"] for r in evaluation["results"]]

    assert len(returned_ids) == len(expected_ids)
    assert set(returned_ids) == expected_ids

    next_comparison_results.append({
        "case_id": row["case_id"],
        "method": row["method"],
        "results": evaluation["results"],
        "finish_reason": finish_reason
    })

    print("Facts evaluated:", len(returned_ids))
    print("Finish reason:", finish_reason)

print("\nNext batch completed.")

In [ ]:
for result in next_comparison_results:

    method = result["method"]

    note = comparison_df.loc[
        comparison_df["method"] == method,
        "generated_note"
    ].iloc[0]

    print("\nMETHOD:", method)

    for item in result["results"]:

        evidence = item.get("note_evidence", "")

        evidence_valid = (
            evidence in note
            if evidence
            else item["status"] == "missing"
        )

        item["evidence_valid"] = evidence_valid

        if not evidence_valid:
            print("Evidence mismatch:", item["fact_id"])

    statuses = [r["status"] for r in result["results"]]

    print("Present:", statuses.count("present"))
    print("Partial:", statuses.count("partial"))
    print("Missing:", statuses.count("missing"))
    print("Contradicted:", statuses.count("contradicted"))

with open(
    "case_0014_fact_comparison_batch2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        next_comparison_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nBatch 2 saved successfully.")

In [ ]:
import copy
import json

corrected_batch2 = copy.deepcopy(next_comparison_results)

reviewed_statuses = {
    "Simple Baseline": {
        "F011": "partial",
        "F012": "partial",
        "F013": "partial",
        "F014": "partial",
        "F015": "missing",
        "F016": "present",
        "F017": "partial",
        "F018": "present",
        "F019": "present",
        "F020": "present"
    },
    "Structured Prompt": {
        "F011": "partial",
        "F012": "partial",
        "F013": "partial",
        "F014": "partial",
        "F015": "missing",
        "F016": "present",
        "F017": "partial",
        "F018": "partial",
        "F019": "present",
        "F020": "present"
    }
}

# Apply reviewed classifications
for result in corrected_batch2:
    method = result["method"]

    for item in result["results"]:
        fid = item["fact_id"]
        item["original_status"] = item["status"]
        item["status"] = reviewed_statuses[method][fid]
        item["review_type"] = "manual_review"
        item["review_note"] = (
            "Classification reviewed against the "
            "original fact and generated clinical note."
        )

        # Discard unverified evidence quotations
        if not item.get("evidence_valid", False):
            item["note_evidence"] = ""

# Save corrected classifications
with open(
    "case_0014_fact_comparison_batch2_corrected.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        corrected_batch2,
        f,
        indent=2,
        ensure_ascii=False
    )

for result in corrected_batch2:
    statuses = [x["status"] for x in result["results"]]

    print("\n", result["method"])
    print("Present:", statuses.count("present"))
    print("Partial:", statuses.count("partial"))
    print("Missing:", statuses.count("missing"))
    print("Contradicted:", statuses.count("contradicted"))

print("\nCorrected batch 2 saved successfully.")

In [ ]:
import re
import json

def create_sentence_map(note):
    # Pilot sentence splitter for these two notes
    sentences = re.split(
        r'(?<=[.!?])\s+|\n+',
        note.strip()
    )

    sentences = [
        s.strip() for s in sentences if s.strip()
    ]

    return {
        f"S{i:03d}": sentence
        for i, sentence in enumerate(sentences, 1)
    }

note_sentence_maps = {}

for _, row in comparison_df.iterrows():
    method = row["method"]

    sentence_map = create_sentence_map(
        row["generated_note"]
    )

    note_sentence_maps[method] = sentence_map

    print("\nMETHOD:", method)
    print("Number of text segments:", len(sentence_map))

    for sid, sentence in list(sentence_map.items())[:5]:
        print(sid, ":", sentence)

with open(
    "case_0014_sentence_maps.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        note_sentence_maps,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSentence maps saved successfully.")

In [ ]:
print("Validation data:", len(val_df))
print("Model:", MODEL)
print("Extracted facts:", len(final_section_facts))
print("Sentence maps:", len(note_sentence_maps))

In [ ]:
import pandas as pd
import json

# Restore the two generated notes
comparison_df = pd.read_csv(
    "baseline_vs_structured_1_case.csv"
)

# Restore the 70 clinical facts
fact_checklist = pd.read_csv(
    "case_0014_fact_checklist.csv"
)

# Restore numbered note segments
with open(
    "case_0014_sentence_maps.json",
    "r",
    encoding="utf-8"
) as f:
    note_sentence_maps = json.load(f)

# Restore the case ID and model setting
case_id = "CASE_0014"
MODEL = "gpt-3.5-turbo"

# Restore facts F011 to F020
next_facts = fact_checklist.iloc[10:20].to_dict(
    orient="records"
)

print("Notes restored:", len(comparison_df))
print("Facts restored:", len(fact_checklist))
print("Methods with sentence maps:", len(note_sentence_maps))
print("Next facts:", len(next_facts))

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Retrieve the saved API key
api_key = userdata.get("OPENAI_API_KEY")

# Restore the OpenAI client
client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")
print("Selected model:", MODEL)

In [ ]:
strict_comparison_rules = """
Evaluate each reference clinical fact independently.

CLASSIFICATION RULES:

present:
The generated note preserves the complete meaning
of the reference fact, including all essential details.

partial:
The generated note includes the core fact but omits
important details such as dosage, frequency, duration,
laterality, numerical value, or negation context.

missing:
The reference fact is not represented in the note.

contradicted:
The note explicitly conflicts with the reference fact.

IMPORTANT:
1. Do not penalize a fact because the note contains
   other unrelated information.
2. Additional medications or findings do not make
   an otherwise correct fact partial.
3. A medication name without its documented dose
   and frequency is partial.
4. An incorrect dose or reversed negation is contradicted.
5. A synonym is acceptable if the medical meaning is preserved.
6. Do not infer absent details.
7. Every fact must receive exactly one classification.
8. Use only the provided clinical facts and generated note.
9. Provide a specific reason for each classification.
"""

print("Strict comparison rules restored.")

In [ ]:
print("=== EXPERIMENT STATUS ===")

print("Clinical notes:", len(comparison_df))
print("Reference facts:", len(fact_checklist))
print("Sentence maps:", len(note_sentence_maps))
print("Next batch facts:", len(next_facts))
print("Model:", MODEL)

print("\nMethods available:")
print(comparison_df["method"].tolist())

In [ ]:
import json

evidence_id_results = []

for _, row in comparison_df.iterrows():

    method = row["method"]
    sentence_map = note_sentence_maps[method]

    print("\nEvaluating:", method)

    prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate each reference fact against the
numbered generated-note segments.

ADDITIONAL RULES:
1. Select evidence only using the provided sentence IDs.
2. Never invent evidence IDs or quotations.
3. Use multiple evidence IDs if necessary.
4. Missing facts must have an empty evidence_ids list.
5. If a medication's documented dosage or frequency
   is missing, classify it as partial.
6. If an allergy reaction is missing, classify it as partial.
7. Do not count unrelated details as errors.
8. Evaluate every fact exactly once.

Return valid JSON:

{{
  "results": [
    {{
      "fact_id": "F011",
      "status": "partial",
      "evidence_ids": ["S001"],
      "reason": "Explanation"
    }}
  ]
}}

REFERENCE FACTS:
{json.dumps(next_facts, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(sentence_map, ensure_ascii=False)}
"""

    response_eval = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2500,
        response_format={"type": "json_object"}
    )

    finish_reason = response_eval.choices[0].finish_reason

    if finish_reason != "stop":
        raise RuntimeError(
            f"Evaluation incomplete: {finish_reason}"
        )

    evaluation = json.loads(
        response_eval.choices[0].message.content
    )

    # Validate fact IDs
    expected_ids = {f["fact_id"] for f in next_facts}
    returned_ids = [
        item["fact_id"] for item in evaluation["results"]
    ]

    assert len(returned_ids) == len(expected_ids)
    assert set(returned_ids) == expected_ids

    # Validate evidence IDs
    allowed_statuses = {
        "present", "partial", "missing", "contradicted"
    }

    for item in evaluation["results"]:

        assert item["status"] in allowed_statuses

        ids = item.get("evidence_ids", [])

        item["evidence_ids_valid"] = (
            isinstance(ids, list)
            and all(sid in sentence_map for sid in ids)
            and (
                item["status"] != "missing"
                or len(ids) == 0
            )
        )

    evidence_id_results.append({
        "case_id": row["case_id"],
        "method": method,
        "results": evaluation["results"],
        "finish_reason": finish_reason
    })

    print("Finish reason:", finish_reason)

print("\nEvidence-ID evaluations completed.")

In [ ]:

print("MODEL:", globals().get("MODEL"))

print(
    "Comparison rows:",
    len(comparison_df)
    if "comparison_df" in globals()
    else "Not available"
)

print(
    "Reference facts:",
    len(next_facts)
    if "next_facts" in globals()
    else "Not available"
)

print(
    "Completed evaluations:",
    len(evidence_id_results)
    if "evidence_id_results" in globals()
    else "Not available"
)

print(
    "API client available:",
    "client" in globals()
)

print("\nDiagnosis completed.")


In [ ]:
# Inspect the response from the failed evaluation
print("Expected fact IDs:")
print(sorted(expected_ids))

print("\nReturned fact IDs:")
print(returned_ids)

print("\nExpected count:", len(expected_ids))
print("Returned count:", len(returned_ids))

print("\nMissing fact IDs:")
print(sorted(set(expected_ids) - set(returned_ids)))

print("\nUnexpected fact IDs:")
print(sorted(set(returned_ids) - set(expected_ids)))

print("\nRepeated fact IDs:")
from collections import Counter

counts = Counter(returned_ids)
print([
    fact_id for fact_id, count in counts.items()
    if count > 1
])

In [ ]:
import copy

# Preserve the five results from the interrupted API response
first_five_results = copy.deepcopy(
    evaluation["results"]
)

print("Saved evaluations:", len(first_five_results))

print("Fact IDs:")
for item in first_five_results:
    print(item["fact_id"], ":", item["status"])

In [ ]:
missing_ids = [
    "F016", "F017", "F018", "F019", "F020"
]

remaining_facts = fact_checklist[
    fact_checklist["fact_id"].isin(missing_ids)
].to_dict(orient="records")

print("Remaining facts:", len(remaining_facts))

for item in remaining_facts:
    print(item["fact_id"], "|", item["fact"])

In [ ]:
import json

# Get the Simple Baseline sentence map
simple_sentence_map = note_sentence_maps["Simple Baseline"]

prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate ONLY these five reference facts.

Rules:
1. Return exactly one result per fact ID.
2. Use only the provided sentence IDs as evidence.
3. Never invent evidence IDs.
4. Missing facts must have an empty evidence_ids list.
5. Preserve numerical values and important details.
6. Do not classify a fact as present if important
   details are omitted.

Return JSON only:

{{
  "results": [
    {{
      "fact_id": "F016",
      "status": "present",
      "evidence_ids": ["S001"],
      "reason": "Explanation"
    }}
  ]
}}

REFERENCE FACTS:
{json.dumps(remaining_facts, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(simple_sentence_map, ensure_ascii=False)}
"""

remaining_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=1800,
    response_format={"type": "json_object"}
)

finish_reason = remaining_response.choices[0].finish_reason

print("Finish reason:", finish_reason)

if finish_reason != "stop":
    raise RuntimeError("Evaluation may be incomplete.")

remaining_evaluation = json.loads(
    remaining_response.choices[0].message.content
)

remaining_results = remaining_evaluation["results"]

expected_remaining_ids = {
    item["fact_id"] for item in remaining_facts
}

returned_remaining_ids = [
    item["fact_id"] for item in remaining_results
]

print("Expected facts:", len(expected_remaining_ids))
print("Returned facts:", len(returned_remaining_ids))

print("Missing IDs:",
      sorted(expected_remaining_ids - set(returned_remaining_ids)))

print("Unexpected IDs:",
      sorted(set(returned_remaining_ids) - expected_remaining_ids))

for item in remaining_results:
    print(item["fact_id"], ":", item["status"])

In [ ]:
import json

# Preserve the F016 result from Code 80
f016_result = remaining_results[0].copy()

# Select the four facts not yet evaluated
missing_four = fact_checklist[
    fact_checklist["fact_id"].isin(
        ["F017", "F018", "F019", "F020"]
    )
].to_dict(orient="records")

individual_results = []

for fact in missing_four:

    print("Evaluating:", fact["fact_id"])

    prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate this ONE reference fact against the
numbered clinical note segments.

REFERENCE FACT:
{json.dumps(fact, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(simple_sentence_map, ensure_ascii=False)}

Return valid JSON with exactly these fields:

{{
    "status": "present",
    "evidence_ids": ["S001"],
    "reason": "Explanation"
}}

Select evidence IDs only from the provided note segments.
Do not invent evidence IDs.
If the fact is missing, use an empty evidence_ids list.
"""

    response_one = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=500,
        response_format={"type": "json_object"}
    )

    if response_one.choices[0].finish_reason != "stop":
        raise RuntimeError(
            f"Incomplete response for {fact['fact_id']}"
        )

    result = json.loads(
        response_one.choices[0].message.content
    )

    result["fact_id"] = fact["fact_id"]

    assert result["status"] in {
        "present", "partial", "missing", "contradicted"
    }

    ids = result.get("evidence_ids", [])

    assert isinstance(ids, list)
    assert all(sid in simple_sentence_map for sid in ids)

    individual_results.append(result)

    print("Status:", result["status"])

print("\nIndividual evaluations completed.")

In [ ]:
# Combine all 10 evaluations
simple_combined_results = (
    first_five_results
    + [f016_result]
    + individual_results
)

# Verify coverage
combined_ids = [
    item["fact_id"] for item in simple_combined_results
]

expected_ids = {
    f"F{i:03d}" for i in range(11, 21)
}

assert len(combined_ids) == 10
assert set(combined_ids) == expected_ids
assert len(set(combined_ids)) == 10

# Add evidence-ID validation
for item in simple_combined_results:
    ids = item.get("evidence_ids", [])

    item["evidence_ids_valid"] = (
        isinstance(ids, list)
        and all(sid in simple_sentence_map for sid in ids)
        and (
            item["status"] != "missing"
            or len(ids) == 0
        )
    )

# Save results
with open(
    "case_0014_simple_baseline_evidence_ids.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "case_id": case_id,
            "method": "Simple Baseline",
            "results": simple_combined_results
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("Total facts:", len(simple_combined_results))
print(
    "Valid evidence-ID selections:",
    sum(x["evidence_ids_valid"] for x in simple_combined_results)
)

print("\nSimple Baseline evaluation saved.")

In [ ]:
import json
from collections import Counter

reviewed_statuses = {
    "F011": "partial",
    "F012": "partial",
    "F013": "partial",
    "F014": "partial",
    "F015": "missing",
    "F016": "present",
    "F017": "partial",
    "F018": "present",
    "F019": "present",
    "F020": "present"
}

with open(
    "case_0014_simple_baseline_evidence_ids.json",
    "r",
    encoding="utf-8"
) as f:
    original_data = json.load(f)

reviewed_data = {
    "case_id": original_data["case_id"],
    "method": original_data["method"],
    "results": []
}

for item in original_data["results"]:
    corrected = item.copy()
    corrected["model_status"] = item["status"]
    corrected["reviewed_status"] = reviewed_statuses[
        item["fact_id"]
    ]
    reviewed_data["results"].append(corrected)

counts = Counter(
    item["reviewed_status"]
    for item in reviewed_data["results"]
)

print("Total facts:", len(reviewed_data["results"]))
print("Reviewed classifications:", dict(counts))

with open(
    "case_0014_simple_baseline_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reviewed_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Reviewed file saved successfully.")

In [ ]:
import json

# Select reference fact F011
fact = fact_checklist[
    fact_checklist["fact_id"] == "F011"
].iloc[0].to_dict()

# Use the Structured Prompt note segments
structured_sentence_map = note_sentence_maps[
    "Structured Prompt"
]

prompt = f"""
You are evaluating clinical fact preservation.

{strict_comparison_rules}

Evaluate exactly ONE reference fact.

Return a JSON object containing:
- status
- evidence_ids
- reason

Do not return a fact_id.

Use only the numbered note segments as evidence.
Never invent evidence IDs.
If the fact is missing, return an empty evidence_ids list.

REFERENCE FACT:
{json.dumps(fact, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(structured_sentence_map, ensure_ascii=False)}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=500,
    response_format={"type": "json_object"}
)

assert response.choices[0].finish_reason == "stop"

structured_f011 = json.loads(
    response.choices[0].message.content
)

structured_f011["fact_id"] = "F011"

print("Fact ID:", structured_f011["fact_id"])
print("Status:", structured_f011["status"])
print("Evidence IDs:", structured_f011["evidence_ids"])
print("Reason:", structured_f011["reason"])

print(
    "Evidence IDs valid:",
    all(
        sid in structured_sentence_map
        for sid in structured_f011["evidence_ids"]
    )
)

In [ ]:
allowed_statuses = {
    "present", "partial", "missing", "contradicted"
}

status = structured_f011["status"]
evidence_ids = structured_f011["evidence_ids"]

status_valid = status in allowed_statuses

ids_exist = (
    isinstance(evidence_ids, list)
    and all(
        sid in structured_sentence_map
        for sid in evidence_ids
    )
)

# Present and partial require at least one evidence ID.
# Missing requires no evidence IDs.
if status in {"present", "partial"}:
    evidence_valid = ids_exist and len(evidence_ids) > 0
elif status == "missing":
    evidence_valid = ids_exist and len(evidence_ids) == 0
else:
    evidence_valid = ids_exist

print("Fact ID:", structured_f011["fact_id"])
print("Model classification:", status)
print("Reviewed classification: partial")
print("Status format valid:", status_valid)
print("Evidence validation passed:", evidence_valid)

print("\nReferenced evidence text:")
for sid in evidence_ids:
    print(sid, ":", structured_sentence_map[sid])

print("\nSegments mentioning Veramist:")
for sid, text in structured_sentence_map.items():
    if "veramist" in text.lower():
        print(sid, ":", text)

In [ ]:
import json

# Preserve the original model output
structured_f011_reviewed = structured_f011.copy()

# Add independently reviewed fields
structured_f011_reviewed["model_status"] = (
    structured_f011["status"]
)

structured_f011_reviewed["reviewed_status"] = "partial"

structured_f011_reviewed["reviewed_evidence_ids"] = [
    "S007"
]

structured_f011_reviewed["review_reason"] = (
    "The note mentions Veramist spray but omits "
    "the documented dose of 27.5 mcg and daily frequency."
)

# Save the reviewed result
with open(
    "case_0014_structured_f011_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_f011_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fact:", structured_f011_reviewed["fact_id"])
print("Model status:", structured_f011_reviewed["model_status"])
print("Reviewed status:", structured_f011_reviewed["reviewed_status"])
print(
    "Reviewed evidence IDs:",
    structured_f011_reviewed["reviewed_evidence_ids"]
)
print("F011 reviewed result saved successfully.")

In [ ]:
import json

fact_f012 = fact_checklist[
    fact_checklist["fact_id"] == "F012"
].iloc[0].to_dict()

prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate exactly ONE clinical fact.

A medication name alone does not preserve its
documented dose or frequency.

Return JSON with:
- status
- evidence_ids
- reason

For present or partial classifications,
at least one evidence ID is required.

Use only the numbered note segments provided.

REFERENCE FACT:
{json.dumps(fact_f012, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(structured_sentence_map, ensure_ascii=False)}
"""

response_f012 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=500,
    response_format={"type": "json_object"}
)

assert response_f012.choices[0].finish_reason == "stop"

structured_f012 = json.loads(
    response_f012.choices[0].message.content
)

structured_f012["fact_id"] = "F012"

ids = structured_f012.get("evidence_ids", [])
status = structured_f012.get("status")

evidence_valid = (
    isinstance(ids, list)
    and all(sid in structured_sentence_map for sid in ids)
    and (
        (status in {"present", "partial"} and len(ids) > 0)
        or (status == "missing" and len(ids) == 0)
        or (status == "contradicted" and len(ids) > 0)
    )
)

print("Fact:", structured_f012["fact_id"])
print("Model classification:", status)
print("Evidence IDs:", ids)
print("Evidence validation passed:", evidence_valid)
print("Reason:", structured_f012.get("reason"))

print("\nReferenced note segments:")
for sid in ids:
    if sid in structured_sentence_map:
        print(sid, ":", structured_sentence_map[sid])

In [ ]:
import json

# Preserve the model's original evaluation
structured_f012_reviewed = structured_f012.copy()

# Record the reviewed classification
structured_f012_reviewed["model_status"] = (
    structured_f012["status"]
)

structured_f012_reviewed["reviewed_status"] = "partial"

structured_f012_reviewed["reviewed_evidence_ids"] = [
    "S004", "S007"
]

structured_f012_reviewed["review_reason"] = (
    "The note mentions Restasis but omits "
    "the documented twice-daily frequency."
)

# Save the reviewed evaluation
with open(
    "case_0014_structured_f012_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_f012_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fact:", structured_f012_reviewed["fact_id"])
print("Model status:", structured_f012_reviewed["model_status"])
print("Reviewed status:", structured_f012_reviewed["reviewed_status"])
print(
    "Reviewed evidence IDs:",
    structured_f012_reviewed["reviewed_evidence_ids"]
)
print("F012 reviewed result saved successfully.")

In [ ]:
import json

fact_f013 = fact_checklist[
    fact_checklist["fact_id"] == "F013"
].iloc[0].to_dict()

prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate exactly ONE clinical fact.

Important:
- Mentioning a medication does not establish its frequency.
- A missing documented frequency requires "partial".
- Do not infer information absent from the note.
- For present or partial, provide at least one evidence ID.
- Use only the numbered note segments provided.

Return JSON containing:
- status
- evidence_ids
- reason

REFERENCE FACT:
{json.dumps(fact_f013, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(structured_sentence_map, ensure_ascii=False)}
"""

response_f013 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=500,
    response_format={"type": "json_object"}
)

assert response_f013.choices[0].finish_reason == "stop"

structured_f013 = json.loads(
    response_f013.choices[0].message.content
)

structured_f013["fact_id"] = "F013"

status = structured_f013.get("status")
ids = structured_f013.get("evidence_ids", [])

evidence_valid = (
    status in {"present", "partial", "missing", "contradicted"}
    and isinstance(ids, list)
    and all(sid in structured_sentence_map for sid in ids)
    and (
        (status == "missing" and len(ids) == 0)
        or (status != "missing" and len(ids) > 0)
    )
)

print("Fact:", structured_f013["fact_id"])
print("Model classification:", status)
print("Evidence IDs:", ids)
print("Evidence validation passed:", evidence_valid)
print("Reason:", structured_f013.get("reason"))

print("\nReferenced note segments:")
for sid in ids:
    if sid in structured_sentence_map:
        print(sid, ":", structured_sentence_map[sid])

In [ ]:
import json

# Preserve the original model evaluation
structured_f013_reviewed = structured_f013.copy()

# Record the reviewed classification
structured_f013_reviewed["model_status"] = (
    structured_f013["status"]
)

structured_f013_reviewed["reviewed_status"] = "partial"

structured_f013_reviewed["reviewed_evidence_ids"] = [
    "S007"
]

structured_f013_reviewed["review_reason"] = (
    "The note mentions ibuprofen but omits "
    "the documented frequency of two to three "
    "times per day."
)

# Save the reviewed evaluation
with open(
    "case_0014_structured_f013_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_f013_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fact:", structured_f013_reviewed["fact_id"])
print("Model status:", structured_f013_reviewed["model_status"])
print("Reviewed status:", structured_f013_reviewed["reviewed_status"])
print(
    "Reviewed evidence IDs:",
    structured_f013_reviewed["reviewed_evidence_ids"]
)
print("F013 reviewed result saved successfully.")

In [ ]:
import json

fact_f014 = fact_checklist[
    fact_checklist["fact_id"] == "F014"
].iloc[0].to_dict()

prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate exactly ONE reference clinical fact.

Important rules:
1. An allergy name without its documented reaction
   is only a partial representation.
2. Do not infer a reaction that is absent from the note.
3. For present or partial, provide at least one evidence ID.
4. Select IDs only from the supplied note segments.
5. Return only a JSON object with:
   status, evidence_ids, and reason.

REFERENCE FACT:
{json.dumps(fact_f014, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(structured_sentence_map, ensure_ascii=False)}
"""

response_f014 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=500,
    response_format={"type": "json_object"}
)

assert response_f014.choices[0].finish_reason == "stop"

structured_f014 = json.loads(
    response_f014.choices[0].message.content
)

structured_f014["fact_id"] = "F014"

status = structured_f014.get("status")
ids = structured_f014.get("evidence_ids", [])

evidence_valid = (
    status in {"present", "partial", "missing", "contradicted"}
    and isinstance(ids, list)
    and all(sid in structured_sentence_map for sid in ids)
    and (
        (status == "missing" and len(ids) == 0)
        or (status != "missing" and len(ids) > 0)
    )
)

print("Fact:", structured_f014["fact_id"])
print("Model classification:", status)
print("Evidence IDs:", ids)
print("Evidence validation passed:", evidence_valid)
print("Reason:", structured_f014.get("reason"))

print("\nReferenced note segments:")
for sid in ids:
    if sid in structured_sentence_map:
        print(sid, ":", structured_sentence_map[sid])

In [ ]:
import json

structured_f014_reviewed = structured_f014.copy()

# Preserve the original model classification
structured_f014_reviewed["model_status"] = (
    structured_f014["status"]
)

# Record the clinically reviewed result
structured_f014_reviewed["reviewed_status"] = "partial"

structured_f014_reviewed["reviewed_evidence_ids"] = [
    "S008"
]

structured_f014_reviewed["review_reason"] = (
    "The note mentions the Bactrim allergy but "
    "omits the documented rash reaction."
)

# Save the reviewed result
with open(
    "case_0014_structured_f014_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_f014_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fact:", structured_f014_reviewed["fact_id"])
print("Model status:", structured_f014_reviewed["model_status"])
print("Reviewed status:", structured_f014_reviewed["reviewed_status"])
print(
    "Reviewed evidence IDs:",
    structured_f014_reviewed["reviewed_evidence_ids"]
)
print("F014 reviewed result saved successfully.")

In [ ]:
import json

fact_f015 = fact_checklist[
    fact_checklist["fact_id"] == "F015"
].iloc[0].to_dict()

prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate exactly ONE reference clinical fact.

Important rules:
1. A denied allergy is a clinically meaningful negative fact.
2. Do not assume that an unmentioned allergy is denied.
3. If the generated note does not mention the reference
   fact, classify it as missing.
4. For missing facts, evidence_ids must be an empty list.
5. Use only the supplied numbered note segments.
6. Return JSON with status, evidence_ids, and reason.

REFERENCE FACT:
{json.dumps(fact_f015, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(structured_sentence_map, ensure_ascii=False)}
"""

response_f015 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=500,
    response_format={"type": "json_object"}
)

assert response_f015.choices[0].finish_reason == "stop"

structured_f015 = json.loads(
    response_f015.choices[0].message.content
)

structured_f015["fact_id"] = "F015"

status = structured_f015.get("status")
ids = structured_f015.get("evidence_ids", [])

evidence_valid = (
    status in {"present", "partial", "missing", "contradicted"}
    and isinstance(ids, list)
    and all(sid in structured_sentence_map for sid in ids)
    and (
        (status == "missing" and len(ids) == 0)
        or (status != "missing" and len(ids) > 0)
    )
)

print("Fact:", structured_f015["fact_id"])
print("Model classification:", status)
print("Evidence IDs:", ids)
print("Evidence validation passed:", evidence_valid)
print("Reason:", structured_f015.get("reason"))

print("\nSegments mentioning latex:")
matches = [
    (sid, text)
    for sid, text in structured_sentence_map.items()
    if "latex" in text.lower()
]

if matches:
    for sid, text in matches:
        print(sid, ":", text)
else:
    print("No segments mention latex.")

In [ ]:
import json

structured_f015_reviewed = structured_f015.copy()

# Preserve the original model classification
structured_f015_reviewed["model_status"] = (
    structured_f015["status"]
)

# Record the reviewed classification
structured_f015_reviewed["reviewed_status"] = "missing"

structured_f015_reviewed["reviewed_evidence_ids"] = []

structured_f015_reviewed["review_reason"] = (
    "The source states that the patient denies "
    "latex allergy, but the generated note "
    "does not mention latex."
)

# Save the reviewed result
with open(
    "case_0014_structured_f015_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_f015_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Fact:", structured_f015_reviewed["fact_id"])
print("Model status:", structured_f015_reviewed["model_status"])
print("Reviewed status:", structured_f015_reviewed["reviewed_status"])
print(
    "Reviewed evidence IDs:",
    structured_f015_reviewed["reviewed_evidence_ids"]
)
print("F015 reviewed result saved successfully.")

In [ ]:
import json

def evaluate_fact_batch(facts, sentence_map):

    prompt = f"""
You are a clinical fact preservation evaluator.

{strict_comparison_rules}

Evaluate every reference fact independently.

Return exactly one result per fact ID.

Rules:
1. Classify each fact as present, partial,
   missing, or contradicted.
2. Use only the supplied evidence IDs.
3. Present and partial require evidence IDs.
4. Missing requires an empty evidence_ids list.
5. Do not infer details absent from the note.
6. Return only valid JSON.

Required JSON structure:
{{
  "results": [
    {{
      "fact_id": "F016",
      "status": "present",
      "evidence_ids": ["S001"],
      "reason": "Explanation"
    }}
  ]
}}

REFERENCE FACTS:
{json.dumps(facts, ensure_ascii=False)}

NUMBERED NOTE SEGMENTS:
{json.dumps(sentence_map, ensure_ascii=False)}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=2000,
        response_format={"type": "json_object"}
    )

    if response.choices[0].finish_reason != "stop":
        raise RuntimeError("Incomplete model response.")

    results = json.loads(
        response.choices[0].message.content
    )["results"]

    expected_ids = {f["fact_id"] for f in facts}

    valid_results = {}
    allowed_statuses = {
        "present", "partial", "missing", "contradicted"
    }

    for item in results:
        fact_id = item.get("fact_id")
        status = item.get("status")
        ids = item.get("evidence_ids")

        if fact_id not in expected_ids:
            continue

        if fact_id in valid_results:
            continue

        if status not in allowed_statuses:
            continue

        if not isinstance(ids, list):
            continue

        if not all(sid in sentence_map for sid in ids):
            continue

        if status == "missing" and ids:
            continue

        if status != "missing" and not ids:
            continue

        valid_results[fact_id] = item

    missing_ids = expected_ids - set(valid_results)

    return {
        "valid_results": valid_results,
        "missing_ids": sorted(missing_ids)
    }

print("Automated evaluation function created successfully.")

In [ ]:
def evaluate_with_recovery(facts, sentence_map):

    # First attempt: evaluate the entire batch
    batch = evaluate_fact_batch(facts, sentence_map)

    collected = dict(batch["valid_results"])
    missing_ids = batch["missing_ids"]

    print("Initial evaluations:", len(collected))
    print("Facts requiring recovery:", len(missing_ids))

    # Recover missing facts individually
    for fact_id in missing_ids:

        fact = next(
            f for f in facts
            if f["fact_id"] == fact_id
        )

        print("Recovering:", fact_id)

        try:
            recovered = evaluate_fact_batch(
                [fact],
                sentence_map
            )

            if fact_id in recovered["valid_results"]:
                collected[fact_id] = (
                    recovered["valid_results"][fact_id]
                )
            else:
                print("Still unresolved:", fact_id)

        except Exception as error:
            print("Recovery error:", fact_id, str(error))

    # Return results in the original fact order
    ordered_results = [
        collected[f["fact_id"]]
        for f in facts
        if f["fact_id"] in collected
    ]

    unresolved = [
        f["fact_id"]
        for f in facts
        if f["fact_id"] not in collected
    ]

    return {
        "results": ordered_results,
        "unresolved_fact_ids": unresolved
    }

print("Automatic recovery function created successfully.")

In [ ]:
import json

# Select facts F016–F020
facts_to_evaluate = fact_checklist[
    fact_checklist["fact_id"].isin([
        "F016", "F017", "F018", "F019", "F020"
    ])
].to_dict(orient="records")

# Run evaluation with automatic recovery
structured_batch2 = evaluate_with_recovery(
    facts_to_evaluate,
    structured_sentence_map
)

# Display the results
print("\n=== STRUCTURED PROMPT RESULTS ===")

for item in structured_batch2["results"]:
    print(
        item["fact_id"],
        "|",
        item["status"],
        "| Evidence:",
        item["evidence_ids"]
    )

print(
    "\nTotal evaluated:",
    len(structured_batch2["results"])
)

print(
    "Unresolved facts:",
    structured_batch2["unresolved_fact_ids"]
)

# Save results before manual review
with open(
    "case_0014_structured_f016_f020_raw.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_batch2,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nRaw evaluation results saved successfully.")

In [ ]:
for item in structured_batch2["results"]:

    fact_id = item["fact_id"]

    reference = fact_checklist[
        fact_checklist["fact_id"] == fact_id
    ].iloc[0]

    print("\n" + "=" * 60)
    print("FACT:", fact_id)
    print("REFERENCE:", reference["fact"])
    print("MODEL STATUS:", item["status"])
    print("MODEL REASON:", item.get("reason", ""))

    print("\nMODEL-SELECTED EVIDENCE:")

    ids = item.get("evidence_ids", [])

    if ids:
        for sid in ids:
            print(
                sid, ":",
                structured_sentence_map.get(
                    sid, "INVALID ID"
                )
            )
    else:
        print("No evidence selected.")

print("\n" + "=" * 60)
print("ADDITIONAL CHECK: MARRIAGE AND CHILDREN")

for sid, text in structured_sentence_map.items():
    if any(
        word in text.lower()
        for word in ["married", "children"]
    ):
        print(sid, ":", text)

In [ ]:
import json
from collections import Counter

reviewed_statuses = {
    "F016": "present",
    "F017": "partial",
    "F018": "partial",
    "F019": "present",
    "F020": "present"
}

review_reasons = {
    "F016": "The note explicitly states lifelong nonsmoker.",
    "F017": "Social drinking is mentioned, but the monthly frequency is omitted.",
    "F018": "Manager is documented, but the New York department of taxation is omitted.",
    "F019": "The note explicitly states married.",
    "F020": "The note explicitly states four children."
}

reviewed_results = []

for item in structured_batch2["results"]:
    corrected = item.copy()
    fact_id = item["fact_id"]

    corrected["model_status"] = item["status"]
    corrected["reviewed_status"] = reviewed_statuses[fact_id]
    corrected["reviewed_evidence_ids"] = ["S009"]
    corrected["review_reason"] = review_reasons[fact_id]

    reviewed_results.append(corrected)

reviewed_batch2 = {
    "case_id": "CASE_0014",
    "method": "Structured Prompt",
    "results": reviewed_results
}

with open(
    "case_0014_structured_f016_f020_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reviewed_batch2,
        f,
        indent=2,
        ensure_ascii=False
    )

counts = Counter(
    item["reviewed_status"]
    for item in reviewed_results
)

print("Reviewed facts:", len(reviewed_results))
print("Reviewed classifications:", dict(counts))

print(
    "Changed classifications:",
    sum(
        item["model_status"] != item["reviewed_status"]
        for item in reviewed_results
    )
)

print("Reviewed batch saved successfully.")

In [ ]:
import json
from collections import Counter

combined_results = []

# Load individually reviewed facts F011–F015
for number in range(11, 16):
    fact_id = f"F{number:03d}"

    filename = (
        f"case_0014_structured_{fact_id.lower()}_reviewed.json"
    )

    with open(filename, "r", encoding="utf-8") as f:
        combined_results.append(json.load(f))

# Load reviewed facts F016–F020
with open(
    "case_0014_structured_f016_f020_reviewed.json",
    "r",
    encoding="utf-8"
) as f:
    second_batch = json.load(f)

combined_results.extend(second_batch["results"])

# Verify all ten fact IDs
expected_ids = {
    f"F{i:03d}" for i in range(11, 21)
}

actual_ids = [
    item["fact_id"] for item in combined_results
]

assert len(actual_ids) == 10
assert set(actual_ids) == expected_ids
assert len(set(actual_ids)) == 10

# Save consolidated results
structured_reviewed = {
    "case_id": "CASE_0014",
    "method": "Structured Prompt",
    "results": combined_results
}

with open(
    "case_0014_structured_f011_f020_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        structured_reviewed,
        f,
        indent=2,
        ensure_ascii=False
    )

counts = Counter(
    item["reviewed_status"]
    for item in combined_results
)

print("Total reviewed facts:", len(combined_results))
print("Reviewed classifications:", dict(counts))
print("Consolidated file saved successfully.")

In [ ]:
import json
import pandas as pd
from collections import Counter

files = {
    "Simple Baseline":
        "case_0014_simple_baseline_reviewed.json",
    "Structured Prompt":
        "case_0014_structured_f011_f020_reviewed.json"
}

metrics = []

for method, filename in files.items():

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = [
        item for item in data["results"]
        if item["fact_id"] in {
            f"F{i:03d}" for i in range(11, 21)
        }
    ]

    counts = Counter(
        item["reviewed_status"] for item in results
    )

    total = len(results)

    assert total == 10

    metrics.append({
        "method": method,
        "total_facts": total,
        "present": counts["present"],
        "partial": counts["partial"],
        "missing": counts["missing"],
        "contradicted": counts["contradicted"],
        "exact_preservation_rate": round(
            counts["present"] / total * 100, 2
        ),
        "partial_preservation_rate": round(
            counts["partial"] / total * 100, 2
        ),
        "omission_rate": round(
            counts["missing"] / total * 100, 2
        )
    })

metrics_df = pd.DataFrame(metrics)

display(metrics_df)

metrics_df.to_csv(
    "case_0014_preliminary_metrics.csv",
    index=False
)

print("Preliminary metrics saved successfully.")

In [ ]:
# Select facts F021 through F070
remaining_fact_ids = {
    f"F{i:03d}" for i in range(21, 71)
}

remaining_facts = fact_checklist[
    fact_checklist["fact_id"].isin(remaining_fact_ids)
].to_dict(orient="records")

# Divide into batches of five
fact_batches = [
    remaining_facts[i:i + 5]
    for i in range(0, len(remaining_facts), 5)
]

print("Remaining facts:", len(remaining_facts))
print("Number of batches:", len(fact_batches))

print("\nBatch coverage:")
for number, batch in enumerate(fact_batches, start=1):
    print(
        f"Batch {number}:",
        batch[0]["fact_id"],
        "to",
        batch[-1]["fact_id"]
    )

In [ ]:
import json

# Select the first batch: F021–F025
first_batch = fact_batches[0]

# Retrieve Simple Baseline note segments
simple_sentence_map = note_sentence_maps[
    "Simple Baseline"
]

# Evaluate with automatic recovery
simple_batch3 = evaluate_with_recovery(
    first_batch,
    simple_sentence_map
)

# Display the model's classifications
print("\n=== SIMPLE BASELINE: F021–F025 ===")

for item in simple_batch3["results"]:
    print(
        item["fact_id"],
        "|",
        item["status"],
        "| Evidence:",
        item["evidence_ids"]
    )

print(
    "\nTotal evaluated:",
    len(simple_batch3["results"])
)

print(
    "Unresolved facts:",
    simple_batch3["unresolved_fact_ids"]
)

# Save raw predictions before clinical review
with open(
    "case_0014_simple_f021_f025_raw.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        simple_batch3,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nRaw batch saved successfully.")

In [ ]:
for item in simple_batch3["results"]:

    fact_id = item["fact_id"]

    reference = fact_checklist[
        fact_checklist["fact_id"] == fact_id
    ].iloc[0]

    print("\n" + "=" * 60)
    print("FACT:", fact_id)
    print("REFERENCE:", reference["fact"])
    print("MODEL CLASSIFICATION:", item["status"])
    print("MODEL REASON:", item.get("reason", ""))

    print("\nSELECTED EVIDENCE:")

    ids = item.get("evidence_ids", [])

    if ids:
        for sid in ids:
            print(
                sid, ":",
                simple_sentence_map.get(
                    sid, "INVALID EVIDENCE ID"
                )
            )
    else:
        print("No evidence selected.")

print("\n" + "=" * 60)
print("ALL SIMPLE BASELINE SEGMENTS:")

for sid, text in simple_sentence_map.items():
    print(sid, ":", text)

In [ ]:
import json
from collections import Counter

reviewed_statuses = {
    "F021": "present",
    "F022": "present",
    "F023": "missing",
    "F024": "missing",
    "F025": "present"
}

reviewed_evidence = {
    "F021": ["S010"],
    "F022": ["S010"],
    "F023": [],
    "F024": [],
    "F025": ["S011"]
}

reviewed_results = []

for item in simple_batch3["results"]:
    corrected = item.copy()
    fact_id = item["fact_id"]

    corrected["model_status"] = item["status"]
    corrected["reviewed_status"] = reviewed_statuses[fact_id]
    corrected["reviewed_evidence_ids"] = reviewed_evidence[fact_id]

    if fact_id == "F025":
        corrected["review_reason"] = (
            "S011 explicitly documents hot flashes. "
            "Other symptoms are evaluated as separate facts."
        )
    else:
        corrected["review_reason"] = (
            "Classification checked against the reference "
            "fact and generated note."
        )

    reviewed_results.append(corrected)

reviewed_batch3 = {
    "case_id": "CASE_0014",
    "method": "Simple Baseline",
    "results": reviewed_results
}

with open(
    "case_0014_simple_f021_f025_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reviewed_batch3,
        f,
        indent=2,
        ensure_ascii=False
    )

counts = Counter(
    item["reviewed_status"]
    for item in reviewed_results
)

print("Reviewed facts:", len(reviewed_results))
print("Reviewed classifications:", dict(counts))
print(
    "Changed classifications:",
    sum(
        x["model_status"] != x["reviewed_status"]
        for x in reviewed_results
    )
)
print("Reviewed batch saved successfully.")

In [ ]:
import json
import os

# Batch 1 (F021–F025) is already completed.
# Process batches 2–10 (F026–F070).

for batch_number, batch in enumerate(fact_batches[1:], start=2):

    first_id = batch[0]["fact_id"].lower()
    last_id = batch[-1]["fact_id"].lower()

    filename = (
        f"case_0014_simple_{first_id}_{last_id}_raw.json"
    )

    # Skip batches already saved in a previous run
    if os.path.exists(filename):
        print("Already saved:", filename)
        continue

    print(f"\nEvaluating batch {batch_number}: "
          f"{first_id.upper()}–{last_id.upper()}")

    try:
        result = evaluate_with_recovery(
            batch,
            simple_sentence_map
        )

        # Save results immediately
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        print("Saved:", filename)
        print("Evaluated:", len(result["results"]))
        print(
            "Unresolved:",
            result["unresolved_fact_ids"]
        )

    except Exception as error:
        print("Batch error:", str(error))
        print("Stopping to preserve completed results.")
        break

print("\nAutomated Simple Baseline evaluation finished.")

In [ ]:
import json
import pandas as pd

review_rows = []

for batch in fact_batches[1:]:
    first_id = batch[0]["fact_id"].lower()
    last_id = batch[-1]["fact_id"].lower()

    filename = (
        f"case_0014_simple_{first_id}_{last_id}_raw.json"
    )

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data["results"]:
        fact_id = item["fact_id"]

        reference = fact_checklist.loc[
            fact_checklist["fact_id"] == fact_id
        ].iloc[0]

        ids = item.get("evidence_ids", [])

        evidence_text = " | ".join(
            f"{sid}: {simple_sentence_map.get(sid, 'INVALID ID')}"
            for sid in ids
        )

        review_rows.append({
            "fact_id": fact_id,
            "section": reference["section"],
            "reference_fact": reference["fact"],
            "model_status": item["status"],
            "evidence_ids": ", ".join(ids),
            "evidence_text": evidence_text,
            "model_reason": item.get("reason", ""),
            "reviewed_status": "",
            "review_notes": ""
        })

simple_review_df = pd.DataFrame(review_rows)
simple_review_df = simple_review_df.sort_values(
    "fact_id"
).reset_index(drop=True)

assert len(simple_review_df) == 45
assert simple_review_df["fact_id"].nunique() == 45

simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

print("Facts in review table:", len(simple_review_df))
print("Unique fact IDs:", simple_review_df["fact_id"].nunique())
print("Review sheet saved successfully.")

display(simple_review_df.head(5))

In [ ]:
# Identify evaluations referencing the note heading
heading_evidence = simple_review_df[
    simple_review_df["evidence_ids"].fillna("")
    .str.split(", ")
    .apply(lambda ids: "S001" in ids)
]

print("Total evaluated facts:", len(simple_review_df))
print("Facts citing S001:", len(heading_evidence))

print("\nClassifications citing S001:")
print(
    heading_evidence["model_status"]
    .value_counts()
    .to_string()
)

print("\nAffected fact IDs:")
print(heading_evidence["fact_id"].tolist())

print("\nOther evidence IDs used:")
print(
    simple_review_df["evidence_ids"]
    .value_counts()
    .to_string()
)

In [ ]:
# Facts F026–F030 concern review-of-systems findings.
# Display their reference facts alongside the actual ROS text.

ros_facts = simple_review_df[
    simple_review_df["fact_id"].isin([
        "F026", "F027", "F028", "F029", "F030"
    ])
]

print("REFERENCE FACTS:")
for _, row in ros_facts.iterrows():
    print(
        row["fact_id"],
        "|",
        row["reference_fact"]
    )

print("\nACTUAL GENERATED NOTE EVIDENCE:")
print("S011:", simple_sentence_map["S011"])

print("\nINCORRECTLY SELECTED EVIDENCE:")
print("S001:", simple_sentence_map["S001"])

In [ ]:
# Create a copy so original predictions remain unchanged
simple_review_df = simple_review_df.copy()

corrected_ids = [
    "F026", "F027", "F028", "F029", "F030"
]

mask = simple_review_df["fact_id"].isin(corrected_ids)

# Record reviewed classifications and evidence
simple_review_df.loc[
    mask, "reviewed_status"
] = "present"

simple_review_df.loc[
    mask, "reviewed_evidence_ids"
] = "S011"

simple_review_df.loc[
    mask, "review_notes"
] = (
    "The reference fact is explicitly documented "
    "in S011. The model incorrectly cited S001."
)

# Save the updated review sheet
simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

print("Reviewed facts:", mask.sum())

print("\nCorrected evidence:")
print(
    simple_review_df.loc[
        mask,
        [
            "fact_id",
            "model_status",
            "reviewed_status",
            "evidence_ids",
            "reviewed_evidence_ids"
        ]
    ].to_string(index=False)
)

print("\nUpdated review sheet saved successfully.")

In [ ]:
# Find the remaining facts that cite S001
remaining_s001 = simple_review_df[
    simple_review_df["evidence_ids"].fillna("")
    .str.split(", ")
    .apply(lambda ids: "S001" in ids)
    & ~simple_review_df["fact_id"].isin([
        "F026", "F027", "F028", "F029", "F030"
    ])
]

print("Remaining facts citing S001:", len(remaining_s001))

for _, row in remaining_s001.iterrows():
    print("\n" + "=" * 55)
    print("FACT:", row["fact_id"])
    print("SECTION:", row["section"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 55)
print("RELEVANT NOTE SEGMENTS:")

for sid in ["S011", "S012", "S014", "S015", "S016"]:
    print("\n", sid, ":", simple_sentence_map[sid])

In [ ]:
# Six facts with unsupported evidence
incorrect_fact_ids = [
    "F041", "F042", "F043",
    "F044", "F056", "F060"
]

mask = simple_review_df["fact_id"].isin(
    incorrect_fact_ids
)

# Preserve original model predictions
# and record independently reviewed corrections
simple_review_df.loc[
    mask, "reviewed_status"
] = "missing"

simple_review_df.loc[
    mask, "reviewed_evidence_ids"
] = ""

simple_review_df.loc[
    mask, "review_notes"
] = (
    "The specific physical examination finding "
    "is absent from the generated note. "
    "S001 is only a heading and provides no evidence."
)

# Save updated review sheet
simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

print("Corrected facts:", mask.sum())

print("\nReviewed classifications:")
print(
    simple_review_df.loc[
        mask,
        [
            "fact_id",
            "model_status",
            "reviewed_status",
            "evidence_ids",
            "reviewed_evidence_ids"
        ]
    ].to_string(index=False)
)

print("\nReview sheet updated successfully.")

In [ ]:
# Select facts that do not yet have a reviewed classification
unreviewed_df = simple_review_df[
    simple_review_df["reviewed_status"].fillna("").str.strip() == ""
].copy()

print("Total facts:", len(simple_review_df))
print(
    "Reviewed facts:",
    len(simple_review_df) - len(unreviewed_df)
)
print("Unreviewed facts:", len(unreviewed_df))

print("\nUNREVIEWED FACTS BY CLINICAL SECTION:")
print(
    unreviewed_df["section"]
    .value_counts()
    .to_string()
)

print("\nUNREVIEWED FACT IDS:")
for section, group in unreviewed_df.groupby(
    "section", sort=False
):
    print(
        section + ":",
        ", ".join(group["fact_id"].tolist())
    )

In [ ]:
# Select unreviewed Review of Systems facts
ros_review = unreviewed_df[
    unreviewed_df["section"] == "REVIEW OF SYSTEMS"
].copy()

print("REVIEW OF SYSTEMS FACTS:")
print("=" * 65)

for _, row in ros_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("ACTUAL GENERATED NOTE:")

print("S011:", simple_sentence_map["S011"])

In [ ]:
# Reviewed classifications for F031–F040
reviewed_ros_statuses = {
    "F031": "present",
    "F032": "present",
    "F033": "present",
    "F034": "missing",
    "F035": "missing",
    "F036": "missing",
    "F037": "missing",
    "F038": "missing",
    "F039": "missing",
    "F040": "missing"
}

mask = simple_review_df["fact_id"].isin(
    reviewed_ros_statuses
)

for fact_id, status in reviewed_ros_statuses.items():

    row_mask = simple_review_df["fact_id"] == fact_id

    simple_review_df.loc[
        row_mask, "reviewed_status"
    ] = status

    simple_review_df.loc[
        row_mask, "reviewed_evidence_ids"
    ] = "S011" if status == "present" else ""

    simple_review_df.loc[
        row_mask, "review_notes"
    ] = (
        "Positive finding explicitly documented in S011."
        if status == "present"
        else "Documented source negation absent from generated note."
    )

# Save updated review sheet
simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

print("Newly reviewed facts:", mask.sum())

print("\nReviewed classifications:")
print(
    simple_review_df.loc[
        mask, "reviewed_status"
    ].value_counts().to_string()
)

remaining = (
    simple_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\nFacts still requiring review:", remaining)
print("Updated review sheet saved successfully.")

In [ ]:
# Select physical examination facts still requiring review
physical_review = simple_review_df[
    (simple_review_df["section"] == "PHYSICAL EXAMINATION")
    & (
        simple_review_df["reviewed_status"]
        .fillna("")
        .str.strip() == ""
    )
].copy()

print("PHYSICAL EXAMINATION FACTS:", len(physical_review))
print("=" * 65)

for _, row in physical_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("ACTUAL GENERATED NOTE EVIDENCE:")
print("S012:", simple_sentence_map["S012"])

In [ ]:
# All remaining physical examination facts
physical_ids = physical_review["fact_id"].tolist()

assert len(physical_ids) == 16
assert "F053" in physical_ids

for fact_id in physical_ids:

    mask = simple_review_df["fact_id"] == fact_id

    if fact_id == "F053":
        simple_review_df.loc[
            mask, "reviewed_status"
        ] = "present"

        simple_review_df.loc[
            mask, "reviewed_evidence_ids"
        ] = "S012"

        simple_review_df.loc[
            mask, "review_notes"
        ] = (
            "The note explicitly documents +1 "
            "lower-extremity edema in S012."
        )

    else:
        simple_review_df.loc[
            mask, "reviewed_status"
        ] = "missing"

        simple_review_df.loc[
            mask, "reviewed_evidence_ids"
        ] = ""

        simple_review_df.loc[
            mask, "review_notes"
        ] = (
            "The specific examination finding "
            "is not documented in the generated note."
        )

# Preserve reviewed classifications
simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

# Check results
physical_results = simple_review_df[
    simple_review_df["fact_id"].isin(physical_ids)
]

changed = (
    physical_results["model_status"]
    != physical_results["reviewed_status"]
).sum()

remaining = (
    simple_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Physical examination facts reviewed:", len(physical_results))
print(
    "Reviewed classifications:",
    physical_results["reviewed_status"].value_counts().to_dict()
)
print("Changed classifications:", changed)
print("Facts still requiring review:", remaining)
print("Updated review sheet saved successfully.")

In [ ]:
# Select the eight unreviewed Assessment and Plan facts
final_review = simple_review_df[
    simple_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
].copy()

assert len(final_review) == 8

print("REMAINING FACTS:", len(final_review))
print("=" * 65)

for _, row in final_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("SECTION:", row["section"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("GENERATED NOTE ASSESSMENT AND PLAN:")

for sid in ["S014", "S015", "S016"]:
    print("\n", sid, ":", simple_sentence_map[sid])

In [ ]:
# Reviewed Assessment and Plan classifications
final_statuses = {
    "F063": "present",
    "F064": "present",
    "F065": "partial",
    "F066": "present",
    "F067": "present",
    "F068": "present",
    "F069": "present",
    "F070": "present"
}

final_evidence = {
    "F063": "S002",
    "F064": "S014",
    "F065": "S014, S016",
    "F066": "S014, S016",
    "F067": "S015",
    "F068": "S015",
    "F069": "S015",
    "F070": "S015"
}

for fact_id, status in final_statuses.items():

    mask = simple_review_df["fact_id"] == fact_id

    simple_review_df.loc[
        mask, "reviewed_status"
    ] = status

    simple_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = final_evidence[fact_id]

    simple_review_df.loc[
        mask, "review_notes"
    ] = (
        "Reviewed against the actual generated note. "
        "F065 preserves surgical candidacy but does not "
        "explicitly document patient interest."
        if fact_id == "F065"
        else "Verified against the generated note."
    )

# Save the completed review sheet
simple_review_df.to_csv(
    "case_0014_simple_f026_f070_review_sheet.csv",
    index=False
)

remaining = (
    simple_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

final_rows = simple_review_df[
    simple_review_df["fact_id"].isin(final_statuses)
]

print("Assessment and Plan facts reviewed:", len(final_rows))
print(
    "Changed classifications:",
    (
        final_rows["model_status"]
        != final_rows["reviewed_status"]
    ).sum()
)
print("Facts still requiring review:", remaining)
print("Completed review sheet saved successfully.")

In [ ]:
from collections import Counter

# Verify complete review coverage
assert len(simple_review_df) == 45
assert simple_review_df["fact_id"].nunique() == 45
assert simple_review_df["reviewed_status"].notna().all()
assert (
    simple_review_df["reviewed_status"].str.strip() != ""
).all()

# Calculate reviewed classifications
review_counts = Counter(
    simple_review_df["reviewed_status"]
)

# Count differences between model and reviewed labels
changed_count = (
    simple_review_df["model_status"]
    != simple_review_df["reviewed_status"]
).sum()

print("=== SIMPLE BASELINE: F026–F070 ===")
print("Total reviewed facts:", len(simple_review_df))

for status in [
    "present", "partial", "missing", "contradicted"
]:
    print(status.capitalize() + ":", review_counts[status])

print("Changed model classifications:", changed_count)

print("\nReviewed classification percentages:")
for status in [
    "present", "partial", "missing", "contradicted"
]:
    percentage = (
        review_counts[status] / len(simple_review_df) * 100
    )
    print(f"{status}: {percentage:.2f}%")

print("\nReview summary completed successfully.")

In [ ]:
import os
import json

filename = "case_0014_fact_comparison_corrected.json"

if os.path.exists(filename):

    with open(filename, "r", encoding="utf-8") as f:
        early_results = json.load(f)

    print("File found:", filename)
    print("Top-level data type:", type(early_results).__name__)

    if isinstance(early_results, dict):
        print("Available keys:", list(early_results.keys()))
    elif isinstance(early_results, list):
        print("Number of entries:", len(early_results))

else:
    print("File not found:", filename)

print("\nFile check completed.")

In [ ]:
import os

target_files = [
    "case_0014_fact_comparison_corrected.json",
    "case_0014_fact_comparison_batch2_corrected.json"
]

search_folders = [
    "/content",
    "/content/drive/MyDrive"
]

print("SEARCHING FOR EARLIER RESULTS\n")

for folder in search_folders:
    if not os.path.exists(folder):
        print("Folder unavailable:", folder)
        continue

    for root, dirs, files in os.walk(folder):
        for filename in target_files:
            if filename in files:
                print("FOUND:", os.path.join(root, filename))

print("\nSearch completed.")

In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

print("Google Drive connected successfully.")

In [ ]:
import os

target_files = [
    "case_0014_fact_comparison_corrected.json",
    "case_0014_fact_comparison_batch2_corrected.json"
]

found_files = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for filename in target_files:
        if filename in files:
            full_path = os.path.join(root, filename)
            found_files.append(full_path)
            print("FOUND:", full_path)

print("\nTotal files found:", len(found_files))

if not found_files:
    print("No matching files found in Google Drive.")

In [ ]:
# Check for the earlier corrected evaluation
if "corrected_comparison" in globals():
    print("FOUND: corrected_comparison")
    print("Data type:", type(corrected_comparison).__name__)

    if isinstance(corrected_comparison, dict):
        print("Available keys:", list(corrected_comparison.keys()))
    elif isinstance(corrected_comparison, list):
        print("Number of entries:", len(corrected_comparison))
else:
    print("Earlier corrected results are not in memory.")

# Confirm that the current review table is intact
print(
    "\nCurrent Simple Baseline review rows:",
    len(simple_review_df)
)

In [ ]:
# Retrieve the first ten clinical facts
first_ten = fact_checklist[
    fact_checklist["fact_id"].isin(
        [f"F{i:03d}" for i in range(1, 11)]
    )
].copy()

assert len(first_ten) == 10

print("REFERENCE FACTS F001–F010")
print("=" * 65)

for _, row in first_ten.iterrows():
    print("\nFACT:", row["fact_id"])
    print("SECTION:", row["section"])
    print("REFERENCE:", row["fact"])

print("\n" + "=" * 65)
print("SIMPLE BASELINE GENERATED NOTE")
print("=" * 65)

for sid, text in simple_sentence_map.items():
    print(sid, ":", text)

In [ ]:
import json
from collections import Counter

reviewed_f001_f010 = {
    "F001": ("present", ["S002", "S003"]),
    "F002": ("present", ["S002", "S003"]),
    "F003": ("present", ["S002", "S003"]),
    "F004": ("present", ["S004"]),
    "F005": ("missing", []),
    "F006": ("present", ["S005"]),
    "F007": ("present", ["S005"]),
    "F008": ("present", ["S005"]),
    "F009": ("partial", ["S003", "S006"]),
    "F010": ("partial", ["S003", "S006"])
}

reconstructed_results = []

for _, row in first_ten.iterrows():
    fact_id = row["fact_id"]
    status, evidence_ids = reviewed_f001_f010[fact_id]

    reconstructed_results.append({
        "fact_id": fact_id,
        "section": row["section"],
        "reference_fact": row["fact"],
        "reviewed_status": status,
        "reviewed_evidence_ids": evidence_ids,
        "review_origin": "reconstructed_manual_review"
    })

assert len(reconstructed_results) == 10
assert len({
    item["fact_id"] for item in reconstructed_results
}) == 10

output = {
    "case_id": "CASE_0014",
    "method": "Simple Baseline",
    "results": reconstructed_results
}

with open(
    "case_0014_simple_f001_f010_reconstructed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

counts = Counter(
    item["reviewed_status"]
    for item in reconstructed_results
)

print("Reconstructed facts:", len(reconstructed_results))
print("Reviewed classifications:", dict(counts))
print("Reconstructed review saved successfully.")

In [ ]:
import json
import pandas as pd
from collections import Counter

all_reviewed = []

# Group 1: F001–F010
with open(
    "case_0014_simple_f001_f010_reconstructed.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)
    all_reviewed.extend(data["results"])

# Group 2: F011–F020
with open(
    "case_0014_simple_baseline_reviewed.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)
    all_reviewed.extend(data["results"])

# Group 3: F021–F025
with open(
    "case_0014_simple_f021_f025_reviewed.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)
    all_reviewed.extend(data["results"])

# Group 4: F026–F070
for _, row in simple_review_df.iterrows():
    all_reviewed.append({
        "fact_id": row["fact_id"],
        "section": row["section"],
        "reference_fact": row["reference_fact"],
        "model_status": row["model_status"],
        "reviewed_status": row["reviewed_status"],
        "reviewed_evidence_ids": (
            str(row["reviewed_evidence_ids"]).split(", ")
            if pd.notna(row["reviewed_evidence_ids"])
            and str(row["reviewed_evidence_ids"]).strip()
            else []
        ),
        "review_origin": "manual_review"
    })

# Verify complete fact coverage
expected_ids = {
    f"F{i:03d}" for i in range(1, 71)
}

actual_ids = [
    item["fact_id"] for item in all_reviewed
]

assert len(all_reviewed) == 70
assert len(set(actual_ids)) == 70
assert set(actual_ids) == expected_ids

# Sort results by fact ID
all_reviewed.sort(key=lambda item: item["fact_id"])

# Save consolidated review
consolidated_simple = {
    "case_id": "CASE_0014",
    "method": "Simple Baseline",
    "total_facts": 70,
    "results": all_reviewed
}

with open(
    "case_0014_simple_all70_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        consolidated_simple,
        f,
        indent=2,
        ensure_ascii=False
    )

counts = Counter(
    item["reviewed_status"]
    for item in all_reviewed
)

print("Total consolidated facts:", len(all_reviewed))
print("Unique fact IDs:", len(set(actual_ids)))
print("Reviewed classifications:", dict(counts))
print("Consolidated file saved successfully.")

In [ ]:
# Select facts not yet evaluated for Structured Prompt
structured_pending_ids = {
    f"F{i:03d}" for i in list(range(1, 11)) + list(range(21, 71))
}

structured_pending_facts = fact_checklist[
    fact_checklist["fact_id"].isin(structured_pending_ids)
].sort_values("fact_id").to_dict(orient="records")

# Create batches of five facts
structured_batches = [
    structured_pending_facts[i:i + 5]
    for i in range(0, len(structured_pending_facts), 5)
]

# Verify coverage
assert len(structured_pending_facts) == 60
assert len(structured_batches) == 12

print("Facts requiring evaluation:", len(structured_pending_facts))
print("Number of batches:", len(structured_batches))

print("\nBatch coverage:")
for number, batch in enumerate(structured_batches, 1):
    print(
        f"Batch {number}:",
        batch[0]["fact_id"],
        "to",
        batch[-1]["fact_id"]
    )

print("\nStructured Prompt batches prepared successfully.")

In [ ]:
import json
import os

structured_sentence_map = note_sentence_maps[
    "Structured Prompt"
]

for batch_number, batch in enumerate(
    structured_batches, start=1
):
    first_id = batch[0]["fact_id"].lower()
    last_id = batch[-1]["fact_id"].lower()

    filename = (
        f"case_0014_structured_{first_id}_{last_id}_raw.json"
    )

    if os.path.exists(filename):
        print("Already saved:", filename)
        continue

    print(
        f"\nEvaluating batch {batch_number}: "
        f"{first_id.upper()}–{last_id.upper()}"
    )

    try:
        result = evaluate_with_recovery(
            batch,
            structured_sentence_map
        )

        # Save this batch immediately
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        print("Saved:", filename)
        print("Evaluated:", len(result["results"]))
        print(
            "Unresolved:",
            result["unresolved_fact_ids"]
        )

    except Exception as error:
        print("Batch error:", str(error))
        print("Stopping after preserving completed batches.")
        break

print("\nStructured Prompt evaluation run finished.")

In [ ]:
import json
import pandas as pd

structured_review_rows = []

for batch in structured_batches:

    first_id = batch[0]["fact_id"].lower()
    last_id = batch[-1]["fact_id"].lower()

    filename = (
        f"case_0014_structured_{first_id}_{last_id}_raw.json"
    )

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert data["unresolved_fact_ids"] == []

    for item in data["results"]:

        fact_id = item["fact_id"]

        reference = fact_checklist.loc[
            fact_checklist["fact_id"] == fact_id
        ].iloc[0]

        ids = item.get("evidence_ids", [])

        evidence_text = " | ".join(
            f"{sid}: {structured_sentence_map.get(sid, 'INVALID ID')}"
            for sid in ids
        )

        structured_review_rows.append({
            "fact_id": fact_id,
            "section": reference["section"],
            "reference_fact": reference["fact"],
            "model_status": item["status"],
            "evidence_ids": ", ".join(ids),
            "evidence_text": evidence_text,
            "model_reason": item.get("reason", ""),
            "reviewed_status": "",
            "reviewed_evidence_ids": "",
            "review_notes": ""
        })

structured_review_df = pd.DataFrame(
    structured_review_rows
).sort_values("fact_id").reset_index(drop=True)

expected_ids = {
    f"F{i:03d}"
    for i in list(range(1, 11)) + list(range(21, 71))
}

assert len(structured_review_df) == 60
assert structured_review_df["fact_id"].nunique() == 60
assert set(structured_review_df["fact_id"]) == expected_ids

structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

print("Facts in review sheet:", len(structured_review_df))
print(
    "Unique fact IDs:",
    structured_review_df["fact_id"].nunique()
)
print(
    "Clinical sections:",
    structured_review_df["section"].nunique()
)
print("Structured Prompt review sheet saved successfully.")

display(structured_review_df.head(5))

In [ ]:
# Select F001–F010
first_ten_structured = structured_review_df[
    structured_review_df["fact_id"].isin(
        [f"F{i:03d}" for i in range(1, 11)]
    )
].copy()

print("STRUCTURED PROMPT: F001–F010")
print("=" * 65)

for _, row in first_ten_structured.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("RELEVANT GENERATED NOTE SEGMENTS")

for sid in ["S004", "S005", "S006", "S007"]:
    print("\n", sid, ":", structured_sentence_map[sid])

In [ ]:
reviewed_statuses = {
    "F001": "present",
    "F002": "present",
    "F003": "present",
    "F004": "present",
    "F005": "missing",
    "F006": "present",
    "F007": "present",
    "F008": "present",
    "F009": "partial",
    "F010": "partial"
}

reviewed_evidence = {
    "F001": "S004",
    "F002": "S004",
    "F003": "S004",
    "F004": "S005",
    "F005": "",
    "F006": "S006",
    "F007": "S006",
    "F008": "S006",
    "F009": "S004, S007",
    "F010": "S004, S007"
}

for fact_id, status in reviewed_statuses.items():

    mask = structured_review_df["fact_id"] == fact_id

    structured_review_df.loc[
        mask, "reviewed_status"
    ] = status

    structured_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = reviewed_evidence[fact_id]

    structured_review_df.loc[
        mask, "review_notes"
    ] = (
        "The note omits denial of cardiac disease "
        "and prior cancer."
        if fact_id == "F005"
        else "Reviewed against the reference fact and note."
    )

structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

reviewed_rows = structured_review_df[
    structured_review_df["fact_id"].isin(reviewed_statuses)
]

print("Facts reviewed:", len(reviewed_rows))
print(
    "Reviewed classifications:",
    reviewed_rows["reviewed_status"].value_counts().to_dict()
)
print(
    "Changed classifications:",
    (
        reviewed_rows["model_status"]
        != reviewed_rows["reviewed_status"]
    ).sum()
)
print(
    "Facts still requiring review:",
    structured_review_df["reviewed_status"]
    .fillna("").str.strip().eq("").sum()
)
print("Review sheet saved successfully.")

In [ ]:
# Select F021–F025
next_review = structured_review_df[
    structured_review_df["fact_id"].isin(
        [f"F{i:03d}" for i in range(21, 26)]
    )
].copy()

print("STRUCTURED PROMPT: F021–F025")
print("=" * 65)

for _, row in next_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("RELEVANT GENERATED NOTE SEGMENTS")

for sid, text in structured_sentence_map.items():
    if any(
        term in text.lower()
        for term in ["family history", "diabetes", "hot flashes"]
    ):
        print(sid, ":", text)

In [ ]:
reviewed_statuses = {
    "F021": "partial",
    "F022": "present",
    "F023": "missing",
    "F024": "missing",
    "F025": "present"
}

reviewed_evidence = {
    "F021": "S010",
    "F022": "S010",
    "F023": "",
    "F024": "",
    "F025": "S011"
}

for fact_id, status in reviewed_statuses.items():

    mask = structured_review_df["fact_id"] == fact_id

    structured_review_df.loc[
        mask, "reviewed_status"
    ] = status

    structured_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = reviewed_evidence[fact_id]

    structured_review_df.loc[
        mask, "review_notes"
    ] = (
        "The family history of type II diabetes is "
        "documented, but the maternal relationship "
        "is not specified."
        if fact_id == "F021"
        else "Verified against the generated note."
    )

structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

reviewed_rows = structured_review_df[
    structured_review_df["fact_id"].isin(reviewed_statuses)
]

print("Facts reviewed:", len(reviewed_rows))
print(
    "Reviewed classifications:",
    reviewed_rows["reviewed_status"].value_counts().to_dict()
)
print(
    "Changed classifications:",
    (
        reviewed_rows["model_status"]
        != reviewed_rows["reviewed_status"]
    ).sum()
)
print(
    "Facts still requiring review:",
    structured_review_df["reviewed_status"]
    .fillna("").str.strip().eq("").sum()
)
print("Review sheet saved successfully.")

In [ ]:
# Select Review of Systems facts F026–F040
ros_review = structured_review_df[
    structured_review_df["fact_id"].isin(
        [f"F{i:03d}" for i in range(26, 41)]
    )
].copy()

assert len(ros_review) == 15

print("STRUCTURED PROMPT: REVIEW OF SYSTEMS")
print("=" * 65)

for _, row in ros_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("ACTUAL GENERATED NOTE EVIDENCE")

print("S011:", structured_sentence_map["S011"])

In [ ]:
reviewed_ros_statuses = {
    "F026": "present",
    "F027": "partial",
    "F028": "partial",
    "F029": "present",
    "F030": "present",
    "F031": "present",
    "F032": "partial",
    "F033": "present",
    "F034": "missing",
    "F035": "missing",
    "F036": "missing",
    "F037": "missing",
    "F038": "missing",
    "F039": "missing",
    "F040": "missing"
}

for fact_id, status in reviewed_ros_statuses.items():

    mask = structured_review_df["fact_id"] == fact_id

    structured_review_df.loc[
        mask, "reviewed_status"
    ] = status

    structured_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = "S011" if status != "missing" else ""

    structured_review_df.loc[
        mask, "review_notes"
    ] = (
        "The symptom is documented, but a reference "
        "qualifier is omitted."
        if status == "partial"
        else (
            "The positive finding is documented in S011."
            if status == "present"
            else "The source negation is absent from the note."
        )
    )

# Save the updated review sheet
structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

ros_rows = structured_review_df[
    structured_review_df["fact_id"].isin(
        reviewed_ros_statuses
    )
]

print("ROS facts reviewed:", len(ros_rows))
print(
    "Reviewed classifications:",
    ros_rows["reviewed_status"].value_counts().to_dict()
)
print(
    "Changed classifications:",
    (
        ros_rows["model_status"] != ros_rows["reviewed_status"]
    ).sum()
)
print(
    "Facts still requiring review:",
    structured_review_df["reviewed_status"]
    .fillna("").str.strip().eq("").sum()
)
print("Review sheet saved successfully.")

In [ ]:
# Select physical examination facts F041–F062
physical_review = structured_review_df[
    structured_review_df["fact_id"].isin(
        [f"F{i:03d}" for i in range(41, 63)]
    )
].copy()

assert len(physical_review) == 22

print("STRUCTURED PROMPT: PHYSICAL EXAMINATION")
print("=" * 65)

for _, row in physical_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("ACTUAL GENERATED NOTE: FINDINGS")

for sid, text in structured_sentence_map.items():
    if (
        "vital signs" in text.lower()
        or "exam" in text.lower()
        or "edema" in text.lower()
    ):
        print(sid, ":", text)

In [ ]:
# Reviewed physical examination classifications
physical_statuses = {
    f"F{i:03d}": "missing"
    for i in range(41, 63)
}

physical_statuses.update({
    "F047": "partial",
    "F053": "present",
    "F056": "partial",
    "F062": "partial"
})

physical_evidence = {
    "F047": "S015",
    "F053": "S018",
    "F056": "S014",
    "F062": "S017"
}

for fact_id, status in physical_statuses.items():

    mask = structured_review_df["fact_id"] == fact_id

    structured_review_df.loc[
        mask, "reviewed_status"
    ] = status

    structured_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = physical_evidence.get(fact_id, "")

    structured_review_df.loc[
        mask, "review_notes"
    ] = (
        "Specific finding explicitly documented."
        if status == "present"
        else (
            "General examination summary does not "
            "preserve the complete reference finding."
            if status == "partial"
            else "Specific examination finding is absent."
        )
    )

# Save the reviewed results
structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

physical_rows = structured_review_df[
    structured_review_df["fact_id"].isin(physical_statuses)
]

print("Physical examination facts reviewed:", len(physical_rows))
print(
    "Reviewed classifications:",
    physical_rows["reviewed_status"].value_counts().to_dict()
)
print(
    "Changed classifications:",
    (
        physical_rows["model_status"]
        != physical_rows["reviewed_status"]
    ).sum()
)
print(
    "Facts still requiring review:",
    structured_review_df["reviewed_status"]
    .fillna("").str.strip().eq("").sum()
)
print("Review sheet saved successfully.")

In [ ]:
# Select the eight unreviewed facts
final_structured_review = structured_review_df[
    structured_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
].copy()

assert len(final_structured_review) == 8

print("STRUCTURED PROMPT: FINAL EIGHT FACTS")
print("=" * 65)

for _, row in final_structured_review.iterrows():
    print("\nFACT:", row["fact_id"])
    print("SECTION:", row["section"])
    print("REFERENCE:", row["reference_fact"])
    print("MODEL STATUS:", row["model_status"])
    print("MODEL EVIDENCE:", row["evidence_ids"])
    print("MODEL REASON:", row["model_reason"])

print("\n" + "=" * 65)
print("ACTUAL GENERATED NOTE: ASSESSMENT AND PLAN")

for sid, text in structured_sentence_map.items():
    if (
        sid in ["S001", "S002", "S003"]
        or int(sid[1:]) >= 19
    ):
        print(sid, ":", text)

In [ ]:
final_statuses = {
    f"F{i:03d}": "present"
    for i in range(63, 71)
}

final_evidence = {
    "F063": "S002",
    "F064": "S002",
    "F065": "S002, S021",
    "F066": "S021, S022",
    "F067": "S026",
    "F068": "S024",
    "F069": "S025",
    "F070": "S025"
}

for fact_id, status in final_statuses.items():

    mask = structured_review_df["fact_id"] == fact_id

    structured_review_df.loc[
        mask, "reviewed_status"
    ] = status

    structured_review_df.loc[
        mask, "reviewed_evidence_ids"
    ] = final_evidence[fact_id]

    structured_review_df.loc[
        mask, "review_notes"
    ] = (
        "The complete reference fact is documented "
        "in the reviewed note segment or segments."
    )

# Save the updated review sheet
structured_review_df.to_csv(
    "case_0014_structured_pending_review_sheet.csv",
    index=False
)

final_rows = structured_review_df[
    structured_review_df["fact_id"].isin(final_statuses)
]

changed = (
    final_rows["model_status"]
    != final_rows["reviewed_status"]
).sum()

remaining = (
    structured_review_df["reviewed_status"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Assessment and Plan facts reviewed:", len(final_rows))
print(
    "Reviewed classifications:",
    final_rows["reviewed_status"].value_counts().to_dict()
)
print("Changed classifications:", changed)
print("Facts still requiring review:", remaining)
print("Review sheet saved successfully.")

In [ ]:
import json
import pandas as pd
from collections import Counter

all_structured_results = []

# Load previously reviewed F011–F020
with open(
    "case_0014_structured_f011_f020_reviewed.json",
    "r",
    encoding="utf-8"
) as f:
    earlier_data = json.load(f)

all_structured_results.extend(earlier_data["results"])

# Add the 60 newly reviewed facts
for _, row in structured_review_df.iterrows():

    evidence = row["reviewed_evidence_ids"]

    if pd.isna(evidence) or not str(evidence).strip():
        evidence_ids = []
    else:
        evidence_ids = [
            sid.strip()
            for sid in str(evidence).split(",")
        ]

    all_structured_results.append({
        "fact_id": row["fact_id"],
        "section": row["section"],
        "reference_fact": row["reference_fact"],
        "model_status": row["model_status"],
        "reviewed_status": row["reviewed_status"],
        "reviewed_evidence_ids": evidence_ids,
        "review_notes": row["review_notes"]
    })

# Verify complete coverage
expected_ids = {
    f"F{i:03d}" for i in range(1, 71)
}

actual_ids = [
    item["fact_id"]
    for item in all_structured_results
]

assert len(actual_ids) == 70
assert len(set(actual_ids)) == 70
assert set(actual_ids) == expected_ids

allowed_statuses = {
    "present", "partial", "missing", "contradicted"
}

assert all(
    item["reviewed_status"] in allowed_statuses
    for item in all_structured_results
)

# Sort by fact ID
all_structured_results.sort(
    key=lambda item: item["fact_id"]
)

# Save consolidated file
consolidated_structured = {
    "case_id": "CASE_0014",
    "method": "Structured Prompt",
    "total_facts": 70,
    "results": all_structured_results
}

with open(
    "case_0014_structured_all70_reviewed.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        consolidated_structured,
        f,
        indent=2,
        ensure_ascii=False
    )

counts = Counter(
    item["reviewed_status"]
    for item in all_structured_results
)

print("Total consolidated facts:", len(all_structured_results))
print("Unique fact IDs:", len(set(actual_ids)))
print("Reviewed classifications:", dict(counts))
print("Consolidated file saved successfully.")

In [ ]:
import json
import pandas as pd
from collections import Counter

review_files = {
    "Simple Baseline": "case_0014_simple_all70_reviewed.json",
    "Structured Prompt": "case_0014_structured_all70_reviewed.json"
}

comparison_metrics = []

for method, filename in review_files.items():

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = data["results"]

    ids = [item["fact_id"] for item in results]

    assert len(results) == 70
    assert len(set(ids)) == 70

    counts = Counter(
        item["reviewed_status"] for item in results
    )

    total = len(results)

    comparison_metrics.append({
        "method": method,
        "total_facts": total,
        "present": counts["present"],
        "partial": counts["partial"],
        "missing": counts["missing"],
        "contradicted": counts["contradicted"],
        "exact_preservation_pct": round(
            100 * counts["present"] / total, 2
        ),
        "partial_preservation_pct": round(
            100 * counts["partial"] / total, 2
        ),
        "omission_pct": round(
            100 * counts["missing"] / total, 2
        ),
        "any_preservation_pct": round(
            100 * (
                counts["present"] + counts["partial"]
            ) / total, 2
        )
    })

full_metrics_df = pd.DataFrame(comparison_metrics)

display(full_metrics_df)

full_metrics_df.to_csv(
    "case_0014_full70_comparison_metrics.csv",
    index=False
)

print("\nComplete comparison metrics saved successfully.")

In [ ]:
import json
import pandas as pd

records = []

review_files = {
    "Simple Baseline": "case_0014_simple_all70_reviewed.json",
    "Structured Prompt": "case_0014_structured_all70_reviewed.json"
}

# Use the reference checklist to assign consistent sections
section_lookup = fact_checklist.set_index("fact_id")["section"].to_dict()

for method, filename in review_files.items():

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data["results"]:
        records.append({
            "method": method,
            "fact_id": item["fact_id"],
            "section": section_lookup[item["fact_id"]],
            "status": item["reviewed_status"]
        })

section_results_df = pd.DataFrame(records)

# Count classifications within each section and method
section_summary = (
    section_results_df
    .groupby(["section", "method", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for status in [
    "present", "partial", "missing", "contradicted"
]:
    if status not in section_summary.columns:
        section_summary[status] = 0

section_summary["total"] = section_summary[
    ["present", "partial", "missing", "contradicted"]
].sum(axis=1)

section_summary["any_preservation_pct"] = (
    100 * (
        section_summary["present"]
        + section_summary["partial"]
    ) / section_summary["total"]
).round(2)

section_summary = section_summary[
    [
        "section", "method", "total",
        "present", "partial", "missing",
        "contradicted", "any_preservation_pct"
    ]
].sort_values(["section", "method"])

display(section_summary)

section_summary.to_csv(
    "case_0014_section_comparison.csv",
    index=False
)

print("\nSection-level comparison saved successfully.")

In [ ]:
import pandas as pd

# Convert reviewed results into a paired comparison
paired_statuses = section_results_df.pivot(
    index="fact_id",
    columns="method",
    values="status"
).reset_index()

# Identify facts missing from both notes
shared_missing = paired_statuses[
    (paired_statuses["Simple Baseline"] == "missing")
    & (paired_statuses["Structured Prompt"] == "missing")
].copy()

# Add reference fact details
shared_missing = shared_missing.merge(
    fact_checklist[
        ["fact_id", "section", "fact"]
    ],
    on="fact_id",
    how="left"
)

shared_missing = shared_missing[
    ["fact_id", "section", "fact"]
].sort_values("fact_id")

print("Facts omitted by both methods:", len(shared_missing))

print("\nShared omissions by section:")
print(
    shared_missing["section"]
    .value_counts()
    .to_string()
)

shared_missing.to_csv(
    "case_0014_shared_omissions.csv",
    index=False
)

print("\nShared omission list saved successfully.")

In [ ]:
# Attach original transcription evidence
shared_omissions_evidence = shared_missing.merge(
    fact_checklist[["fact_id", "source_evidence"]],
    on="fact_id",
    how="left",
    validate="one_to_one"
)

# Check for missing or empty evidence
evidence_text = (
    shared_omissions_evidence["source_evidence"]
    .fillna("")
    .astype(str)
    .str.strip()
)

has_evidence = evidence_text.ne("")

print("Shared omission facts:", len(shared_omissions_evidence))
print("Facts with source evidence:", has_evidence.sum())
print("Facts without source evidence:", (~has_evidence).sum())

print("\nFIRST FIVE EXAMPLES:")

for _, row in shared_omissions_evidence.head(5).iterrows():
    print("\nFact:", row["fact_id"])
    print("Reference:", row["fact"])
    print("Source evidence:", row["source_evidence"])

# Save for the evidence-alignment experiment
shared_omissions_evidence.to_csv(
    "case_0014_shared_omissions_with_evidence.csv",
    index=False
)

print("\nEvidence verification file saved successfully.")

In [ ]:
import re
import pandas as pd

# Retrieve the original transcription for CASE_0014
original_transcription = comparison_df.loc[
    comparison_df["case_id"] == "CASE_0014",
    "transcription"
].iloc[0]

def normalize_clinical_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

normalized_transcription = normalize_clinical_text(
    original_transcription
)

# Verify each excerpt against the transcription
shared_omissions_evidence = (
    shared_omissions_evidence.copy()
)

shared_omissions_evidence["evidence_verified"] = (
    shared_omissions_evidence["source_evidence"]
    .apply(
        lambda evidence:
        normalize_clinical_text(evidence)
        in normalized_transcription
    )
)

verified = shared_omissions_evidence[
    "evidence_verified"
].sum()

print("Total evidence excerpts:", len(shared_omissions_evidence))
print("Exact matches after whitespace normalization:", verified)
print(
    "Excerpts requiring further verification:",
    len(shared_omissions_evidence) - verified
)

print("\nUNMATCHED FACT IDS:")
print(
    shared_omissions_evidence.loc[
        ~shared_omissions_evidence["evidence_verified"],
        "fact_id"
    ].tolist()
)

shared_omissions_evidence.to_csv(
    "case_0014_shared_omissions_evidence_verified.csv",
    index=False
)

print("\nEvidence verification results saved successfully.")

In [ ]:
import pandas as pd

# Work only with verified source excerpts
evidence_registry = (
    shared_omissions_evidence[
        shared_omissions_evidence["evidence_verified"]
    ][
        ["fact_id", "section", "fact", "source_evidence"]
    ]
    .sort_values("fact_id")
    .reset_index(drop=True)
    .copy()
)

# Assign stable evidence identifiers
evidence_registry.insert(
    0,
    "evidence_id",
    [
        f"E{i:03d}"
        for i in range(1, len(evidence_registry) + 1)
    ]
)

# Check registry integrity
assert len(evidence_registry) == 29
assert evidence_registry["evidence_id"].is_unique
assert evidence_registry["fact_id"].is_unique

# Save the registry
evidence_registry.to_csv(
    "case_0014_source_evidence_registry.csv",
    index=False
)

print("Evidence records:", len(evidence_registry))
print("Unique evidence IDs:", evidence_registry["evidence_id"].nunique())
print("Unique fact IDs:", evidence_registry["fact_id"].nunique())

print("\nFIRST FIVE RECORDS:")
display(evidence_registry.head(5))

print("\nSource evidence registry saved successfully.")

In [ ]:
import json

audit_findings = []

for _, row in evidence_registry.iterrows():
    audit_findings.append({
        "case_id": "CASE_0014",
        "fact_id": row["fact_id"],
        "clinical_section": row["section"],
        "omitted_fact": row["fact"],
        "source_evidence_id": row["evidence_id"],
        "source_evidence": row["source_evidence"],
        "simple_baseline_status": "missing",
        "structured_prompt_status": "missing",
        "source_text_verified": True,
        "finding_type": "shared_omission"
    })

audit_report = {
    "case_id": "CASE_0014",
    "report_type": "reviewed_shared_omissions",
    "evidence_source": "verified_transcription_excerpts",
    "detection_method": "reviewed_fact_comparison",
    "total_findings": len(audit_findings),
    "findings": audit_findings
}

assert len(audit_findings) == 29
assert len({
    item["fact_id"] for item in audit_findings
}) == 29

with open(
    "case_0014_evidence_grounded_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Case:", audit_report["case_id"])
print("Audit findings:", audit_report["total_findings"])
print("Detection method:", audit_report["detection_method"])

print("\nFIRST THREE FINDINGS:")
for item in audit_findings[:3]:
    print(
        item["fact_id"],
        "|",
        item["source_evidence_id"],
        "|",
        item["omitted_fact"]
    )

print("\nEvidence-grounded audit report saved successfully.")

In [ ]:
# Select five facts for the independent audit test
pilot_ids = ["F005", "F009", "F015", "F025", "F041"]

blind_pilot_facts = (
    fact_checklist[
        fact_checklist["fact_id"].isin(pilot_ids)
    ][["fact_id", "fact", "source_evidence"]]
    .sort_values("fact_id")
    .to_dict(orient="records")
)

# Verify all selected facts are included
assert len(blind_pilot_facts) == 5
assert {
    item["fact_id"] for item in blind_pilot_facts
} == set(pilot_ids)

print("Blind test facts:", len(blind_pilot_facts))

for item in blind_pilot_facts:
    print(item["fact_id"], "|", item["fact"])

print("\nNo reviewed classifications included.")
print("Blind evaluation batch prepared successfully.")

In [ ]:
import json

# Use the existing automated evaluator
blind_pilot_result = evaluate_with_recovery(
    blind_pilot_facts,
    simple_sentence_map
)

print("\n=== AUTOMATED AUDITOR PILOT ===")

for item in blind_pilot_result["results"]:
    print(
        item["fact_id"],
        "|",
        item["status"],
        "| Evidence:",
        item["evidence_ids"]
    )

print(
    "\nFacts evaluated:",
    len(blind_pilot_result["results"])
)

print(
    "Unresolved facts:",
    blind_pilot_result["unresolved_fact_ids"]
)

# Save predictions before comparing with reviewed labels
with open(
    "case_0014_blind_auditor_pilot_raw.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        blind_pilot_result,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nPilot predictions saved successfully.")

In [ ]:
import pandas as pd

# Load reviewed classifications for the Simple Baseline
reviewed_lookup = {
    item["fact_id"]: item["reviewed_status"]
    for item in consolidated_simple["results"]
}

pilot_rows = []

for item in blind_pilot_result["results"]:

    fact_id = item["fact_id"]
    predicted = item["status"]
    reviewed = reviewed_lookup[fact_id]

    evidence_ids = item.get("evidence_ids", [])

    pilot_rows.append({
        "fact_id": fact_id,
        "auditor_prediction": predicted,
        "reviewed_status": reviewed,
        "classification_correct": predicted == reviewed,
        "evidence_ids": ", ".join(evidence_ids)
    })

pilot_evaluation_df = pd.DataFrame(pilot_rows)

display(pilot_evaluation_df)

correct = pilot_evaluation_df[
    "classification_correct"
].sum()

total = len(pilot_evaluation_df)

print("Correct classifications:", correct)
print("Total facts:", total)
print("Classification agreement:", round(
    100 * correct / total, 2
), "%")

pilot_evaluation_df.to_csv(
    "case_0014_blind_auditor_pilot_evaluation.csv",
    index=False
)

print("\nPilot evaluation saved successfully.")

In [ ]:
# Exclude headings that contain no clinical facts
note_headings = {
    "Clinical Note:",
    "Assessment:",
    "Plan:",
    "History:",
    "Findings:",
    "Chief Complaint:"
}

filtered_simple_sentence_map = {
    sid: text
    for sid, text in simple_sentence_map.items()
    if text.strip() not in note_headings
}

print("Original segments:", len(simple_sentence_map))
print(
    "Clinical text segments:",
    len(filtered_simple_sentence_map)
)

print("\nExcluded segments:")
for sid, text in simple_sentence_map.items():
    if sid not in filtered_simple_sentence_map:
        print(sid, ":", text)

print("\nFiltered sentence map created successfully.")

In [ ]:
import json

# Repeat the same pilot with clinical text segments only
filtered_pilot_result = evaluate_with_recovery(
    blind_pilot_facts,
    filtered_simple_sentence_map
)

print("\n=== FILTERED-EVIDENCE PILOT ===")

for item in filtered_pilot_result["results"]:
    print(
        item["fact_id"],
        "|",
        item["status"],
        "| Evidence:",
        item["evidence_ids"]
    )

print(
    "\nFacts evaluated:",
    len(filtered_pilot_result["results"])
)

print(
    "Unresolved facts:",
    filtered_pilot_result["unresolved_fact_ids"]
)

# Save separately to preserve the earlier experiment
with open(
    "case_0014_filtered_auditor_pilot_raw.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        filtered_pilot_result,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nFiltered pilot predictions saved successfully.")

In [ ]:
problem_ids = {"F005", "F015", "F025"}

for item in filtered_pilot_result["results"]:

    if item["fact_id"] not in problem_ids:
        continue

    print("\n" + "=" * 60)
    print("FACT:", item["fact_id"])
    print("REFERENCE:", next(
        f["fact"]
        for f in blind_pilot_facts
        if f["fact_id"] == item["fact_id"]
    ))
    print("MODEL STATUS:", item["status"])
    print("MODEL REASON:", item.get("reason", ""))

    print("\nSELECTED EVIDENCE:")
    for sid in item["evidence_ids"]:
        print(sid, ":", filtered_simple_sentence_map[sid])

    if not item["evidence_ids"]:
        print("No evidence selected.")

print("\n" + "=" * 60)
print("HOT FLASHES APPEAR IN:")
print("S011:", filtered_simple_sentence_map["S011"])

In [ ]:
# Explicit clinical concepts for the three pilot facts
pilot_concepts = {
    "F005": ["cardiac disease", "cancer"],
    "F015": ["latex"],
    "F025": ["hot flashes"]
}

# Search the actual generated note, excluding headings
concept_check_results = {}

for fact_id, concepts in pilot_concepts.items():

    matches = {}

    for concept in concepts:
        matching_segments = [
            sid
            for sid, text in filtered_simple_sentence_map.items()
            if concept.lower() in text.lower()
        ]

        matches[concept] = matching_segments

    concept_check_results[fact_id] = matches

    print("\nFACT:", fact_id)

    for concept, segments in matches.items():
        print(
            f"Concept '{concept}':",
            segments if segments else "NOT FOUND"
        )

print("\nDeterministic concept check completed.")

In [ ]:
import pandas as pd

diagnostic_rows = []

for item in filtered_pilot_result["results"]:
    fact_id = item["fact_id"]

    if fact_id not in concept_check_results:
        continue

    concepts = concept_check_results[fact_id]

    found_concepts = [
        concept
        for concept, segments in concepts.items()
        if segments
    ]

    missing_concepts = [
        concept
        for concept, segments in concepts.items()
        if not segments
    ]

    evidence_segments = sorted({
        sid
        for segments in concepts.values()
        for sid in segments
    })

    diagnostic_rows.append({
        "fact_id": fact_id,
        "model_status": item["status"],
        "reviewed_status": reviewed_lookup[fact_id],
        "concepts_found": ", ".join(found_concepts),
        "concepts_not_found": ", ".join(missing_concepts),
        "candidate_evidence_ids": ", ".join(evidence_segments),
        "requires_review": (
            item["status"] != reviewed_lookup[fact_id]
        )
    })

hybrid_diagnostic_df = pd.DataFrame(diagnostic_rows)

display(hybrid_diagnostic_df)

hybrid_diagnostic_df.to_csv(
    "case_0014_hybrid_diagnostic_pilot.csv",
    index=False
)

print("Diagnostic facts:", len(hybrid_diagnostic_df))
print(
    "Incorrect model classifications:",
    hybrid_diagnostic_df["requires_review"].sum()
)
print("Hybrid diagnostic file saved successfully.")

In [ ]:
def retrieve_evidence_candidates(
    concepts,
    sentence_map
):
    """
    Retrieve note segments containing specified
    clinical concepts.

    This is literal concept matching, not
    semantic or negation-aware verification.
    """
    results = {}

    for concept in concepts:
        matching_ids = []

        for sid, text in sentence_map.items():
            if concept.casefold() in text.casefold():
                matching_ids.append(sid)

        results[concept] = matching_ids

    return results


# Test the reusable function
for fact_id, concepts in pilot_concepts.items():

    candidates = retrieve_evidence_candidates(
        concepts,
        filtered_simple_sentence_map
    )

    print("\nFACT:", fact_id)
    print("Candidates:", candidates)

print("\nEvidence-candidate function created successfully.")

In [ ]:
import re

def detect_simple_negation(sentence, concept):
    """
    Pilot negation detector for simple clinical statements.

    Returns:
        negated
        affirmed
        not_mentioned
        uncertain
    """
    sentence_lower = sentence.casefold()
    concept_lower = concept.casefold()

    # Match concept as a complete phrase
    pattern = (
        r"(?<!\w)"
        + re.escape(concept_lower)
        + r"(?!\w)"
    )

    match = re.search(pattern, sentence_lower)

    if match is None:
        return "not_mentioned"

    # Check a short text window preceding the concept
    preceding_text = sentence_lower[
        max(0, match.start() - 80):match.start()
    ]

    negation_patterns = [
        r"\bdenies?\b",
        r"\bno\b",
        r"\bwithout\b",
        r"\bnegative for\b"
    ]

    if any(
        re.search(pattern, preceding_text)
        for pattern in negation_patterns
    ):
        return "negated"

    return "affirmed"


# Test simple positive and negative examples
test_cases = [
    ("Patient denies latex allergy.", "latex allergy"),
    ("Patient reports hot flashes.", "hot flashes"),
    ("No cervical lymphadenopathy.", "cervical lymphadenopathy"),
    ("Patient has hypertension.", "hypertension"),
    ("Patient denies diabetes but has asthma.", "asthma")
]

for sentence, concept in test_cases:
    print(
        concept,
        "->",
        detect_simple_negation(sentence, concept)
    )

print("\nSimple negation detector created successfully.")

In [ ]:
import re

def detect_clause_negation(sentence, concept):
    """
    Experimental clause-aware negation detection.
    Returns affirmed, negated, or not_mentioned.
    """

    concept_pattern = (
        r"(?<!\w)"
        + re.escape(concept.casefold())
        + r"(?!\w)"
    )

    # Separate clauses at selected conjunctions
    clauses = re.split(
        r"\b(?:but|however|although|whereas)\b|[.;]",
        sentence.casefold()
    )

    for clause in clauses:

        if not re.search(concept_pattern, clause):
            continue

        negation_pattern = (
            r"\b(?:denies?|no|without|negative for)\b"
        )

        if re.search(negation_pattern, clause):
            return "negated"

        return "affirmed"

    return "not_mentioned"


test_cases = [
    ("Patient denies latex allergy.", "latex allergy"),
    ("Patient reports hot flashes.", "hot flashes"),
    ("No cervical lymphadenopathy.", "cervical lymphadenopathy"),
    ("Patient has hypertension.", "hypertension"),
    ("Patient denies diabetes but has asthma.", "asthma"),
    ("Patient denies diabetes but has asthma.", "diabetes")
]

for sentence, concept in test_cases:
    print(
        concept,
        "->",
        detect_clause_negation(sentence, concept)
    )

print("\nClause-aware negation detector created successfully.")

In [ ]:
# Test examples relevant to clinical documentation
negation_test_cases = [
    ("Patient denies diabetes, hypertension, and gout.", "diabetes", "negated"),
    ("Patient denies diabetes, hypertension, and gout.", "hypertension", "negated"),
    ("Patient denies diabetes, hypertension, and gout.", "gout", "negated"),
    ("No fever, cough, or chest pain.", "chest pain", "negated"),
    ("Denies latex allergy but reports Bactrim allergy.", "latex allergy", "negated"),
    ("Denies latex allergy but reports Bactrim allergy.", "Bactrim allergy", "affirmed"),
    ("Patient has asthma but denies sleep apnea.", "asthma", "affirmed"),
    ("Patient has asthma but denies sleep apnea.", "sleep apnea", "negated")
]

correct = 0

for sentence, concept, expected in negation_test_cases:
    predicted = detect_clause_negation(sentence, concept)

    is_correct = predicted == expected
    correct += int(is_correct)

    print(
        concept,
        "| Expected:", expected,
        "| Predicted:", predicted,
        "| Correct:", is_correct
    )

print("\nCorrect:", correct, "of", len(negation_test_cases))
print("Negation test completed.")

In [ ]:
# Actual source excerpts from CASE_0014
real_negation_tests = [
    (
        "She also denies cardiac disease and prior history of cancer.",
        "cardiac disease",
        "negated"
    ),
    (
        "She also denies cardiac disease and prior history of cancer.",
        "cancer",
        "negated"
    ),
    (
        "The patient denies latex allergy.",
        "latex allergy",
        "negated"
    ),
    (
        "She denies obstructive sleep apnea, kidney stones, "
        "bloody bowel movements, ulcerative colitis, "
        "Crohn's disease, dark tarry stools and melena.",
        "obstructive sleep apnea",
        "negated"
    ),
    (
        "She denies obstructive sleep apnea, kidney stones, "
        "bloody bowel movements, ulcerative colitis, "
        "Crohn's disease, dark tarry stools and melena.",
        "melena",
        "negated"
    )
]

correct = 0

for sentence, concept, expected in real_negation_tests:

    # Confirm that the excerpt occurs in the source
    source_verified = (
        normalize_clinical_text(sentence)
        in normalize_clinical_text(original_transcription)
    )

    prediction = detect_clause_negation(
        sentence, concept
    )

    passed = source_verified and prediction == expected
    correct += int(passed)

    print(
        concept,
        "| Source verified:", source_verified,
        "| Expected:", expected,
        "| Predicted:", prediction,
        "| Passed:", passed
    )

print("\nPassed:", correct, "of", len(real_negation_tests))
print("Real-transcription negation test completed.")

In [ ]:
challenging_tests = [
    ("No fever, cough present.", "fever", "negated"),
    ("No fever, cough present.", "cough", "affirmed"),
    ("No nausea, vomiting reported.", "nausea", "negated"),
    ("No nausea, vomiting reported.", "vomiting", "affirmed"),
    ("Patient does not deny chest pain.", "chest pain", "uncertain"),
    ("Cannot rule out pneumonia.", "pneumonia", "uncertain"),
    ("Family history of diabetes.", "diabetes", "affirmed"),
    ("Patient denies fever but reports chills.", "chills", "affirmed")
]

results = []

for sentence, concept, expected in challenging_tests:
    predicted = detect_clause_negation(sentence, concept)

    results.append({
        "sentence": sentence,
        "concept": concept,
        "expected": expected,
        "predicted": predicted,
        "correct": predicted == expected
    })

import pandas as pd

negation_challenge_df = pd.DataFrame(results)

display(negation_challenge_df)

print(
    "Correct:",
    negation_challenge_df["correct"].sum(),
    "of",
    len(negation_challenge_df)
)

print("Challenging negation test completed.")

In [ ]:
import re

def detect_negation_with_uncertainty(sentence, concept):
    """
    Experimental detector with uncertainty recognition.

    Returns:
        affirmed
        negated
        uncertain
        not_mentioned
    """

    text = sentence.casefold()
    target = concept.casefold()

    concept_pattern = (
        r"(?<!\w)"
        + re.escape(target)
        + r"(?!\w)"
    )

    # Split contrasting clauses
    clauses = re.split(
        r"\b(?:but|however|although|whereas)\b|[.;]",
        text
    )

    for clause in clauses:

        match = re.search(concept_pattern, clause)

        if match is None:
            continue

        preceding = clause[:match.start()]

        uncertainty_patterns = [
            r"\bdoes not deny\b",
            r"\bdo not deny\b",
            r"\bcannot rule out\b",
            r"\bcan't rule out\b",
            r"\bpossible\b",
            r"\bsuspected\b"
        ]

        if any(
            re.search(pattern, preceding)
            for pattern in uncertainty_patterns
        ):
            return "uncertain"

        negation_patterns = [
            r"\bdenies?\b",
            r"\bno\b",
            r"\bwithout\b",
            r"\bnegative for\b"
        ]

        if any(
            re.search(pattern, preceding)
            for pattern in negation_patterns
        ):
            return "negated"

        return "affirmed"

    return "not_mentioned"


# Test the two previously failed uncertainty examples
uncertainty_tests = [
    (
        "Patient does not deny chest pain.",
        "chest pain",
        "uncertain"
    ),
    (
        "Cannot rule out pneumonia.",
        "pneumonia",
        "uncertain"
    )
]

for sentence, concept, expected in uncertainty_tests:
    predicted = detect_negation_with_uncertainty(
        sentence, concept
    )

    print(
        concept,
        "| Expected:", expected,
        "| Predicted:", predicted,
        "| Correct:", predicted == expected
    )

print("\nUncertainty-aware detector created successfully.")

In [ ]:
import re

def detect_ambiguous_negation_scope(sentence, concept):
    """
    Flag potentially ambiguous negation scope.

    Returns:
        uncertain
        or the existing detector's classification
    """

    text = sentence.casefold()
    target = concept.casefold()

    # Detect a comma-separated statement beginning with "No"
    # and ending with an affirmative reporting expression.
    ambiguous_pattern = (
        r"\bno\s+[^.;]+,\s*[^.;]+\b"
        r"(?:present|reported|noted)\b"
    )

    if (
        re.search(ambiguous_pattern, text)
        and re.search(
            r"(?<!\w)" + re.escape(target) + r"(?!\w)",
            text
        )
    ):
        return "uncertain"

    return detect_negation_with_uncertainty(
        sentence, concept
    )


tests = [
    ("No fever, cough present.", "cough"),
    ("No nausea, vomiting reported.", "vomiting"),
    ("Patient denies latex allergy.", "latex allergy"),
    ("Patient reports hot flashes.", "hot flashes")
]

for sentence, concept in tests:
    print(
        concept,
        "->",
        detect_ambiguous_negation_scope(sentence, concept)
    )

print("\nAmbiguity detection test completed.")

In [ ]:
import pandas as pd

regression_tests = [
    ("Patient denies latex allergy.", "latex allergy", "negated"),
    ("Patient reports hot flashes.", "hot flashes", "affirmed"),
    ("No cervical lymphadenopathy.", "cervical lymphadenopathy", "negated"),
    ("Patient has hypertension.", "hypertension", "affirmed"),
    ("Patient denies diabetes but has asthma.", "diabetes", "negated"),
    ("Patient denies diabetes but has asthma.", "asthma", "affirmed"),
    ("No fever, cough, or chest pain.", "chest pain", "negated"),
    ("Patient does not deny chest pain.", "chest pain", "uncertain"),
    ("Cannot rule out pneumonia.", "pneumonia", "uncertain"),
    ("No fever, cough present.", "cough", "uncertain"),
    ("No nausea, vomiting reported.", "vomiting", "uncertain"),
    ("Patient denies fever but reports chills.", "chills", "affirmed"),
    ("Family history of diabetes.", "diabetes", "affirmed")
]

regression_results = []

for sentence, concept, expected in regression_tests:
    predicted = detect_ambiguous_negation_scope(
        sentence, concept
    )

    regression_results.append({
        "concept": concept,
        "expected": expected,
        "predicted": predicted,
        "correct": predicted == expected
    })

regression_df = pd.DataFrame(regression_results)

display(regression_df)

print(
    "\nPassed:",
    regression_df["correct"].sum(),
    "of",
    len(regression_df)
)

print("\nFailed examples:")
display(regression_df[~regression_df["correct"]])

print("\nNegation regression test completed.")

In [ ]:
# Clinical concepts and expected interpretations
note_tests = [
    ("diabetes mellitus", "negated"),
    ("prior DVT", "negated"),
    ("hypertension", "affirmed"),
    ("high cholesterol", "affirmed"),
    ("hot flashes", "affirmed"),
    ("snoring", "affirmed"),
    ("latex allergy", "not_mentioned"),
    ("cardiac disease", "not_mentioned")
]

results = []

for concept, expected in note_tests:

    matches = []

    for sid, sentence in filtered_simple_sentence_map.items():

        prediction = detect_ambiguous_negation_scope(
            sentence,
            concept
        )

        if prediction != "not_mentioned":
            matches.append((sid, prediction))

    if len(matches) == 0:
        final_prediction = "not_mentioned"
    elif len(set(status for _, status in matches)) == 1:
        final_prediction = matches[0][1]
    else:
        final_prediction = "uncertain"

    results.append({
        "concept": concept,
        "expected": expected,
        "predicted": final_prediction,
        "evidence_ids": ", ".join(sid for sid, _ in matches),
        "correct": final_prediction == expected
    })

import pandas as pd

actual_note_test_df = pd.DataFrame(results)

display(actual_note_test_df)

print(
    "\nPassed:",
    actual_note_test_df["correct"].sum(),
    "of",
    len(actual_note_test_df)
)

print("\nActual-note test completed.")

In [ ]:
def compare_clinical_polarity(source_status, note_status):
    """
    Compare source and generated-note polarity.

    This only evaluates polarity, not complete
    clinical fact preservation.
    """

    valid_statuses = {
        "affirmed",
        "negated",
        "uncertain",
        "not_mentioned"
    }

    if (
        source_status not in valid_statuses
        or note_status not in valid_statuses
    ):
        return "invalid_input"

    if note_status == "not_mentioned":
        return "missing"

    if (
        source_status == "uncertain"
        or note_status == "uncertain"
    ):
        return "requires_review"

    if source_status == "not_mentioned":
        return "requires_review"

    if source_status != note_status:
        return "contradicted"

    return "polarity_preserved"


polarity_tests = [
    ("negated", "negated"),
    ("negated", "affirmed"),
    ("affirmed", "negated"),
    ("affirmed", "affirmed"),
    ("negated", "not_mentioned"),
    ("affirmed", "not_mentioned"),
    ("uncertain", "affirmed")
]

for source_status, note_status in polarity_tests:
    print(
        source_status,
        "->",
        note_status,
        ":",
        compare_clinical_polarity(
            source_status,
            note_status
        )
    )

print("\nPolarity comparison function created successfully.")

In [ ]:
import pandas as pd

polarity_pilot = {
    "F005": "cardiac disease",
    "F015": "latex allergy",
    "F025": "hot flashes"
}

polarity_results = []

for fact_id, concept in polarity_pilot.items():

    reference = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id
    ].iloc[0]

    source_text = reference["source_evidence"]

    source_polarity = detect_ambiguous_negation_scope(
        source_text,
        concept
    )

    note_matches = []

    for sid, sentence in filtered_simple_sentence_map.items():

        status = detect_ambiguous_negation_scope(
            sentence,
            concept
        )

        if status != "not_mentioned":
            note_matches.append((sid, status))

    if not note_matches:
        note_polarity = "not_mentioned"
    elif len({status for _, status in note_matches}) == 1:
        note_polarity = note_matches[0][1]
    else:
        note_polarity = "uncertain"

    comparison = compare_clinical_polarity(
        source_polarity,
        note_polarity
    )

    polarity_results.append({
        "fact_id": fact_id,
        "concept": concept,
        "source_polarity": source_polarity,
        "note_polarity": note_polarity,
        "comparison": comparison,
        "note_evidence_ids": ", ".join(
            sid for sid, _ in note_matches
        )
    })

polarity_pilot_df = pd.DataFrame(polarity_results)

display(polarity_pilot_df)

polarity_pilot_df.to_csv(
    "case_0014_polarity_pilot.csv",
    index=False
)

print("Facts checked:", len(polarity_pilot_df))
print("Polarity pilot saved successfully.")

In [ ]:
# Synthetic statements for testing contradiction detection
contradiction_tests = [
    {
        "source": "Patient denies diabetes.",
        "note": "Patient has diabetes.",
        "concept": "diabetes",
        "expected": "contradicted"
    },
    {
        "source": "Patient has hypertension.",
        "note": "Patient denies hypertension.",
        "concept": "hypertension",
        "expected": "contradicted"
    },
    {
        "source": "Patient denies latex allergy.",
        "note": "Patient denies latex allergy.",
        "concept": "latex allergy",
        "expected": "polarity_preserved"
    },
    {
        "source": "Patient denies latex allergy.",
        "note": "The patient has a Bactrim allergy.",
        "concept": "latex allergy",
        "expected": "missing"
    }
]

for test in contradiction_tests:

    source_status = detect_ambiguous_negation_scope(
        test["source"],
        test["concept"]
    )

    note_status = detect_ambiguous_negation_scope(
        test["note"],
        test["concept"]
    )

    result = compare_clinical_polarity(
        source_status,
        note_status
    )

    print(
        test["concept"],
        "| Expected:", test["expected"],
        "| Predicted:", result,
        "| Correct:", result == test["expected"]
    )

print("\nContradiction detection test completed.")

In [ ]:
def align_clinical_concept(
    concept,
    source_text,
    note_sentence_map
):
    """
    Align one clinical concept between source evidence
    and generated-note segments.

    Uses literal concept matching and experimental
    rule-based negation detection.
    """

    source_status = detect_ambiguous_negation_scope(
        source_text,
        concept
    )

    note_matches = []

    for sid, sentence in note_sentence_map.items():
        status = detect_ambiguous_negation_scope(
            sentence,
            concept
        )

        if status != "not_mentioned":
            note_matches.append({
                "sentence_id": sid,
                "polarity": status,
                "text": sentence
            })

    note_polarities = {
        match["polarity"]
        for match in note_matches
    }

    if not note_polarities:
        note_status = "not_mentioned"
    elif len(note_polarities) == 1:
        note_status = next(iter(note_polarities))
    else:
        note_status = "uncertain"

    alignment_status = compare_clinical_polarity(
        source_status,
        note_status
    )

    return {
        "concept": concept,
        "source_polarity": source_status,
        "note_polarity": note_status,
        "alignment_status": alignment_status,
        "note_evidence_ids": [
            match["sentence_id"]
            for match in note_matches
        ]
    }


# Test on previously verified concepts
for concept, source_text in [
    (
        "cardiac disease",
        "She also denies cardiac disease and prior history of cancer."
    ),
    (
        "latex allergy",
        "The patient denies latex allergy."
    ),
    (
        "hot flashes",
        "Review of systems is positive for hot flashes."
    )
]:
    result = align_clinical_concept(
        concept,
        source_text,
        filtered_simple_sentence_map
    )

    print(
        result["concept"],
        "|",
        result["alignment_status"],
        "| Evidence:",
        result["note_evidence_ids"]
    )

print("\nEvidence-alignment function created successfully.")

In [ ]:
import pandas as pd

# Use previously verified reference facts
alignment_test_concepts = {
    "F005": "cardiac disease",
    "F015": "latex allergy",
    "F025": "hot flashes",
    "F034": "obstructive sleep apnea",
    "F053": "+1 edema"
}

alignment_rows = []

for fact_id, concept in alignment_test_concepts.items():

    reference = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        sentence_map = note_sentence_maps[method]

        # Exclude section headings
        clinical_segments = {
            sid: text
            for sid, text in sentence_map.items()
            if text.strip() not in note_headings
        }

        result = align_clinical_concept(
            concept,
            reference["source_evidence"],
            clinical_segments
        )

        alignment_rows.append({
            "fact_id": fact_id,
            "method": method,
            "concept": concept,
            "source_polarity": result["source_polarity"],
            "note_polarity": result["note_polarity"],
            "alignment_status": result["alignment_status"],
            "evidence_ids": ", ".join(
                result["note_evidence_ids"]
            )
        })

alignment_comparison_df = pd.DataFrame(alignment_rows)

display(alignment_comparison_df)

alignment_comparison_df.to_csv(
    "case_0014_cross_method_alignment_pilot.csv",
    index=False
)

print("Alignment comparisons:", len(alignment_comparison_df))
print("Cross-method alignment pilot saved successfully.")

In [ ]:
import pandas as pd

# Retrieve independently reviewed classifications
reviewed_lookup_by_method = {}

for method, data in [
    ("Simple Baseline", consolidated_simple),
    ("Structured Prompt", consolidated_structured)
]:
    for item in data["results"]:
        reviewed_lookup_by_method[
            (method, item["fact_id"])
        ] = item["reviewed_status"]

alignment_validation_rows = []

for _, row in alignment_comparison_df.iterrows():

    reviewed_status = reviewed_lookup_by_method[
        (row["method"], row["fact_id"])
    ]

    # Compare only concepts with clear missing/preserved polarity
    expected_alignment = (
        "missing"
        if reviewed_status == "missing"
        else "polarity_preserved"
        if reviewed_status == "present"
        else "requires_review"
    )

    alignment_validation_rows.append({
        "fact_id": row["fact_id"],
        "method": row["method"],
        "alignment_status": row["alignment_status"],
        "reviewed_status": reviewed_status,
        "expected_alignment": expected_alignment,
        "matches_review": (
            row["alignment_status"] == expected_alignment
        )
    })

alignment_validation_df = pd.DataFrame(
    alignment_validation_rows
)

display(alignment_validation_df)

print(
    "Matching comparisons:",
    alignment_validation_df["matches_review"].sum()
)
print(
    "Total comparisons:",
    len(alignment_validation_df)
)

alignment_validation_df.to_csv(
    "case_0014_alignment_validation.csv",
    index=False
)

print("\nAlignment validation saved successfully.")

In [ ]:
# Select facts from different clinical sections
candidate_ids = [
    "F001",  # Hypertension
    "F004",  # Multiple denied conditions
    "F009",  # Crestor dosage
    "F014",  # Bactrim allergy
    "F017",  # Alcohol use
    "F021",  # Family history
    "F027",  # Asthma
    "F043",  # Blood pressure
    "F064",  # BMI
    "F067"   # Preoperative weight-loss plan
]

candidate_facts = fact_checklist[
    fact_checklist["fact_id"].isin(candidate_ids)
].sort_values("fact_id")

print("Candidate facts:", len(candidate_facts))
print("=" * 65)

for _, row in candidate_facts.iterrows():
    print("\nFACT:", row["fact_id"])
    print("SECTION:", row["section"])
    print("REFERENCE:", row["fact"])
    print("SOURCE EVIDENCE:", row["source_evidence"])

print("\nCandidate inspection completed.")

In [ ]:
import pandas as pd

numeric_concepts = {
    "F043": "126/80",
    "F064": "body mass index of 41",
    "F067": "10% preoperative weight loss"
}

numeric_results = []

for fact_id, concept in numeric_concepts.items():

    source_text = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id,
        "source_evidence"
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        sentence_map = {
            sid: text
            for sid, text in note_sentence_maps[method].items()
            if text.strip() not in note_headings
        }

        result = align_clinical_concept(
            concept,
            source_text,
            sentence_map
        )

        numeric_results.append({
            "fact_id": fact_id,
            "method": method,
            "concept": concept,
            "alignment": result["alignment_status"],
            "evidence_ids": ", ".join(
                result["note_evidence_ids"]
            )
        })

numeric_alignment_df = pd.DataFrame(numeric_results)

display(numeric_alignment_df)

print("\nComparisons:", len(numeric_alignment_df))
print("Numerical alignment test completed.")

In [ ]:
import re

def normalize_clinical_terms(text):
    """
    Normalize selected equivalent clinical expressions.
    This is a small, controlled terminology prototype.
    """
    text = str(text).casefold()

    # Expand the abbreviation BMI
    text = re.sub(
        r"\bbmi\b",
        "body mass index",
        text
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test equivalent BMI expressions
test_expressions = [
    "BMI of 41",
    "Body mass index of 41",
    "Patient has a BMI of 41"
]

for expression in test_expressions:
    print(
        expression,
        "->",
        normalize_clinical_terms(expression)
    )

# Verify the actual Simple Baseline segment
normalized_note = normalize_clinical_terms(
    simple_sentence_map["S014"]
)

normalized_concept = normalize_clinical_terms(
    "body mass index of 41"
)

print("\nBMI concept found:", normalized_concept in normalized_note)
print("BMI normalization test completed.")

In [ ]:
import re

def retrieve_normalized_evidence(concept, sentence_map):
    """
    Find segments containing a clinical concept
    after terminology and whitespace normalization.

    This retrieves candidate evidence only.
    It does not verify polarity or full fact preservation.
    """

    normalized_concept = normalize_clinical_terms(concept)

    # Require phrase boundaries to avoid substring matches
    pattern = (
        r"(?<!\w)"
        + re.escape(normalized_concept)
        + r"(?!\w)"
    )

    matching_ids = []

    for sid, sentence in sentence_map.items():

        normalized_sentence = normalize_clinical_terms(
            sentence
        )

        if re.search(pattern, normalized_sentence):
            matching_ids.append(sid)

    return matching_ids


concept = "body mass index of 41"

for method in ["Simple Baseline", "Structured Prompt"]:

    matches = retrieve_normalized_evidence(
        concept,
        note_sentence_maps[method]
    )

    print("\nMETHOD:", method)
    print("Evidence IDs:", matches)

    for sid in matches:
        print(sid, ":", note_sentence_maps[method][sid])

print("\nNormalized evidence retrieval created successfully.")

In [ ]:
import os
import glob
import shutil

backup_dir = "/content/drive/MyDrive/Clinical_Auditor_Research/CASE_0014"

os.makedirs(backup_dir, exist_ok=True)

patterns = [
    "case_0014_*.json",
    "case_0014_*.csv",
    "baseline_vs_structured_1_case.csv"
]

files_to_copy = sorted(set(
    filename
    for pattern in patterns
    for filename in glob.glob(pattern)
))

copied = []

for filename in files_to_copy:
    destination = os.path.join(
        backup_dir,
        os.path.basename(filename)
    )

    shutil.copy2(filename, destination)
    copied.append(destination)

print("Files backed up:", len(copied))
print("Backup folder:", backup_dir)

important_files = [
    "case_0014_simple_all70_reviewed.json",
    "case_0014_structured_all70_reviewed.json",
    "case_0014_full70_comparison_metrics.csv",
    "case_0014_source_evidence_registry.csv"
]

print("\nImportant files verified:")

for filename in important_files:
    path = os.path.join(backup_dir, filename)
    print(filename, ":", os.path.exists(path))

print("\nBackup completed successfully.")

In [ ]:
def audit_clinical_concept(
    concept,
    source_text,
    note_sentence_map
):
    """
    Experimental clinical concept auditor.

    Combines:
    - Clinical terminology normalization
    - Evidence retrieval
    - Negation and uncertainty detection
    - Source-to-note polarity comparison

    Does not yet assess complete fact preservation.
    """

    # Remove nonclinical section headings
    clinical_segments = {
        sid: text
        for sid, text in note_sentence_map.items()
        if text.strip() not in note_headings
    }

    # Retrieve candidate evidence using normalization
    evidence_ids = retrieve_normalized_evidence(
        concept,
        clinical_segments
    )

    # Determine source polarity
    source_polarity = detect_ambiguous_negation_scope(
        source_text,
        concept
    )

    # Determine note polarity
    note_polarities = []

    for sid in evidence_ids:
        polarity = detect_ambiguous_negation_scope(
            clinical_segments[sid],
            concept
        )

        note_polarities.append(polarity)

    if not evidence_ids:
        note_polarity = "not_mentioned"
    elif len(set(note_polarities)) == 1:
        note_polarity = note_polarities[0]
    else:
        note_polarity = "uncertain"

    # Compare clinical polarity
    result = compare_clinical_polarity(
        source_polarity,
        note_polarity
    )

    return {
        "concept": concept,
        "source_polarity": source_polarity,
        "note_polarity": note_polarity,
        "alignment_status": result,
        "evidence_ids": evidence_ids
    }


print("Integrated Clinical Auditor function created.")

In [ ]:
def audit_clinical_concept(
    concept,
    source_text,
    note_sentence_map
):
    """
    Integrated clinical concept auditor with
    consistent terminology normalization.

    Experimental rule-driven prototype.
    """

    normalized_concept = normalize_clinical_terms(concept)

    # Exclude headings but preserve sentence IDs
    clinical_segments = {
        sid: text
        for sid, text in note_sentence_map.items()
        if text.strip() not in note_headings
    }

    # Normalize source evidence before polarity analysis
    normalized_source = normalize_clinical_terms(source_text)

    source_polarity = detect_ambiguous_negation_scope(
        normalized_source,
        normalized_concept
    )

    # Retrieve candidate note segments
    evidence_ids = retrieve_normalized_evidence(
        concept,
        clinical_segments
    )

    note_polarities = []

    for sid in evidence_ids:
        normalized_sentence = normalize_clinical_terms(
            clinical_segments[sid]
        )

        note_polarities.append(
            detect_ambiguous_negation_scope(
                normalized_sentence,
                normalized_concept
            )
        )

    if not evidence_ids:
        note_polarity = "not_mentioned"
    elif len(set(note_polarities)) == 1:
        note_polarity = note_polarities[0]
    else:
        note_polarity = "uncertain"

    alignment_status = compare_clinical_polarity(
        source_polarity,
        note_polarity
    )

    return {
        "concept": concept,
        "source_polarity": source_polarity,
        "note_polarity": note_polarity,
        "alignment_status": alignment_status,
        "evidence_ids": evidence_ids
    }


# Test the previously problematic BMI example
result = audit_clinical_concept(
    concept="body mass index of 41",
    source_text=(
        "The patient has a body mass index of 41."
    ),
    note_sentence_map=note_sentence_maps["Simple Baseline"]
)

print("Concept:", result["concept"])
print("Source polarity:", result["source_polarity"])
print("Note polarity:", result["note_polarity"])
print("Alignment:", result["alignment_status"])
print("Evidence IDs:", result["evidence_ids"])

print("\nIntegrated auditor normalization test completed.")

In [ ]:
import pandas as pd

test_concepts = {
    "F005": "cardiac disease",
    "F015": "latex allergy",
    "F025": "hot flashes",
    "F034": "obstructive sleep apnea",
    "F053": "+1 edema",
    "F064": "body mass index of 41"
}

test_rows = []

for fact_id, concept in test_concepts.items():

    source_text = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id,
        "source_evidence"
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        result = audit_clinical_concept(
            concept,
            source_text,
            note_sentence_maps[method]
        )

        test_rows.append({
            "fact_id": fact_id,
            "method": method,
            "source_polarity": result["source_polarity"],
            "note_polarity": result["note_polarity"],
            "alignment": result["alignment_status"],
            "evidence_ids": ", ".join(result["evidence_ids"])
        })

integrated_test_df = pd.DataFrame(test_rows)

display(integrated_test_df)

print("Comparisons completed:", len(integrated_test_df))
print(
    "Uncertain or invalid source polarity:",
    integrated_test_df["source_polarity"]
    .isin(["uncertain", "not_mentioned"])
    .sum()
)

print("\nIntegrated auditor regression test completed.")

In [ ]:
import inspect
import os

module_functions = [
    normalize_clinical_terms,
    retrieve_normalized_evidence,
    detect_negation_with_uncertainty,
    detect_ambiguous_negation_scope,
    compare_clinical_polarity,
    audit_clinical_concept
]

module_path = (
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

os.makedirs(
    os.path.dirname(module_path),
    exist_ok=True
)

module_source = (
    '"""Experimental rule-driven clinical auditor.\n'
    'Not validated for clinical use.\n'
    '"""\n\n'
    'import re\n\n'
    + "\n\n".join(
        inspect.getsource(function)
        for function in module_functions
    )
    + "\n"
)

with open(
    module_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(module_source)

print("Module saved:", module_path)
print("Functions included:", len(module_functions))
print("File size:", os.path.getsize(module_path), "bytes")

In [ ]:
import sys
import importlib

module_folder = (
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Allow Python to locate our saved module
if module_folder not in sys.path:
    sys.path.insert(0, module_folder)

# Import the module
import clinical_auditor
importlib.reload(clinical_auditor)

# Test a function imported from the saved file
result = clinical_auditor.normalize_clinical_terms(
    "Patient has a BMI of 41"
)

print("Module imported successfully.")
print("Normalization test:", result)

# Verify the integrated function is available
print(
    "Integrated auditor available:",
    callable(clinical_auditor.audit_clinical_concept)
)

In [ ]:
from pathlib import Path
import importlib

module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

source = module_path.read_text(encoding="utf-8")

heading_definition = '''
note_headings = {
    "Clinical Note:",
    "Assessment:",
    "Plan:",
    "History:",
    "Findings:",
    "Chief Complaint:"
}

'''

# Add the missing definition only once
if "note_headings = {" not in source:
    source = source.replace(
        "import re\n",
        "import re\n\n" + heading_definition,
        1
    )
    module_path.write_text(source, encoding="utf-8")
    print("Missing heading definition added.")
else:
    print("Heading definition already exists.")

# Reload the corrected module
importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Repeat the failed integration test
module_test = clinical_auditor.audit_clinical_concept(
    concept="hot flashes",
    source_text="Review of systems is positive for hot flashes.",
    note_sentence_map=note_sentence_maps["Simple Baseline"]
)

print("Alignment:", module_test["alignment_status"])
print("Evidence IDs:", module_test["evidence_ids"])

assert module_test["alignment_status"] == "polarity_preserved"
assert module_test["evidence_ids"] == ["S011"]

print("Saved module integration test passed.")

In [ ]:
import pandas as pd

# Previously examined clinical concepts
pilot_concepts = {
    "F005": "cardiac disease",
    "F015": "latex allergy",
    "F025": "hot flashes",
    "F034": "obstructive sleep apnea",
    "F053": "+1 edema",
    "F064": "body mass index of 41"
}

audit_rows = []

for fact_id, concept in pilot_concepts.items():

    source_text = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id,
        "source_evidence"
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        result = clinical_auditor.audit_clinical_concept(
            concept=concept,
            source_text=source_text,
            note_sentence_map=note_sentence_maps[method]
        )

        audit_rows.append({
            "fact_id": fact_id,
            "method": method,
            "concept": concept,
            "prediction": result["alignment_status"],
            "evidence_ids": ", ".join(result["evidence_ids"])
        })

multi_fact_audit_df = pd.DataFrame(audit_rows)

display(multi_fact_audit_df)

print("Facts audited:", len(pilot_concepts))
print("Comparisons:", len(multi_fact_audit_df))
print("Multi-fact audit completed.")

In [ ]:
print("SIMPLE BASELINE REVIEW")
print("Container type:", type(consolidated_simple).__name__)

if isinstance(consolidated_simple, list):
    print("First record:", consolidated_simple[0])
elif isinstance(consolidated_simple, dict):
    first_key = next(iter(consolidated_simple))
    print("First key:", first_key)
    print("First value:", consolidated_simple[first_key])

print("\nSTRUCTURED PROMPT REVIEW")
print("Container type:", type(consolidated_structured).__name__)

if isinstance(consolidated_structured, list):
    print("First record:", consolidated_structured[0])
elif isinstance(consolidated_structured, dict):
    first_key = next(iter(consolidated_structured))
    print("First key:", first_key)
    print("First value:", consolidated_structured[first_key])

print("\nManual review structure inspection completed.")

In [ ]:
for method, review in [
    ("Simple Baseline", consolidated_simple),
    ("Structured Prompt", consolidated_structured)
]:
    print("\nMETHOD:", method)
    print("Top-level keys:", list(review.keys()))

    for key, value in review.items():
        if isinstance(value, list):
            print(f"\nList field: {key}")
            print("Number of items:", len(value))

            if value:
                print("First item:", value[0])

        elif isinstance(value, dict):
            print(f"\nDictionary field: {key}")
            print("Keys:", list(value.keys())[:10])

print("\nReview structure inspection completed.")

In [ ]:
import pandas as pd

# Collect manual labels from both review datasets
manual_rows = []

for method, review in [
    ("Simple Baseline", consolidated_simple),
    ("Structured Prompt", consolidated_structured)
]:
    for record in review["results"]:
        manual_rows.append({
            "fact_id": record["fact_id"],
            "method": method,
            "manual_status": record["reviewed_status"]
        })

manual_df = pd.DataFrame(manual_rows)

# Convert concept-level predictions into provisional labels
label_mapping = {
    "polarity_preserved": "present",
    "missing": "missing",
    "contradicted": "contradicted",
    "requires_review": "requires_review"
}

comparison_12_df = multi_fact_audit_df.merge(
    manual_df,
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

comparison_12_df["provisional_status"] = (
    comparison_12_df["prediction"].map(label_mapping)
)

comparison_12_df["agreement"] = (
    comparison_12_df["provisional_status"]
    == comparison_12_df["manual_status"]
)

display(
    comparison_12_df[
        [
            "fact_id",
            "method",
            "prediction",
            "manual_status",
            "agreement"
        ]
    ]
)

print("\nAgreement count:",
      comparison_12_df["agreement"].sum(),
      "/", len(comparison_12_df))

print("\nDisagreements:")
display(
    comparison_12_df.loc[
        ~comparison_12_df["agreement"],
        [
            "fact_id",
            "method",
            "prediction",
            "manual_status",
            "evidence_ids"
        ]
    ]
)

In [ ]:
# Exclude facts already used in the six-concept pilot
pilot_fact_ids = set(pilot_concepts.keys())

remaining_facts_df = fact_checklist[
    ~fact_checklist["fact_id"].isin(pilot_fact_ids)
].copy()

print("Total reference facts:", len(fact_checklist))
print("Previously tested facts:", len(pilot_fact_ids))
print("Remaining facts:", len(remaining_facts_df))

print("\nFirst 12 remaining facts:")

display(
    remaining_facts_df[
        ["fact_id", "section", "fact"]
    ].head(12)
)

print("\nManual label distribution for remaining facts:")

remaining_manual_df = manual_df[
    ~manual_df["fact_id"].isin(pilot_fact_ids)
]

display(
    pd.crosstab(
        remaining_manual_df["method"],
        remaining_manual_df["manual_status"]
    )
)

print("\nExpanded evaluation set prepared.")

In [ ]:
import pandas as pd

# Exclude the original six development facts
eligible = remaining_manual_df.copy()

# One label per fact per method is already present
# Use Simple Baseline labels for stratified sampling
simple_labels = eligible[
    eligible["method"] == "Simple Baseline"
]

selected_parts = []

for status in ["present", "partial", "missing"]:
    subset = simple_labels[
        simple_labels["manual_status"] == status
    ]

    selected_parts.append(
        subset.sample(
            n=min(4, len(subset)),
            random_state=42
        )
    )

selected_labels = pd.concat(
    selected_parts,
    ignore_index=True
)

evaluation_fact_ids = selected_labels["fact_id"].tolist()

evaluation_sample_df = fact_checklist[
    fact_checklist["fact_id"].isin(evaluation_fact_ids)
].copy()

display(
    evaluation_sample_df[
        ["fact_id", "section", "fact"]
    ].sort_values("fact_id")
)

print("\nSelected facts:", len(evaluation_fact_ids))
print("Fact IDs:", sorted(evaluation_fact_ids))
print("\nSample created successfully.")

In [ ]:
evaluation_concepts = {
    "F009": "Crestor 20 mg",
    "F010": "hydrochlorothiazide 20 mg",
    "F014": "Bactrim",
    "F018": "manager",
    "F019": "married",
    "F026": "snoring",
    "F045": "pupils",
    "F049": "breath sounds",
    "F056": "extraocular motions",
    "F061": "carotid bruit",
    "F063": "56-year-old female",
    "F065": "gastric bypass surgery"
}

query_rows = []

for fact_id, concept in evaluation_concepts.items():
    record = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id
    ].iloc[0]

    query_rows.append({
        "fact_id": fact_id,
        "reference_fact": record["fact"],
        "concept_query": concept,
        "source_evidence": record["source_evidence"]
    })

evaluation_queries_df = pd.DataFrame(query_rows)

pd.set_option("display.max_colwidth", 160)

display(evaluation_queries_df)

print("Evaluation queries prepared:", len(evaluation_queries_df))
print("All selected facts covered:",
      set(evaluation_concepts) == set(evaluation_fact_ids))

In [ ]:
expanded_audit_rows = []

for fact_id, concept in evaluation_concepts.items():

    source_text = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id,
        "source_evidence"
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        result = clinical_auditor.audit_clinical_concept(
            concept=concept,
            source_text=source_text,
            note_sentence_map=note_sentence_maps[method]
        )

        expanded_audit_rows.append({
            "fact_id": fact_id,
            "method": method,
            "concept": concept,
            "source_polarity": result["source_polarity"],
            "note_polarity": result["note_polarity"],
            "prediction": result["alignment_status"],
            "evidence_ids": ", ".join(result["evidence_ids"])
        })

expanded_audit_df = pd.DataFrame(expanded_audit_rows)

display(expanded_audit_df)

print("\nFacts audited:", len(evaluation_concepts))
print("Total comparisons:", len(expanded_audit_df))

print("\nPrediction distribution:")
display(
    pd.crosstab(
        expanded_audit_df["method"],
        expanded_audit_df["prediction"]
    )
)

print("\nExpanded evaluation completed.")

In [ ]:
# Join automated predictions to existing manual reviews
expanded_comparison_df = expanded_audit_df.merge(
    manual_df,
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

# Provisional mapping for diagnostic comparison only
expanded_comparison_df["provisional_status"] = (
    expanded_comparison_df["prediction"].map({
        "polarity_preserved": "present",
        "missing": "missing",
        "contradicted": "contradicted",
        "requires_review": "requires_review"
    })
)

expanded_comparison_df["agreement"] = (
    expanded_comparison_df["provisional_status"]
    == expanded_comparison_df["manual_status"]
)

print("EXPANDED EVALUATION RESULTS")

display(
    expanded_comparison_df[
        [
            "fact_id",
            "method",
            "provisional_status",
            "manual_status",
            "agreement"
        ]
    ]
)

print(
    "\nProvisional agreement:",
    int(expanded_comparison_df["agreement"].sum()),
    "/",
    len(expanded_comparison_df)
)

print("\nDISAGREEMENTS")

display(
    expanded_comparison_df.loc[
        ~expanded_comparison_df["agreement"],
        [
            "fact_id",
            "method",
            "concept",
            "prediction",
            "manual_status",
            "evidence_ids"
        ]
    ]
)

print("\nExpanded comparison completed.")

In [ ]:
review_sources = {
    "Simple Baseline": consolidated_simple,
    "Structured Prompt": consolidated_structured
}

disagreements = expanded_comparison_df[
    ~expanded_comparison_df["agreement"]
]

for _, row in disagreements.iterrows():

    fact_id = row["fact_id"]
    method = row["method"]

    review_record = next(
        record
        for record in review_sources[method]["results"]
        if record["fact_id"] == fact_id
    )

    print("\n" + "=" * 65)
    print("FACT:", fact_id)
    print("METHOD:", method)
    print("REFERENCE:", review_record["reference_fact"])
    print("MANUAL STATUS:", review_record["reviewed_status"])
    print("AUDITOR:", row["prediction"])

    print("\nMANUALLY REVIEWED EVIDENCE:")

    evidence_ids = review_record.get(
        "reviewed_evidence_ids", []
    )

    if not evidence_ids:
        print("No evidence IDs recorded.")

    for sid in evidence_ids:
        print(
            sid,
            ":",
            note_sentence_maps[method].get(
                sid, "[Segment not found]"
            )
        )

print("\nDisagreement evidence inspection completed.")

In [ ]:
# Resume after F009 and F010
remaining_disagreements = disagreements[
    ~disagreements["fact_id"].isin(["F009", "F010"])
]

for _, row in remaining_disagreements.iterrows():

    fact_id = row["fact_id"]
    method = row["method"]

    review_record = next(
        record
        for record in review_sources[method]["results"]
        if record["fact_id"] == fact_id
    )

    # Retrieve reference fact from the consistent checklist
    reference_fact = fact_checklist.loc[
        fact_checklist["fact_id"] == fact_id,
        "fact"
    ].iloc[0]

    print("\n" + "=" * 65)
    print("FACT:", fact_id)
    print("METHOD:", method)
    print("REFERENCE:", reference_fact)
    print("MANUAL STATUS:", review_record["reviewed_status"])
    print("AUDITOR:", row["prediction"])

    print("\nMANUALLY REVIEWED EVIDENCE:")

    evidence_ids = review_record.get(
        "reviewed_evidence_ids", []
    ) or []

    if not evidence_ids:
        print("No evidence IDs recorded.")

    for sid in evidence_ids:
        print(
            sid,
            ":",
            note_sentence_maps[method].get(
                sid, "[Segment not found]"
            )
        )

print("\nRemaining disagreement inspection completed.")

In [ ]:
import re
import pandas as pd

medication_requirements = {
    "F009": {
        "medication": r"\bcrestor\b",
        "dose": r"\b20\s*mg\b",
        "route": r"\b(?:p\.?\s*o\.?|orally|oral|by mouth)\b",
        "frequency": r"\b(?:daily|once daily|every day|qd)\b"
    },
    "F010": {
        "medication": r"\bhydrochlorothiazide\b",
        "dose": r"\b20\s*mg\b",
        "route": r"\b(?:p\.?\s*o\.?|orally|oral|by mouth)\b",
        "frequency": r"\b(?:daily|once daily|every day|qd)\b"
    }
}

component_rows = []

for fact_id, requirements in medication_requirements.items():
    for method in ["Simple Baseline", "Structured Prompt"]:

        # First identify sentences mentioning the medication
        medication_pattern = requirements["medication"]

        matching_segments = {
            sid: sentence
            for sid, sentence in note_sentence_maps[method].items()
            if re.search(
                medication_pattern,
                sentence,
                flags=re.IGNORECASE
            )
        }

        # Examine only medication-related segments
        relevant_text = " ".join(matching_segments.values())

        component_results = {
            component: bool(
                re.search(
                    pattern,
                    relevant_text,
                    flags=re.IGNORECASE
                )
            )
            for component, pattern in requirements.items()
        }

        component_rows.append({
            "fact_id": fact_id,
            "method": method,
            **component_results,
            "evidence_ids": ", ".join(matching_segments)
        })

medication_component_df = pd.DataFrame(component_rows)

display(medication_component_df)

print("\nMedication component inspection completed.")

In [ ]:
def classify_component_preservation(component_results):
    """
    Classify preservation across explicitly required components.
    """

    values = list(component_results.values())

    if all(values):
        return "present"
    elif any(values):
        return "partial"
    else:
        return "missing"


component_columns = [
    "medication",
    "dose",
    "route",
    "frequency"
]

medication_component_df["predicted_status"] = (
    medication_component_df.apply(
        lambda row: classify_component_preservation({
            column: bool(row[column])
            for column in component_columns
        }),
        axis=1
    )
)

# Compare with existing manual labels
medication_comparison_df = medication_component_df.merge(
    manual_df,
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

medication_comparison_df["agreement"] = (
    medication_comparison_df["predicted_status"]
    == medication_comparison_df["manual_status"]
)

display(
    medication_comparison_df[
        [
            "fact_id",
            "method",
            "predicted_status",
            "manual_status",
            "agreement"
        ]
    ]
)

print(
    "\nAgreement:",
    int(medication_comparison_df["agreement"].sum()),
    "/",
    len(medication_comparison_df)
)

print("\nMedication classification test completed.")

In [ ]:
import re
import pandas as pd

additional_requirements = {
    "F014": {
        "bactrim": r"\bbactrim\b",
        "rash": r"\brash\b"
    },
    "F018": {
        "manager": r"\bmanager\b",
        "employer": (
            r"\bnew york\b.*\b(?:department|taxation)\b"
            r"|\bdepartment of taxation\b"
        )
    }
}

additional_rows = []

for fact_id, requirements in additional_requirements.items():

    for method in ["Simple Baseline", "Structured Prompt"]:

        segments = note_sentence_maps[method]

        component_matches = {}
        evidence_ids = set()

        for component, pattern in requirements.items():

            matching_ids = [
                sid
                for sid, sentence in segments.items()
                if re.search(pattern, sentence, re.IGNORECASE)
            ]

            component_matches[component] = bool(matching_ids)
            evidence_ids.update(matching_ids)

        predicted_status = classify_component_preservation(
            component_matches
        )

        additional_rows.append({
            "fact_id": fact_id,
            "method": method,
            **component_matches,
            "predicted_status": predicted_status,
            "evidence_ids": ", ".join(sorted(evidence_ids))
        })

additional_component_df = pd.DataFrame(additional_rows)

additional_comparison_df = additional_component_df.merge(
    manual_df,
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

additional_comparison_df["agreement"] = (
    additional_comparison_df["predicted_status"]
    == additional_comparison_df["manual_status"]
)

display(additional_comparison_df)

print(
    "\nAgreement:",
    int(additional_comparison_df["agreement"].sum()),
    "/",
    len(additional_comparison_df)
)

print("\nAdditional component inspection completed.")

In [ ]:
fact_id = "F056"

reference = fact_checklist.loc[
    fact_checklist["fact_id"] == fact_id
].iloc[0]

print("FACT:", fact_id)
print("REFERENCE:", reference["fact"])
print("SOURCE:", reference["source_evidence"])

for method, review in [
    ("Simple Baseline", consolidated_simple),
    ("Structured Prompt", consolidated_structured)
]:
    record = next(
        item for item in review["results"]
        if item["fact_id"] == fact_id
    )

    print("\nMETHOD:", method)
    print("Manual status:", record["reviewed_status"])

    evidence_ids = record.get("reviewed_evidence_ids", []) or []

    print("Reviewed evidence IDs:", evidence_ids)

    for sid in evidence_ids:
        print(sid, ":", note_sentence_maps[method][sid])

    print("\nEye-related note segments:")

    for sid, sentence in note_sentence_maps[method].items():
        if any(
            term in sentence.casefold()
            for term in ["eye", "ocular", "pupil", "vision"]
        ):
            print(sid, ":", sentence)

print("\nF056 evidence inspection completed.")

In [ ]:
import re
import pandas as pd

def classify_extraocular_preservation(sentence_map):
    """
    Experimental rule for F056 only.

    Explicit normal extraocular movements -> present
    General normal eye examination -> partial
    Otherwise -> missing
    """

    explicit_pattern = (
        r"\b(?:extraocular (?:motions|movements)|EOM)\b"
        r".{0,35}\b(?:intact|normal|full)\b"
    )

    general_pattern = (
        r"\b(?:eye|ocular) exam(?:ination)?\s+"
        r"(?:is\s+)?normal\b"
    )

    explicit_ids = []
    general_ids = []

    for sid, sentence in sentence_map.items():
        if re.search(explicit_pattern, sentence, re.IGNORECASE):
            explicit_ids.append(sid)

        if re.search(general_pattern, sentence, re.IGNORECASE):
            general_ids.append(sid)

    if explicit_ids:
        status = "present"
        evidence_ids = explicit_ids
    elif general_ids:
        status = "partial"
        evidence_ids = general_ids
    else:
        status = "missing"
        evidence_ids = []

    return status, evidence_ids


rows = []

for method in ["Simple Baseline", "Structured Prompt"]:
    status, evidence_ids = classify_extraocular_preservation(
        note_sentence_maps[method]
    )

    manual_status = manual_df.loc[
        (manual_df["fact_id"] == "F056")
        & (manual_df["method"] == method),
        "manual_status"
    ].iloc[0]

    rows.append({
        "method": method,
        "predicted_status": status,
        "manual_status": manual_status,
        "evidence_ids": ", ".join(evidence_ids),
        "agreement": status == manual_status
    })

f056_comparison_df = pd.DataFrame(rows)

display(f056_comparison_df)

print(
    "\nAgreement:",
    int(f056_comparison_df["agreement"].sum()),
    "/",
    len(f056_comparison_df)
)

print("\nF056 examination-summary test completed.")

In [ ]:
import re

def verify_fact_components(sentence_map, requirements):
    """
    Experimental component-level fact verifier.

    requirements:
        Dictionary mapping component names
        to regular-expression patterns.

    Returns:
        Component matches, evidence IDs,
        and provisional preservation status.

    Limitation:
        Does not verify relationships between
        components or their clinical context.
    """

    component_results = {}
    component_evidence = {}

    for component, pattern in requirements.items():

        matching_ids = [
            sid
            for sid, sentence in sentence_map.items()
            if re.search(pattern, sentence, re.IGNORECASE)
        ]

        component_results[component] = bool(matching_ids)
        component_evidence[component] = matching_ids

    predicted_status = classify_component_preservation(
        component_results
    )

    return {
        "predicted_status": predicted_status,
        "components": component_results,
        "component_evidence": component_evidence
    }


# Verify the reusable function with F014
test_result = verify_fact_components(
    note_sentence_maps["Structured Prompt"],
    {
        "bactrim": r"\bbactrim\b",
        "rash": r"\brash\b"
    }
)

print("Prediction:", test_result["predicted_status"])
print("Components:", test_result["components"])
print("Evidence:", test_result["component_evidence"])

assert test_result["predicted_status"] == "partial"
assert test_result["components"] == {
    "bactrim": True,
    "rash": False
}

print("\nReusable component verifier test passed.")

In [ ]:
import inspect
import shutil
import importlib
from pathlib import Path

module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

backup_path = module_path.with_suffix(".py.bak")

# Preserve the current working module
shutil.copy2(module_path, backup_path)

source = module_path.read_text(encoding="utf-8")

new_functions = [
    classify_component_preservation,
    verify_fact_components
]

# Check that we are not duplicating definitions
for function in new_functions:
    definition = f"def {function.__name__}("

    if definition not in source:
        source += "\n\n" + inspect.getsource(function)
        print("Added:", function.__name__)
    else:
        print("Already exists:", function.__name__)

# Check syntax before saving
compile(source, str(module_path), "exec")

module_path.write_text(source, encoding="utf-8")

# Reload the updated module
importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Verify the saved implementation
result = clinical_auditor.verify_fact_components(
    note_sentence_maps["Structured Prompt"],
    {
        "bactrim": r"\bbactrim\b",
        "rash": r"\brash\b"
    }
)

assert result["predicted_status"] == "partial"
assert result["components"]["bactrim"] is True
assert result["components"]["rash"] is False

print("\nModule functions available:", 8)
print("Module backup:", backup_path.name)
print("Updated module verification passed.")

In [ ]:
import pandas as pd

candidate_variables = [
    "train_df",
    "val_df",
    "validation_df",
    "test_df",
    "df",
    "df_clean",
    "clean_df",
    "comparison_df",
    "fact_checklist"
]

print("AVAILABLE DATASETS\n")

for name in candidate_variables:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {len(obj)} rows")
        print("Columns:", list(obj.columns)[:12])
        print()

    else:
        print(f"{name}: not available as a DataFrame")

print("\nDataset inspection completed.")

In [ ]:
from pathlib import Path

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive/Clinical_Auditor_Research")
]

keywords = [
    "mtsamples",
    "train",
    "validation",
    "valid",
    "test",
    "split",
    "clean"
]

found_files = {}

for folder in search_locations:
    if not folder.exists():
        continue

    for path in folder.rglob("*.csv"):
        if any(
            keyword in path.name.casefold()
            for keyword in keywords
        ):
            found_files[str(path)] = path.stat().st_size

print("Potential dataset files:", len(found_files))

for path, size in sorted(found_files.items()):
    print(f"{size:,} bytes | {path}")

print("\nDataset file search completed.")

In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
validation_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("DATASET SIZES")
print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(validation_df) + len(test_df))

print("\nDATASET COLUMNS")
print("Training:", train_df.columns.tolist())
print("Validation:", validation_df.columns.tolist())
print("Test:", test_df.columns.tolist())

# Identify any shared identifiers if available
candidate_ids = [
    "case_id",
    "group_id",
    "original_index",
    "source_index"
]

for column in candidate_ids:
    if all(
        column in dataset.columns
        for dataset in [train_df, validation_df, test_df]
    ):
        train_ids = set(train_df[column].dropna())
        validation_ids = set(validation_df[column].dropna())
        test_ids = set(test_df[column].dropna())

        print(f"\nOverlap check: {column}")
        print("Train/validation:", len(train_ids & validation_ids))
        print("Train/test:", len(train_ids & test_ids))
        print("Validation/test:", len(validation_ids & test_ids))

print("\nDataset restoration completed.")

In [ ]:
# Retrieve similarity groups from each dataset
train_groups = set(train_df["similarity_group"].dropna())
validation_groups = set(validation_df["similarity_group"].dropna())
test_groups = set(test_df["similarity_group"].dropna())

# Check for shared groups between splits
overlaps = {
    "Train / Validation": train_groups & validation_groups,
    "Train / Test": train_groups & test_groups,
    "Validation / Test": validation_groups & test_groups
}

print("SIMILARITY-GROUP LEAKAGE CHECK\n")

for split_pair, shared_groups in overlaps.items():
    print(f"{split_pair}: {len(shared_groups)} shared groups")

print("\nUnique similarity groups:")
print("Training:", len(train_groups))
print("Validation:", len(validation_groups))
print("Test:", len(test_groups))

print("\nMissing group IDs:")
for name, dataset in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(name + ":", dataset["similarity_group"].isna().sum())

total_overlap = sum(len(groups) for groups in overlaps.values())

assert total_overlap == 0, (
    "Potential leakage: similarity groups overlap across splits."
)

print("\nGroup-level split integrity verified.")

In [ ]:
# Exclude the original development case
eligible_validation_df = validation_df[
    validation_df["case_id"] != "CASE_0014"
].copy()

# Select five reproducible evaluation cases
new_cases_df = eligible_validation_df.sample(
    n=5,
    random_state=42
).copy()

# Sort for consistent presentation
new_cases_df = new_cases_df.sort_values(
    "case_id"
).reset_index(drop=True)

# Save the selected cases
selection_path = (
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "new_validation_cases.csv"
)

new_cases_df.to_csv(
    selection_path,
    index=False
)

print("NEW VALIDATION CASES\n")

display(
    new_cases_df[
        [
            "case_id",
            "medical_specialty",
            "sample_name",
            "word_count"
        ]
    ]
)

print("\nSelected cases:", len(new_cases_df))
print(
    "Development case excluded:",
    "CASE_0014" not in new_cases_df["case_id"].values
)
print("Saved to:", selection_path)

In [ ]:
import pandas as pd

case_validation_rows = []

for _, row in new_cases_df.iterrows():
    transcription = row["transcription"]

    valid = (
        isinstance(transcription, str)
        and len(transcription.strip()) > 100
    )

    actual_words = (
        len(transcription.split())
        if isinstance(transcription, str)
        else 0
    )

    case_validation_rows.append({
        "case_id": row["case_id"],
        "specialty": row["medical_specialty"],
        "recorded_words": row["word_count"],
        "actual_words": actual_words,
        "characters": len(transcription) if valid else 0,
        "valid_transcription": valid
    })

case_validation_df = pd.DataFrame(case_validation_rows)

display(case_validation_df)

print("\nValid cases:", int(case_validation_df["valid_transcription"].sum()))
print("Total selected:", len(case_validation_df))

assert case_validation_df["valid_transcription"].all(), (
    "At least one case has invalid transcription text."
)

print("\nAll five transcriptions passed validation.")

In [ ]:
# Search the current notebook variables for existing prompts.
# Do not display API keys or other credentials.

import re

prompt_variables = []

for name, value in list(globals().items()):

    if name.startswith("_"):
        continue

    if not isinstance(value, str):
        continue

    if not re.search(r"prompt|instruction|template", name, re.I):
        continue

    if re.search(r"key|secret|token|password", name, re.I):
        continue

    prompt_variables.append((name, value))

print("Potential prompt variables:", len(prompt_variables))

for name, value in sorted(prompt_variables):
    print("\n" + "=" * 60)
    print("VARIABLE:", name)
    print("LENGTH:", len(value), "characters")
    print("CONTENT:")
    print(value[:1500])

print("\nPrompt inspection completed.")

In [ ]:
import json
from pathlib import Path

# Search locally accessible notebooks for generation code
notebooks = list(Path("/content").glob("*.ipynb"))

print("Notebook files found:", len(notebooks))

search_terms = [
    "Simple Baseline",
    "Structured Prompt",
    "chat.completions.create",
    "gpt-3.5-turbo"
]

for path in notebooks:
    print("\nNOTEBOOK:", path.name)

    try:
        notebook = json.loads(
            path.read_text(encoding="utf-8")
        )

        for index, cell in enumerate(
            notebook.get("cells", [])
        ):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(
                term in source
                for term in search_terms
            ):
                print("\nCELL:", index)
                print(source[:2500])
                print("-" * 50)

    except Exception as error:
        print("Could not inspect:", type(error).__name__)

print("\nGeneration code search completed.")

In [ ]:
# Restore the original Simple Baseline prompt
baseline_prompt = """Write a clinical note from this transcription.

Transcription:
{transcription}"""

# Restore the original Structured Prompt
structured_prompt = """
You are a clinical documentation assistant.

Step 1: Extract key clinical facts from the transcription:

- Symptoms
- Negations
- Duration
- Medical history
- Medications
- Tests
- Diagnosis
- Treatment plan

Step 2: Write a structured clinical note using only these facts.

Use sections:

- Chief Complaint
- History
- Findings
- Assessment
- Plan

Do not invent information.

Transcription:
{transcription}
"""

MODEL = "gpt-3.5-turbo"
TEMPERATURE = 0
MAX_TOKENS = 1200

print("Model:", MODEL)
print("Temperature:", TEMPERATURE)
print("Maximum output tokens:", MAX_TOKENS)
print("Baseline prompt ready:", "{transcription}" in baseline_prompt)
print("Structured prompt ready:", "{transcription}" in structured_prompt)
print("API client available:", "client" in globals())
print("Validation cases ready:", len(new_cases_df))

In [ ]:
import pandas as pd
from pathlib import Path

case = new_cases_df.iloc[0]

case_id = case["case_id"]
transcription = case["transcription"]

output_path = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research/"
    f"{case_id}_generated_notes.csv"
)

# Avoid duplicate API calls if results are already saved
if output_path.exists():
    first_case_results_df = pd.read_csv(output_path)
    print("Existing results loaded. No new API calls.")

else:
    results = []

    prompts = {
        "Simple Baseline": baseline_prompt,
        "Structured Prompt": structured_prompt
    }

    for method, template in prompts.items():

        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": template.format(
                        transcription=transcription
                    )
                }
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        generated_note = response.choices[0].message.content
        finish_reason = response.choices[0].finish_reason

        results.append({
            "case_id": case_id,
            "method": method,
            "model": MODEL,
            "transcription": transcription,
            "generated_note": generated_note,
            "finish_reason": finish_reason
        })

        # Save each completed result immediately
        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        print(method, "completed.")
        print("Finish reason:", finish_reason)

    first_case_results_df = pd.DataFrame(results)

print("\nCASE:", case_id)
print("Notes saved:", len(first_case_results_df))
print("Methods:", first_case_results_df["method"].tolist())
print("File:", output_path)

display(
    first_case_results_df[
        ["case_id", "method", "model", "finish_reason"]
    ]
)

In [ ]:
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

prompts = {
    "Simple Baseline": baseline_prompt,
    "Structured Prompt": structured_prompt
}

summary_rows = []

for _, case in new_cases_df.iloc[1:].iterrows():

    case_id = case["case_id"]
    transcription = case["transcription"]
    output_path = research_dir / f"{case_id}_generated_notes.csv"

    if output_path.exists():
        results = pd.read_csv(output_path).to_dict("records")
    else:
        results = []

    for method, template in prompts.items():

        completed = any(
            record.get("method") == method
            and isinstance(record.get("generated_note"), str)
            and record["generated_note"].strip()
            for record in results
        )

        if completed:
            print(case_id, method, ": already saved")
            continue

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": template.format(
                    transcription=transcription
                )
            }],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        results = [
            record for record in results
            if record.get("method") != method
        ]

        results.append({
            "case_id": case_id,
            "method": method,
            "model": MODEL,
            "transcription": transcription,
            "generated_note": response.choices[0].message.content,
            "finish_reason": response.choices[0].finish_reason
        })

        pd.DataFrame(results).to_csv(output_path, index=False)

        print(
            case_id,
            method,
            ":",
            response.choices[0].finish_reason
        )

    for record in results:
        summary_rows.append({
            "case_id": case_id,
            "method": record["method"],
            "finish_reason": record.get("finish_reason"),
            "saved": output_path.exists()
        })

generation_summary_df = pd.DataFrame(summary_rows)

print("\nGENERATION SUMMARY")
display(generation_summary_df)

print("\nTotal remaining-case notes:", len(generation_summary_df))
print("Generation step completed.")

In [ ]:
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

all_notes = []

for case_id in new_cases_df["case_id"]:
    file_path = research_dir / f"{case_id}_generated_notes.csv"

    assert file_path.exists(), f"Missing file: {file_path}"

    case_notes = pd.read_csv(file_path)
    all_notes.append(case_notes)

evaluation_notes_df = pd.concat(
    all_notes,
    ignore_index=True
)

# Verify the experiment configuration
assert len(evaluation_notes_df) == 10
assert not evaluation_notes_df.duplicated(
    ["case_id", "method"]
).any()
assert evaluation_notes_df["generated_note"].notna().all()
assert evaluation_notes_df["model"].eq(MODEL).all()
assert evaluation_notes_df["finish_reason"].eq("stop").all()

expected_methods = {"Simple Baseline", "Structured Prompt"}

assert all(
    set(group["method"]) == expected_methods
    for _, group in evaluation_notes_df.groupby("case_id")
)

# Record generated note lengths
evaluation_notes_df["generated_words"] = (
    evaluation_notes_df["generated_note"]
    .str.split()
    .str.len()
)

output_path = research_dir / "five_case_generation_results.csv"

evaluation_notes_df.to_csv(output_path, index=False)

display(
    evaluation_notes_df[
        ["case_id", "method", "model",
         "generated_words", "finish_reason"]
    ]
)

print("\nCases:", evaluation_notes_df["case_id"].nunique())
print("Notes:", len(evaluation_notes_df))
print("All checks passed.")
print("Saved:", output_path)

In [ ]:
import json

FACT_EXTRACTION_PROMPT = """
You are extracting reference clinical facts for a
clinical documentation preservation research study.

Use ONLY the original clinical transcription.

Extract distinct, clinically meaningful facts covering:
- Symptoms and their duration
- Relevant positive and negative findings
- Medical and surgical history
- Medications, dosage, route, and frequency
- Allergies and reactions
- Examination findings
- Diagnostic tests and results
- Assessments and diagnoses
- Treatment plans and follow-up instructions

Rules:
1. Do not invent or infer clinical information.
2. Preserve numbers, units, negations, and laterality.
3. Keep related details together in a single fact.
4. Quote a short, exact source evidence span for each fact.
5. Every evidence span must appear in the transcription.
6. Do not use or refer to generated clinical notes.
7. Return a JSON object with a "facts" list.

Each fact must contain:
- section
- fact
- source_evidence

Original transcription:
{transcription}
"""

def extract_reference_facts(transcription):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": FACT_EXTRACTION_PROMPT.format(
                transcription=transcription
            )
        }],
        temperature=0,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    result = json.loads(content)

    if not isinstance(result.get("facts"), list):
        raise ValueError("Expected a JSON object containing a facts list.")

    return result["facts"]


print("Reference-fact extraction function ready.")
print("Model:", MODEL)
print("Cases awaiting extraction:", new_cases_df["case_id"].nunique())
print("No API requests made.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

case_id = "CASE_0811"

case = new_cases_df.loc[
    new_cases_df["case_id"] == case_id
].iloc[0]

transcription = case["transcription"]

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    f"{case_id}_candidate_facts.json"
)

# Avoid repeating an API request if results are saved
if output_path.exists():
    facts = json.loads(
        output_path.read_text(encoding="utf-8")
    )
    print("Loaded saved candidate facts.")

else:
    facts = extract_reference_facts(transcription)

    # Save immediately
    output_path.write_text(
        json.dumps(facts, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print("Candidate facts extracted and saved.")

# Check each evidence span against the source text
review_rows = []

for i, fact in enumerate(facts, start=1):

    evidence = str(fact.get("source_evidence", ""))

    evidence_verified = (
        bool(evidence.strip())
        and evidence.strip().casefold()
        in transcription.casefold()
    )

    review_rows.append({
        "fact_id": f"F{i:03d}",
        "section": fact.get("section", ""),
        "fact": fact.get("fact", ""),
        "source_evidence": evidence,
        "exact_source_match": evidence_verified
    })

case_0811_facts_df = pd.DataFrame(review_rows)

pd.set_option("display.max_colwidth", 120)

display(case_0811_facts_df)

print("\nCandidate facts:", len(case_0811_facts_df))
print(
    "Exact evidence matches:",
    int(case_0811_facts_df["exact_source_match"].sum()),
    "/",
    len(case_0811_facts_df)
)

print("Saved:", output_path)

In [ ]:
import re

# Use the original CASE_0811 transcription
source_text = new_cases_df.loc[
    new_cases_df["case_id"] == "CASE_0811",
    "transcription"
].iloc[0]

search_terms = [
    "gingivitis",
    "impacted wisdom teeth",
    "calculus",
    "dental rehabilitation",
    "wisdom teeth were removed"
]

for term in search_terms:

    print("\n" + "=" * 65)
    print("SEARCH TERM:", term)

    matches = list(
        re.finditer(
            re.escape(term),
            source_text,
            flags=re.IGNORECASE
        )
    )

    print("Matches:", len(matches))

    for match in matches[:4]:
        start = max(0, match.start() - 160)
        end = min(len(source_text), match.end() + 220)

        passage = source_text[start:end]

        print("\nSOURCE PASSAGE:")
        print(repr(passage))

print("\nOriginal evidence inspection completed.")

In [ ]:
import re
import pandas as pd

# Each expression identifies a passage in the source transcription.
evidence_patterns = {
    "F001": r"2\. Moderate gingivitis\.",
    "F002": r"1\. Impacted wisdom teeth\.",
    "F003": (
        r"She presented with moderate calculus "
        r"on the buccal surfaces of her maxillary, "
        r"first molars and lower molars\."
    ),
    "F004": (
        r"POSTOPERATIVE DIAGNOSES:,"
        r"1\. Impacted wisdom teeth\.,"
        r"2\. Moderate gingivitis\."
    ),
    "F005": (
        r"Her teeth #1, 16, 17, and 32 were impacted "
        r"and are going to be removed following "
        r"my full mouth dental rehabilitation "
        r"by Dr\. Alexander\."
    )
}

repaired_rows = []

for _, row in case_0811_facts_df.iterrows():

    fact_id = row["fact_id"]

    match = re.search(
        evidence_patterns[fact_id],
        source_text,
        flags=re.IGNORECASE
    )

    exact_evidence = match.group(0) if match else None

    repaired_rows.append({
        "fact_id": fact_id,
        "section": row["section"],
        "fact": row["fact"],
        "source_evidence": exact_evidence,
        "exact_source_match": (
            exact_evidence is not None
            and exact_evidence in source_text
        )
    })

repaired_facts_df = pd.DataFrame(repaired_rows)

display(repaired_facts_df)

print(
    "\nVerified spans:",
    int(repaired_facts_df["exact_source_match"].sum()),
    "/",
    len(repaired_facts_df)
)

print("Original candidate file remains unchanged.")

In [ ]:
# Display the original transcription in manageable passages
# without changing its text or punctuation.

case_id = "CASE_0811"

source_text = new_cases_df.loc[
    new_cases_df["case_id"] == case_id,
    "transcription"
].iloc[0]

passage_size = 850

passages = [
    source_text[i:i + passage_size]
    for i in range(0, len(source_text), passage_size)
]

print("CASE:", case_id)
print("Total characters:", len(source_text))
print("Passages:", len(passages))

for i, passage in enumerate(passages, start=1):
    print("\n" + "=" * 65)
    print(f"PASSAGE {i}")
    print("=" * 65)
    print(passage)

print("\nSource transcription inspection completed.")

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Additional candidate facts.
# Evidence quotations are taken directly from the source.
additional_facts = [
    ("Procedure", "Dental prophylaxis under general anesthesia",
     "Dental prophylaxis under general anesthesia."),
    ("Operative outcome", "No complications documented",
     "COMPLICATIONS: ,None."),
    ("Operative outcome", "Estimated blood loss was minimal",
     "ESTIMATED BLOOD LOSS: ,Minimal."),
    ("Operative outcome", "Surgery lasted one hour and 17 minutes",
     "DURATION OF SURGERY: ,One hour 17 minutes."),
    ("Preoperative evaluation", "Clinical and radiographic examination showed no dental caries",
     "This clinical and radiographic examination revealed no dental caries"),
    ("Preoperative evaluation", "Informed consent was obtained before treatment",
     "they gave their informed consent to proceed with the treatment."),
    ("Anesthesia", "Anesthesia was induced by mask",
     "Anesthesia was induced by mask."),
    ("Anesthesia", "A nasal endotracheal tube was placed",
     "The patient was then intubated with a nasal endotracheal tube"),
    ("Treatment", "Dental prophylaxis followed scaling and root planing",
     "A prophylaxis was then performed using a Prophy cup and fluoridated Prophy paste after scaling and replaning was done."),
    ("Dental findings", "No restorative dentistry was required",
     "She did not require any restorative dentistry."),
    ("Treatment", "Topical fluoride was applied",
     "Topical fluoride was then placed on the teeth."),
    ("Dental findings", "The patient had impacted teeth 1, 16, 17, and 32",
     "Her teeth #1, 16, 17, and 32 were impacted"),
    ("Dental findings", "Oral hygiene was fair",
     "Oral hygiene was fair."),
    ("Dental findings", "Generalized plaque and calculus were present",
     "There was generalized plaque and calculus throughout."),
    ("Recovery", "Patient was documented as breathing spontaneously with stable vital signs",
     "breathing spontaneously with stable vital signs."),
    ("Follow-up", "Postoperative instructions and pain management were delegated to Dr. X",
     "pain control will be managed by them.")
]

expanded_rows = repaired_facts_df[
    ["section", "fact", "source_evidence"]
].to_dict("records")

expanded_rows.extend(
    {
        "section": section,
        "fact": fact,
        "source_evidence": evidence
    }
    for section, fact, evidence in additional_facts
)

expanded_case_0811_df = pd.DataFrame(expanded_rows)

expanded_case_0811_df.insert(
    0,
    "fact_id",
    [f"F{i:03d}" for i in range(1, len(expanded_case_0811_df) + 1)]
)

expanded_case_0811_df["exact_source_match"] = (
    expanded_case_0811_df["source_evidence"].apply(
        lambda evidence: (
            isinstance(evidence, str)
            and bool(evidence.strip())
            and evidence in source_text
        )
    )
)

display(expanded_case_0811_df)

print("\nTotal candidate facts:", len(expanded_case_0811_df))
print(
    "Exact evidence matches:",
    int(expanded_case_0811_df["exact_source_match"].sum()),
    "/",
    len(expanded_case_0811_df)
)

print(
    "\nUnverified evidence:",
    (~expanded_case_0811_df["exact_source_match"]).sum()
)

In [ ]:
import pandas as pd

# Preserve the original 21 candidate records
case_0811_qc_df = expanded_case_0811_df.copy()

case_0811_qc_df["qc_flag"] = ""
case_0811_qc_df["qc_note"] = ""

quality_flags = {
    "F001": (
        "overlapping_fact",
        "Moderate gingivitis is also included in F004."
    ),
    "F002": (
        "overlapping_fact",
        "Impacted wisdom teeth are also included in F004 and F017."
    ),
    "F004": (
        "compound_overlapping_fact",
        "Combines two diagnoses already represented individually."
    ),
    "F021": (
        "incomplete_evidence",
        "Evidence supports pain-control delegation, but the "
        "postoperative-instruction component needs an additional quotation."
    )
}

for fact_id, (flag, note) in quality_flags.items():
    mask = case_0811_qc_df["fact_id"] == fact_id
    case_0811_qc_df.loc[mask, "qc_flag"] = flag
    case_0811_qc_df.loc[mask, "qc_note"] = note

# Record the source-level airway discrepancy separately
source_discrepancies_df = pd.DataFrame([{
    "case_id": "CASE_0811",
    "issue": "Conflicting airway documentation",
    "statement_1": (
        "At the end of the procedure, the child was "
        "undraped, extubated, and awakened in the operating room"
    ),
    "statement_2": (
        "Following my dental surgery, the patient "
        "continued to intubated and was prepped for oral surgery procedures"
    ),
    "resolution": "Unresolved: requires clinical review"
}])

print("REFERENCE FACT QUALITY FLAGS")
display(
    case_0811_qc_df.loc[
        case_0811_qc_df["qc_flag"] != "",
        ["fact_id", "fact", "qc_flag", "qc_note"]
    ]
)

print("\nSOURCE DOCUMENTATION DISCREPANCIES")
display(source_discrepancies_df)

print("\nFacts flagged:",
      (case_0811_qc_df["qc_flag"] != "").sum())

print("Original 21 candidate facts remain unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path

# Work on a copy. Preserve the original 21 facts.
revised_facts_df = case_0811_qc_df.copy()

# Exclude duplicate compound diagnosis F004
revised_facts_df = revised_facts_df[
    revised_facts_df["fact_id"] != "F004"
].copy()

# Narrow F021 to the information supported by its quotation
revised_facts_df.loc[
    revised_facts_df["fact_id"] == "F021",
    "fact"
] = "Pain control was delegated to the subsequent surgical team"

# Record review decisions
revised_facts_df["review_decision"] = "retain_provisional"

revised_facts_df.loc[
    revised_facts_df["fact_id"] == "F021",
    "review_decision"
] = "reworded_to_match_evidence"

# Verify that every quotation remains in the transcription
revised_facts_df["exact_source_match"] = (
    revised_facts_df["source_evidence"].apply(
        lambda evidence: (
            isinstance(evidence, str)
            and bool(evidence.strip())
            and evidence in source_text
        )
    )
)

assert revised_facts_df["exact_source_match"].all()

# Save the provisional checklist and discrepancy register
research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

facts_path = research_dir / "CASE_0811_reference_facts_v1.csv"
discrepancy_path = research_dir / "CASE_0811_source_discrepancies.csv"

revised_facts_df.to_csv(facts_path, index=False)
source_discrepancies_df.to_csv(discrepancy_path, index=False)

print("Provisional facts:", len(revised_facts_df))
print(
    "Verified evidence spans:",
    int(revised_facts_df["exact_source_match"].sum())
)
print("Excluded duplicate fact: F004")
print("Reworded fact: F021")
print("Unresolved source discrepancies:",
      len(source_discrepancies_df))
print("\nSaved:", facts_path)
print("Saved:", discrepancy_path)

In [ ]:
import re
import json
from pathlib import Path

case_id = "CASE_0811"

case_notes = evaluation_notes_df[
    evaluation_notes_df["case_id"] == case_id
]

def segment_clinical_note(note):
    """
    Split a clinical note into numbered,
    nonempty lines or sentences.

    Experimental segmentation; headings may
    remain attached to adjacent clinical text.
    """
    segments = re.split(
        r"(?<=[.!?])\s+|\n+",
        str(note)
    )

    segments = [
        segment.strip()
        for segment in segments
        if segment.strip()
    ]

    return {
        f"S{i:03d}": segment
        for i, segment in enumerate(segments, start=1)
    }


case_0811_sentence_maps = {}

for _, row in case_notes.iterrows():
    method = row["method"]

    sentence_map = segment_clinical_note(
        row["generated_note"]
    )

    case_0811_sentence_maps[method] = sentence_map

    print("\nMETHOD:", method)
    print("Segments:", len(sentence_map))

    for sid, sentence in sentence_map.items():
        print(sid, ":", sentence)

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_sentence_maps.json"
)

output_path.write_text(
    json.dumps(
        case_0811_sentence_maps,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nSentence maps saved:", output_path)

In [ ]:
import re
import json
from pathlib import Path

def segment_clinical_note_v2(note):
    """
    Segment clinical notes without splitting common
    title abbreviations such as Dr. or Mr.

    Preserve the original clinical wording.
    """

    text = str(note)

    # Temporarily protect selected abbreviations
    protected = {
        "Dr.": "Dr<DOT>",
        "Mr.": "Mr<DOT>",
        "Mrs.": "Mrs<DOT>",
        "Ms.": "Ms<DOT>"
    }

    for abbreviation, replacement in protected.items():
        text = text.replace(abbreviation, replacement)

    segments = re.split(
        r"(?<=[.!?])\s+|\n+",
        text
    )

    restored_segments = []

    for segment in segments:
        segment = segment.strip()

        for abbreviation, replacement in protected.items():
            segment = segment.replace(
                replacement,
                abbreviation
            )

        if segment:
            restored_segments.append(segment)

    return {
        f"S{i:03d}": segment
        for i, segment in enumerate(
            restored_segments,
            start=1
        )
    }


case_0811_sentence_maps_v2 = {}

for _, row in case_notes.iterrows():

    method = row["method"]

    sentence_map = segment_clinical_note_v2(
        row["generated_note"]
    )

    case_0811_sentence_maps_v2[method] = sentence_map

    print("\nMETHOD:", method)
    print("Segments:", len(sentence_map))

    for sid, sentence in sentence_map.items():
        print(sid, ":", sentence)

output_path_v2 = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_sentence_maps_v2.json"
)

output_path_v2.write_text(
    json.dumps(
        case_0811_sentence_maps_v2,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nImproved sentence maps saved:", output_path_v2)

In [ ]:
import pandas as pd
from pathlib import Path

case_id = "CASE_0811"

# Use the revised source-grounded checklist
reference_facts = revised_facts_df[
    ["fact_id", "section", "fact", "source_evidence"]
].copy()

audit_rows = []

for _, fact_row in reference_facts.iterrows():

    for method in ["Simple Baseline", "Structured Prompt"]:

        audit_rows.append({
            "case_id": case_id,
            "fact_id": fact_row["fact_id"],
            "section": fact_row["section"],
            "reference_fact": fact_row["fact"],
            "source_evidence": fact_row["source_evidence"],
            "method": method,
            "generated_note": case_notes.loc[
                case_notes["method"] == method,
                "generated_note"
            ].iloc[0]
        })

case_0811_audit_manifest_df = pd.DataFrame(audit_rows)

assert len(case_0811_audit_manifest_df) == 40
assert not case_0811_audit_manifest_df.duplicated(
    ["case_id", "fact_id", "method"]
).any()

manifest_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_audit_manifest.csv"
)

case_0811_audit_manifest_df.to_csv(
    manifest_path,
    index=False
)

print("Case:", case_id)
print("Reference facts:", reference_facts["fact_id"].nunique())
print("Generation methods:", case_0811_audit_manifest_df["method"].nunique())
print("Total fact-note comparisons:", len(case_0811_audit_manifest_df))
print("Duplicate comparisons:", case_0811_audit_manifest_df.duplicated(
    ["case_id", "fact_id", "method"]
).sum())
print("Saved:", manifest_path)

display(
    case_0811_audit_manifest_df[
        ["fact_id", "section", "reference_fact", "method"]
    ].head(8)
)

In [ ]:
import pandas as pd

# Explicit concept queries for six source facts
case_0811_concepts = {
    "F001": "moderate gingivitis",
    "F002": "impacted wisdom teeth",
    "F006": "dental prophylaxis",
    "F008": "minimal",
    "F009": "one hour 17 minutes",
    "F018": "oral hygiene was fair"
}

audit_results = []

for fact_id, concept in case_0811_concepts.items():

    source_evidence = revised_facts_df.loc[
        revised_facts_df["fact_id"] == fact_id,
        "source_evidence"
    ].iloc[0]

    for method in ["Simple Baseline", "Structured Prompt"]:

        result = clinical_auditor.audit_clinical_concept(
            concept=concept,
            source_text=source_evidence,
            note_sentence_map=case_0811_sentence_maps_v2[method]
        )

        audit_results.append({
            "case_id": "CASE_0811",
            "fact_id": fact_id,
            "method": method,
            "concept": concept,
            "source_polarity": result["source_polarity"],
            "note_polarity": result["note_polarity"],
            "alignment": result["alignment_status"],
            "evidence_ids": ", ".join(result["evidence_ids"])
        })

case_0811_pilot_audit_df = pd.DataFrame(audit_results)

display(case_0811_pilot_audit_df)

print("\nFacts audited:", len(case_0811_concepts))
print("Comparisons:", len(case_0811_pilot_audit_df))
print("\nPrediction distribution:")
display(
    pd.crosstab(
        case_0811_pilot_audit_df["method"],
        case_0811_pilot_audit_df["alignment"]
    )
)

print("\nCASE_0811 pilot audit completed.")

In [ ]:
sentence = case_0811_sentence_maps_v2[
    "Simple Baseline"
]["S003"]

print("SENTENCE:")
print(sentence)

test_concepts = [
    "complications",
    "minimal",
    "blood loss",
    "minimal estimated blood loss"
]

print("\nPOLARITY DETECTION RESULTS")

for concept in test_concepts:
    result = clinical_auditor.detect_ambiguous_negation_scope(
        sentence,
        concept
    )

    print(f"{concept}: {result}")

print("\nNegation-scope diagnosis completed.")

In [ ]:
import re

def detect_negation_scope_v2(sentence, concept):
    """
    Experimental clause-aware polarity detector.

    Separates independent findings connected by:
    ', with' and ', without'.

    Preserves the existing detector for other cases.
    """

    normalized_sentence = normalize_clinical_terms(sentence)
    normalized_concept = normalize_clinical_terms(concept)

    # Split only at explicitly marked clause boundaries
    clauses = re.split(
        r",\s+(?=with\b|without\b)",
        normalized_sentence,
        flags=re.IGNORECASE
    )

    # Find the clause containing the clinical concept
    matching_clauses = [
        clause
        for clause in clauses
        if re.search(
            r"(?<!\w)" + re.escape(normalized_concept) + r"(?!\w)",
            clause
        )
    ]

    if not matching_clauses:
        return "not_mentioned"

    # Multiple matches require review instead of guessing
    if len(matching_clauses) > 1:
        return "uncertain"

    return clinical_auditor.detect_ambiguous_negation_scope(
        matching_clauses[0],
        normalized_concept
    )


sentence = case_0811_sentence_maps_v2[
    "Simple Baseline"
]["S003"]

for concept in [
    "complications",
    "minimal",
    "blood loss",
    "minimal estimated blood loss"
]:
    print(
        concept,
        "->",
        detect_negation_scope_v2(sentence, concept)
    )

print("\nClause-aware negation test completed.")

In [ ]:
import pandas as pd

regression_cases = [
    (
        "No complications were noted, with minimal blood loss.",
        "complications",
        "negated"
    ),
    (
        "No complications were noted, with minimal blood loss.",
        "blood loss",
        "affirmed"
    ),
    (
        "Patient denies fever but reports cough.",
        "fever",
        "negated"
    ),
    (
        "Patient denies fever but reports cough.",
        "cough",
        "affirmed"
    ),
    (
        "No carotid bruit was detected.",
        "carotid bruit",
        "negated"
    ),
    (
        "Patient has hypertension.",
        "hypertension",
        "affirmed"
    ),
    (
        "Cannot rule out pneumonia.",
        "pneumonia",
        "uncertain"
    ),
    (
        "No fever, cough present.",
        "cough",
        "uncertain"
    )
]

regression_rows = []

for sentence, concept, expected in regression_cases:

    predicted = detect_negation_scope_v2(
        sentence,
        concept
    )

    regression_rows.append({
        "concept": concept,
        "expected": expected,
        "predicted": predicted,
        "passed": predicted == expected
    })

negation_regression_df = pd.DataFrame(regression_rows)

display(negation_regression_df)

print(
    "\nTests passed:",
    int(negation_regression_df["passed"].sum()),
    "/",
    len(negation_regression_df)
)

print("Negation regression testing completed.")

In [ ]:
import inspect
import shutil
import importlib
from pathlib import Path

module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

# Create a new backup before changing the module
backup_path = module_path.with_suffix(".pre_negation_v2.bak")
shutil.copy2(module_path, backup_path)

source = module_path.read_text(encoding="utf-8")

# Add the tested improved detector
definition = "def detect_negation_scope_v2("

if definition not in source:
    source += "\n\n" + inspect.getsource(
        detect_negation_scope_v2
    )
    print("Added detect_negation_scope_v2.")
else:
    print("Detector already exists.")

# Update calls inside the integrated auditor only
start = source.index("def audit_clinical_concept(")
end = source.find("\ndef ", start + 1)

if end == -1:
    end = len(source)

auditor_source = source[start:end]

old_call = "detect_ambiguous_negation_scope("
new_call = "detect_negation_scope_v2("

assert old_call in auditor_source, (
    "Expected original polarity detector not found."
)

updated_auditor = auditor_source.replace(
    old_call,
    new_call
)

source = (
    source[:start]
    + updated_auditor
    + source[end:]
)

# Validate Python syntax
compile(source, str(module_path), "exec")

module_path.write_text(source, encoding="utf-8")

# Reload the improved module
importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Verify the original false contradiction is corrected
result = clinical_auditor.audit_clinical_concept(
    concept="minimal",
    source_text="ESTIMATED BLOOD LOSS: ,Minimal.",
    note_sentence_map=case_0811_sentence_maps_v2["Simple Baseline"]
)

print("\nF008 alignment:", result["alignment_status"])
print("Evidence IDs:", result["evidence_ids"])

assert result["alignment_status"] == "polarity_preserved"
assert result["evidence_ids"] == ["S003"]

print("\nUpdated Clinical Auditor passed verification.")
print("Backup:", backup_path.name)

In [ ]:
from pathlib import Path
import importlib

module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

source = module_path.read_text(encoding="utf-8")

# Fix the self-reference inside the saved module
old_call = "clinical_auditor.detect_ambiguous_negation_scope("
new_call = "detect_ambiguous_negation_scope("

assert old_call in source, "Expected reference not found."

source = source.replace(old_call, new_call)

# Verify syntax before saving
compile(source, str(module_path), "exec")

module_path.write_text(source, encoding="utf-8")

# Reload the corrected module
importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Repeat the F008 test
result = clinical_auditor.audit_clinical_concept(
    concept="minimal",
    source_text="ESTIMATED BLOOD LOSS: ,Minimal.",
    note_sentence_map=case_0811_sentence_maps_v2["Simple Baseline"]
)

print("Source polarity:", result["source_polarity"])
print("Note polarity:", result["note_polarity"])
print("Alignment:", result["alignment_status"])
print("Evidence IDs:", result["evidence_ids"])

assert result["source_polarity"] == "affirmed"
assert result["note_polarity"] == "affirmed"
assert result["alignment_status"] == "polarity_preserved"
assert result["evidence_ids"] == ["S003"]

print("\nModule correction successful.")

In [ ]:
import re
import pandas as pd

# Accept only explicitly defined equivalent expressions.
equivalent_expressions = {
    "F009": [
        r"\bone hour(?: and)? 17 minutes\b"
    ],
    "F018": [
        r"\boral hygiene was fair\b",
        r"\bfair oral hygiene\b"
    ]
}

rows = []

for fact_id, patterns in equivalent_expressions.items():
    for method in ["Simple Baseline", "Structured Prompt"]:

        sentence_map = case_0811_sentence_maps_v2[method]

        matching_ids = [
            sid
            for sid, sentence in sentence_map.items()
            if any(
                re.search(pattern, sentence, re.IGNORECASE)
                for pattern in patterns
            )
        ]

        rows.append({
            "fact_id": fact_id,
            "method": method,
            "expression_found": bool(matching_ids),
            "evidence_ids": ", ".join(matching_ids)
        })

equivalence_test_df = pd.DataFrame(rows)

display(equivalence_test_df)

print("\nComparisons:", len(equivalence_test_df))
print("Controlled expression matching completed.")

In [ ]:
import re
import inspect
import importlib
import shutil
from pathlib import Path

def retrieve_equivalent_expressions(sentence_map, expressions):
    """
    Retrieve evidence matching any explicitly supplied
    equivalent clinical expression.

    Expressions are regular-expression patterns.

    This function performs retrieval only.
    It does not establish complete fact preservation.
    """
    evidence_ids = []

    for sid, sentence in sentence_map.items():
        normalized_sentence = normalize_clinical_terms(sentence)

        if any(
            re.search(
                pattern,
                normalized_sentence,
                flags=re.IGNORECASE
            )
            for pattern in expressions
        ):
            evidence_ids.append(sid)

    return evidence_ids


module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

# Preserve the previous working version
shutil.copy2(
    module_path,
    module_path.with_suffix(".pre_equivalence.bak")
)

source = module_path.read_text(encoding="utf-8")

if "def retrieve_equivalent_expressions(" not in source:
    source += "\n\n" + inspect.getsource(
        retrieve_equivalent_expressions
    )

compile(source, str(module_path), "exec")
module_path.write_text(source, encoding="utf-8")

importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Regression checks using the saved module
tests = [
    ("F009", "Simple Baseline", ["S004"]),
    ("F009", "Structured Prompt", ["S015"]),
    ("F018", "Simple Baseline", ["S010"]),
    ("F018", "Structured Prompt", ["S010"])
]

for fact_id, method, expected in tests:
    actual = clinical_auditor.retrieve_equivalent_expressions(
        case_0811_sentence_maps_v2[method],
        equivalent_expressions[fact_id]
    )

    assert actual == expected, (
        f"{fact_id} / {method}: {actual} != {expected}"
    )

    print(f"{fact_id} | {method}: {actual}")

print("\nSaved module updated.")
print("Regression tests passed: 4/4")

In [ ]:
import re
import pandas as pd

case_0811_patterns = {
    "F001": [r"\bmoderate gingivitis\b"],
    "F002": [r"\bimpacted wisdom teeth\b"],
    "F003": [r"\bcalculus\b"],
    "F005": [r"\bwisdom teeth\b", r"\bdental rehabilitation\b"],
    "F006": [r"\bdental prophylaxis\b"],
    "F007": [r"\bno complications\b", r"\bwithout complications\b"],
    "F008": [r"\bminimal\b.*\bblood loss\b",
             r"\bminimal estimated blood loss\b"],
    "F009": [r"\bone hour(?: and)? 17 minutes\b"],
    "F010": [r"\bno dental caries\b", r"\bno caries\b"],
    "F011": [r"\binformed consent\b", r"\bconsent\b"],
    "F012": [r"\banesthesia was induced by mask\b",
             r"\bmask induction\b"],
    "F013": [r"\bnasal endotracheal\b", r"\bnasal intubation\b"],
    "F014": [r"\bprophylaxis\b", r"\bscaling\b"],
    "F015": [r"\bno restorative dentistry\b",
             r"\bno need for restorative dentistry\b"],
    "F016": [r"\btopical fluoride\b", r"\bfluoride\b"],
    "F017": [r"\b#?1,\s*16,\s*17,\s*(?:and\s*)?32\b"],
    "F018": [r"\boral hygiene was fair\b",
             r"\bfair oral hygiene\b"],
    "F019": [r"\bplaque\b", r"\bcalculus\b"],
    "F020": [r"\bstable vital signs\b",
             r"\bbreathing spontaneously\b"],
    "F021": [r"\bpain management\b", r"\bpain control\b"]
}

expected_ids = set(revised_facts_df["fact_id"])

assert set(case_0811_patterns) == expected_ids, (
    "Retrieval configuration does not cover all reference facts."
)

# Check regular-expression syntax
for fact_id, patterns in case_0811_patterns.items():
    for pattern in patterns:
        re.compile(pattern)

print("Reference facts:", len(expected_ids))
print("Configured facts:", len(case_0811_patterns))
print("All expressions valid: True")

display(
    pd.DataFrame([
        {
            "fact_id": fact_id,
            "reference_fact": revised_facts_df.loc[
                revised_facts_df["fact_id"] == fact_id,
                "fact"
            ].iloc[0],
            "retrieval_patterns": " | ".join(patterns)
        }
        for fact_id, patterns in case_0811_patterns.items()
    ])
)

print("\nRetrieval configuration ready.")

In [ ]:
import pandas as pd
from pathlib import Path

retrieval_rows = []

for _, fact_row in revised_facts_df.iterrows():

    fact_id = fact_row["fact_id"]
    patterns = case_0811_patterns[fact_id]

    for method in ["Simple Baseline", "Structured Prompt"]:

        evidence_ids = (
            clinical_auditor.retrieve_equivalent_expressions(
                case_0811_sentence_maps_v2[method],
                patterns
            )
        )

        retrieval_rows.append({
            "case_id": "CASE_0811",
            "fact_id": fact_id,
            "method": method,
            "reference_fact": fact_row["fact"],
            "candidate_found": bool(evidence_ids),
            "evidence_ids": ", ".join(evidence_ids),
            "requires_fact_verification": True
        })

case_0811_retrieval_df = pd.DataFrame(retrieval_rows)

assert len(case_0811_retrieval_df) == 40
assert not case_0811_retrieval_df.duplicated(
    ["case_id", "fact_id", "method"]
).any()

display(
    case_0811_retrieval_df[
        ["fact_id", "method", "candidate_found", "evidence_ids"]
    ]
)

print("\nCANDIDATE RETRIEVAL SUMMARY")

display(
    pd.crosstab(
        case_0811_retrieval_df["method"],
        case_0811_retrieval_df["candidate_found"]
    )
)

print(
    "\nComparisons requiring full fact verification:",
    int(case_0811_retrieval_df["requires_fact_verification"].sum())
)

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_evidence_retrieval_v1.csv"
)

case_0811_retrieval_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd
from pathlib import Path

review_rows = []

for _, row in case_0811_retrieval_df.iterrows():

    method = row["method"]
    sentence_map = case_0811_sentence_maps_v2[method]

    ids = [
        sid.strip()
        for sid in str(row["evidence_ids"]).split(",")
        if sid.strip()
    ]

    candidate_evidence = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in ids
    )

    full_numbered_note = "\n".join(
        f"{sid}: {sentence}"
        for sid, sentence in sentence_map.items()
    )

    review_rows.append({
        "case_id": row["case_id"],
        "fact_id": row["fact_id"],
        "method": method,
        "reference_fact": row["reference_fact"],
        "source_evidence": revised_facts_df.loc[
            revised_facts_df["fact_id"] == row["fact_id"],
            "source_evidence"
        ].iloc[0],
        "candidate_evidence": candidate_evidence,
        "full_numbered_note": full_numbered_note,
        "reviewed_status": "",
        "reviewed_evidence_ids": "",
        "review_notes": ""
    })

case_0811_review_df = pd.DataFrame(review_rows)

review_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_preservation_review_v1.csv"
)

case_0811_review_df.to_csv(review_path, index=False)

print("Review records:", len(case_0811_review_df))
print("Methods:", case_0811_review_df["method"].nunique())
print("Facts:", case_0811_review_df["fact_id"].nunique())

print("\nFirst six review records:")
display(
    case_0811_review_df[
        [
            "fact_id",
            "method",
            "reference_fact",
            "candidate_evidence"
        ]
    ].head(6)
)

print("\nSaved:", review_path)
print("Manual review labels remain empty.")

In [ ]:
import pandas as pd
from pathlib import Path

review_rows = []

for _, row in case_0811_retrieval_df.iterrows():

    method = row["method"]
    sentence_map = case_0811_sentence_maps_v2[method]

    ids = [
        sid.strip()
        for sid in str(row["evidence_ids"]).split(",")
        if sid.strip()
    ]

    candidate_evidence = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in ids
    )

    full_numbered_note = "\n".join(
        f"{sid}: {sentence}"
        for sid, sentence in sentence_map.items()
    )

    review_rows.append({
        "case_id": row["case_id"],
        "fact_id": row["fact_id"],
        "method": method,
        "reference_fact": row["reference_fact"],
        "source_evidence": revised_facts_df.loc[
            revised_facts_df["fact_id"] == row["fact_id"],
            "source_evidence"
        ].iloc[0],
        "candidate_evidence": candidate_evidence,
        "full_numbered_note": full_numbered_note,
        "reviewed_status": "",
        "reviewed_evidence_ids": "",
        "review_notes": ""
    })

case_0811_review_df = pd.DataFrame(review_rows)

review_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_preservation_review_v1.csv"
)

case_0811_review_df.to_csv(review_path, index=False)

print("Review records:", len(case_0811_review_df))
print("Methods:", case_0811_review_df["method"].nunique())
print("Facts:", case_0811_review_df["fact_id"].nunique())

print("\nFirst six review records:")
display(
    case_0811_review_df[
        [
            "fact_id",
            "method",
            "reference_fact",
            "candidate_evidence"
        ]
    ].head(6)
)

print("\nSaved:", review_path)
print("Manual review labels remain empty.")

In [ ]:
import pandas as pd
import json
from pathlib import Path
from google.colab import drive

# Reconnect Google Drive if necessary
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Restore evidence-retrieval results
case_0811_retrieval_df = pd.read_csv(
    research_dir / "CASE_0811_evidence_retrieval_v1.csv"
).fillna({"evidence_ids": ""})

# Restore provisional reference facts
revised_facts_df = pd.read_csv(
    research_dir / "CASE_0811_reference_facts_v1.csv"
)

# Restore corrected sentence maps
case_0811_sentence_maps_v2 = json.loads(
    (
        research_dir / "CASE_0811_sentence_maps_v2.json"
    ).read_text(encoding="utf-8")
)

print("Evidence comparisons:", len(case_0811_retrieval_df))
print("Reference facts:", len(revised_facts_df))

for method, segments in case_0811_sentence_maps_v2.items():
    print(method, "segments:", len(segments))

assert len(case_0811_retrieval_df) == 40
assert len(revised_facts_df) == 20
assert set(case_0811_sentence_maps_v2) == {
    "Simple Baseline",
    "Structured Prompt"
}

print("\nCASE_0811 variables restored successfully.")

In [ ]:
import pandas as pd
from pathlib import Path

review_rows = []

for _, row in case_0811_retrieval_df.iterrows():

    method = row["method"]
    sentence_map = case_0811_sentence_maps_v2[method]

    ids = [
        sid.strip()
        for sid in str(row["evidence_ids"]).split(",")
        if sid.strip()
    ]

    candidate_evidence = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in ids
    )

    full_numbered_note = "\n".join(
        f"{sid}: {sentence}"
        for sid, sentence in sentence_map.items()
    )

    review_rows.append({
        "case_id": row["case_id"],
        "fact_id": row["fact_id"],
        "method": method,
        "reference_fact": row["reference_fact"],
        "source_evidence": revised_facts_df.loc[
            revised_facts_df["fact_id"] == row["fact_id"],
            "source_evidence"
        ].iloc[0],
        "candidate_evidence": candidate_evidence,
        "full_numbered_note": full_numbered_note,
        "reviewed_status": "",
        "reviewed_evidence_ids": "",
        "review_notes": ""
    })

case_0811_review_df = pd.DataFrame(review_rows)

review_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_preservation_review_v1.csv"
)

# Prevent overwriting existing manual reviews
if review_path.exists():
    existing = pd.read_csv(review_path)
    if existing["reviewed_status"].notna().any():
        raise RuntimeError(
            "Existing manual review labels detected. "
            "Original file preserved."
        )

case_0811_review_df.to_csv(review_path, index=False)

print("Review records:", len(case_0811_review_df))
print("Reference facts:", case_0811_review_df["fact_id"].nunique())
print("Methods:", case_0811_review_df["method"].nunique())

print("\nFirst six records:")
display(
    case_0811_review_df[
        [
            "fact_id",
            "method",
            "reference_fact",
            "candidate_evidence"
        ]
    ].head(6)
)

print("\nSaved:", review_path)
print("Manual review labels remain empty.")

In [ ]:
import json

PRESERVATION_EVALUATION_PROMPT = """
You are evaluating clinical fact preservation in a generated note.

Compare ONE reference clinical fact with the supplied
numbered generated note.

Classify the reference fact as:

present:
The complete clinical meaning is preserved, including
essential numerical values, anatomy, negation, and details.

partial:
The core clinical fact is represented, but one or more
important details are omitted or generalized.

missing:
The fact is not represented in the generated note.

contradicted:
The generated note explicitly conflicts with the fact.

Rules:
1. Use the entire supplied numbered note.
2. Do not infer undocumented details.
3. Equivalent clinical expressions are acceptable.
4. A general finding does not automatically establish
   preservation of a specific examination finding.
5. Never invent evidence IDs.
6. For missing facts, return an empty evidence_ids list.
7. Return a JSON object containing:
   status, evidence_ids, reason.

Reference fact:
{reference_fact}

Original source evidence:
{source_evidence}

Generated note:
{numbered_note}
"""

def evaluate_preservation_candidate(
    reference_fact,
    source_evidence,
    sentence_map
):
    numbered_note = "\n".join(
        f"{sid}: {sentence}"
        for sid, sentence in sentence_map.items()
    )

    prompt_text = PRESERVATION_EVALUATION_PROMPT.format(
        reference_fact=reference_fact,
        source_evidence=source_evidence,
        numbered_note=numbered_note
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt_text}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(
        response.choices[0].message.content
    )

    valid_statuses = {
        "present", "partial", "missing", "contradicted"
    }

    if result.get("status") not in valid_statuses:
        raise ValueError("Invalid preservation classification.")

    evidence_ids = result.get("evidence_ids", [])

    if not isinstance(evidence_ids, list):
        raise ValueError("Evidence IDs must be a list.")

    invalid_ids = [
        sid for sid in evidence_ids
        if sid not in sentence_map
    ]

    if invalid_ids:
        raise ValueError(f"Invalid evidence IDs: {invalid_ids}")

    if result["status"] == "missing" and evidence_ids:
        raise ValueError("Missing fact cannot have evidence IDs.")

    if not isinstance(result.get("reason"), str):
        raise ValueError("Missing classification reason.")

    return result


print("Controlled preservation evaluator ready.")
print("Model:", MODEL)
print("Pending comparisons:", len(case_0811_review_df))
print("No API requests made.")

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Restore the original experiment configuration
MODEL = "gpt-3.5-turbo"
TEMPERATURE = 0

# Restore the OpenAI API client
client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

# Verify the evaluator function is available
assert callable(evaluate_preservation_candidate)

print("Model:", MODEL)
print("Temperature:", TEMPERATURE)
print("API client initialized:", client is not None)
print(
    "Preservation evaluator available:",
    callable(evaluate_preservation_candidate)
)
print("Pending comparisons:", len(case_0811_review_df))
print("\nEvaluation configuration restored.")

In [ ]:
import pandas as pd
import json
from pathlib import Path

pilot_cases = [
    ("F001", "Simple Baseline"),
    ("F003", "Structured Prompt"),
    ("F007", "Structured Prompt"),
    ("F017", "Structured Prompt")
]

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_preservation_pilot_v1.json"
)

# Resume safely if some results have already been saved
if output_path.exists():
    pilot_results = json.loads(
        output_path.read_text(encoding="utf-8")
    )
else:
    pilot_results = []

for fact_id, method in pilot_cases:

    already_completed = any(
        item["fact_id"] == fact_id
        and item["method"] == method
        for item in pilot_results
    )

    if already_completed:
        print(f"{fact_id} / {method}: already saved")
        continue

    record = case_0811_review_df.loc[
        (case_0811_review_df["fact_id"] == fact_id)
        & (case_0811_review_df["method"] == method)
    ].iloc[0]

    result = evaluate_preservation_candidate(
        reference_fact=record["reference_fact"],
        source_evidence=record["source_evidence"],
        sentence_map=case_0811_sentence_maps_v2[method]
    )

    pilot_results.append({
        "fact_id": fact_id,
        "method": method,
        "status": result["status"],
        "evidence_ids": result["evidence_ids"],
        "reason": result["reason"]
    })

    # Save immediately after each API response
    output_path.write_text(
        json.dumps(
            pilot_results,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(f"{fact_id} / {method}: {result['status']}")

print("\nPILOT RESULTS")
display(pd.DataFrame(pilot_results))

print("\nComparisons completed:", len(pilot_results))
print("Saved:", output_path)

In [ ]:
import pandas as pd
import json
from pathlib import Path

pilot_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_preservation_pilot_v1.json"
)

pilot_results = json.loads(
    pilot_path.read_text(encoding="utf-8")
)

diagnostic_rows = []

for item in pilot_results:

    fact_id = item["fact_id"]
    method = item["method"]
    status = item["status"]
    evidence_ids = item["evidence_ids"]

    sentence_map = case_0811_sentence_maps_v2[method]

    flags = []

    if status in {"present", "partial", "contradicted"}:
        if not evidence_ids:
            flags.append("non_missing_status_without_evidence")

    if status == "missing" and evidence_ids:
        flags.append("missing_status_with_evidence")

    invalid_ids = [
        sid for sid in evidence_ids
        if sid not in sentence_map
    ]

    if invalid_ids:
        flags.append("invalid_evidence_ids")

    candidate_record = case_0811_retrieval_df.loc[
        (case_0811_retrieval_df["fact_id"] == fact_id)
        & (case_0811_retrieval_df["method"] == method)
    ].iloc[0]

    if not candidate_record["candidate_found"]:
        if status == "present":
            flags.append("present_without_retrieval_match")

    diagnostic_rows.append({
        "fact_id": fact_id,
        "method": method,
        "llm_status": status,
        "evidence_ids": ", ".join(evidence_ids),
        "flags": "; ".join(flags),
        "requires_review": bool(flags)
    })

pilot_diagnostic_df = pd.DataFrame(diagnostic_rows)

display(pilot_diagnostic_df)

print(
    "\nPredictions flagged:",
    int(pilot_diagnostic_df["requires_review"].sum()),
    "/",
    len(pilot_diagnostic_df)
)

print("\nConsistency inspection completed.")

In [ ]:
import pandas as pd
from pathlib import Path

reviewed_pilot = [
    {
        "fact_id": "F001",
        "method": "Simple Baseline",
        "reviewed_status": "present",
        "reviewed_evidence_ids": "S002",
        "review_notes": "Moderate gingivitis explicitly documented."
    },
    {
        "fact_id": "F003",
        "method": "Structured Prompt",
        "reviewed_status": "partial",
        "reviewed_evidence_ids": "S010",
        "review_notes": (
            "Calulus mentioned, but severity and "
            "specific anatomical location omitted."
        )
    },
    {
        "fact_id": "F007",
        "method": "Structured Prompt",
        "reviewed_status": "missing",
        "reviewed_evidence_ids": "",
        "review_notes": "No complication status documented."
    },
    {
        "fact_id": "F017",
        "method": "Structured Prompt",
        "reviewed_status": "present",
        "reviewed_evidence_ids": "S009",
        "review_notes": "All four impacted tooth numbers documented."
    }
]

reviewed_pilot_df = pd.DataFrame(reviewed_pilot)

# Correct the explanatory typo
reviewed_pilot_df["review_notes"] = (
    reviewed_pilot_df["review_notes"]
    .str.replace("Calulus", "Calculus", regex=False)
)

# Verify referenced evidence segments exist
for _, row in reviewed_pilot_df.iterrows():
    method = row["method"]
    ids = [
        sid.strip()
        for sid in row["reviewed_evidence_ids"].split(",")
        if sid.strip()
    ]

    assert all(
        sid in case_0811_sentence_maps_v2[method]
        for sid in ids
    )

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_pilot_review_provisional.csv"
)

reviewed_pilot_df.to_csv(output_path, index=False)

display(reviewed_pilot_df)

print("\nProvisional review records:", len(reviewed_pilot_df))
print("Evidence IDs verified.")
print("Saved:", output_path)

In [ ]:
import json
import pandas as pd

def validate_audit_prediction(prediction, sentence_map):
    """
    Validate an automated clinical preservation prediction.

    Structural validation only:
    - Valid classification
    - Evidence IDs exist
    - Non-missing labels have evidence
    - Missing labels have no evidence
    """

    valid_statuses = {
        "present", "partial", "missing", "contradicted"
    }

    status = prediction.get("status")
    evidence_ids = prediction.get("evidence_ids", [])

    issues = []

    if status not in valid_statuses:
        issues.append("invalid_status")

    if not isinstance(evidence_ids, list):
        issues.append("invalid_evidence_format")
        evidence_ids = []

    invalid_ids = [
        sid for sid in evidence_ids
        if sid not in sentence_map
    ]

    if invalid_ids:
        issues.append("unknown_evidence_ids")

    if status in {"present", "partial", "contradicted"}:
        if not evidence_ids:
            issues.append("missing_supporting_evidence")

    if status == "missing" and evidence_ids:
        issues.append("unexpected_evidence_for_missing")

    return {
        "original_status": status,
        "validation_status": (
            "requires_review" if issues else "structurally_valid"
        ),
        "issues": issues,
        "evidence_ids": evidence_ids
    }


validation_rows = []

for item in pilot_results:

    result = validate_audit_prediction(
        item,
        case_0811_sentence_maps_v2[item["method"]]
    )

    validation_rows.append({
        "fact_id": item["fact_id"],
        "method": item["method"],
        "llm_status": result["original_status"],
        "validation_status": result["validation_status"],
        "issues": ", ".join(result["issues"])
    })

evidence_gate_df = pd.DataFrame(validation_rows)

display(evidence_gate_df)

print(
    "\nFlagged for review:",
    int(
        (evidence_gate_df["validation_status"] == "requires_review").sum()
    ),
    "/",
    len(evidence_gate_df)
)

print("\nEvidence-validation gate test completed.")

In [ ]:
import pandas as pd

# Original LLM predictions
llm_df = pd.DataFrame(pilot_results)[
    ["fact_id", "method", "status"]
].rename(columns={
    "status": "llm_status"
})

# Provisional reviewed classifications
reference_df = reviewed_pilot_df[
    ["fact_id", "method", "reviewed_status"]
]

# Evidence-validation results
gate_df = evidence_gate_df[
    ["fact_id", "method", "validation_status"]
]

# Combine all three
pilot_evaluation_df = (
    llm_df
    .merge(
        reference_df,
        on=["fact_id", "method"],
        validate="one_to_one"
    )
    .merge(
        gate_df,
        on=["fact_id", "method"],
        validate="one_to_one"
    )
)

pilot_evaluation_df["label_agreement"] = (
    pilot_evaluation_df["llm_status"]
    == pilot_evaluation_df["reviewed_status"]
)

pilot_evaluation_df["incorrect_label"] = (
    ~pilot_evaluation_df["label_agreement"]
)

pilot_evaluation_df["flagged"] = (
    pilot_evaluation_df["validation_status"]
    == "requires_review"
)

display(pilot_evaluation_df)

print(
    "\nLabel agreement:",
    int(pilot_evaluation_df["label_agreement"].sum()),
    "/",
    len(pilot_evaluation_df)
)

print(
    "Incorrect labels flagged:",
    int(
        (
            pilot_evaluation_df["incorrect_label"]
            & pilot_evaluation_df["flagged"]
        ).sum()
    ),
    "/",
    int(pilot_evaluation_df["incorrect_label"].sum())
)

print(
    "Correct labels unnecessarily flagged:",
    int(
        (
            pilot_evaluation_df["label_agreement"]
            & pilot_evaluation_df["flagged"]
        ).sum()
    )
)

print("\nPilot evaluation completed.")

In [ ]:
import pandas as pd
from pathlib import Path

evidence_first_rows = []

for _, row in case_0811_review_df.iterrows():

    fact_id = row["fact_id"]
    method = row["method"]

    retrieval_record = case_0811_retrieval_df.loc[
        (case_0811_retrieval_df["fact_id"] == fact_id)
        & (case_0811_retrieval_df["method"] == method)
    ].iloc[0]

    evidence_ids = [
        sid.strip()
        for sid in str(retrieval_record["evidence_ids"]).split(",")
        if sid.strip()
    ]

    sentence_map = case_0811_sentence_maps_v2[method]

    candidate_passages = [
        {
            "evidence_id": sid,
            "text": sentence_map[sid]
        }
        for sid in evidence_ids
    ]

    evidence_first_rows.append({
        "case_id": "CASE_0811",
        "fact_id": fact_id,
        "method": method,
        "reference_fact": row["reference_fact"],
        "source_evidence": row["source_evidence"],
        "candidate_evidence_ids": evidence_ids,
        "candidate_passages": candidate_passages,
        "retrieval_status": (
            "candidates_found"
            if evidence_ids
            else "no_candidates"
        ),
        "requires_review": True
    })

evidence_first_df = pd.DataFrame(evidence_first_rows)

assert len(evidence_first_df) == 40

print("Evidence-first records:", len(evidence_first_df))

print("\nRetrieval status:")
display(
    pd.crosstab(
        evidence_first_df["method"],
        evidence_first_df["retrieval_status"]
    )
)

print("\nExample: F017 / Structured Prompt")

example = evidence_first_df.loc[
    (evidence_first_df["fact_id"] == "F017")
    & (evidence_first_df["method"] == "Structured Prompt")
].iloc[0]

print("Reference:", example["reference_fact"])
print("Candidate passages:", example["candidate_passages"])

print("\nEvidence-first preparation completed.")

In [ ]:
def evidence_supported_gate(
    prediction,
    sentence_map,
    candidate_ids
):
    """
    Validate a clinical preservation prediction
    against available numbered evidence.

    This checks evidence consistency only.
    It does not establish clinical correctness.
    """

    status = prediction.get("status")
    evidence_ids = prediction.get("evidence_ids", [])

    issues = []

    valid_statuses = {
        "present", "partial", "missing", "contradicted"
    }

    if status not in valid_statuses:
        issues.append("invalid_status")

    if not isinstance(evidence_ids, list):
        issues.append("invalid_evidence_format")
        evidence_ids = []

    if len(evidence_ids) != len(set(evidence_ids)):
        issues.append("duplicate_evidence_ids")

    if any(sid not in sentence_map for sid in evidence_ids):
        issues.append("unknown_evidence_ids")

    if status in {"present", "partial", "contradicted"}:
        if not evidence_ids:
            issues.append("missing_supporting_evidence")

    if status == "missing" and evidence_ids:
        issues.append("unexpected_evidence_for_missing")

    # Candidate retrieval is not exhaustive.
    # Flag differences for review, not automatic rejection.
    if evidence_ids and not set(evidence_ids).intersection(
        candidate_ids
    ):
        issues.append("evidence_outside_retrieval_candidates")

    return {
        "prediction": status,
        "evidence_ids": evidence_ids,
        "validation_status": (
            "requires_review" if issues
            else "provisional_evidence_linked"
        ),
        "issues": issues
    }


# Test with the previous problematic F017 prediction
test_prediction = {
    "status": "present",
    "evidence_ids": []
}

test_result = evidence_supported_gate(
    prediction=test_prediction,
    sentence_map=case_0811_sentence_maps_v2["Structured Prompt"],
    candidate_ids=["S009"]
)

print("Prediction:", test_result["prediction"])
print("Validation:", test_result["validation_status"])
print("Issues:", test_result["issues"])

assert test_result["validation_status"] == "requires_review"
assert "missing_supporting_evidence" in test_result["issues"]

print("\nEvidence-supported gate test passed.")

In [ ]:
import pandas as pd

gate_results = []

for prediction in pilot_results:

    fact_id = prediction["fact_id"]
    method = prediction["method"]

    # Retrieve candidate evidence IDs
    record = evidence_first_df.loc[
        (evidence_first_df["fact_id"] == fact_id)
        & (evidence_first_df["method"] == method)
    ].iloc[0]

    result = evidence_supported_gate(
        prediction=prediction,
        sentence_map=case_0811_sentence_maps_v2[method],
        candidate_ids=record["candidate_evidence_ids"]
    )

    gate_results.append({
        "fact_id": fact_id,
        "method": method,
        "llm_prediction": result["prediction"],
        "evidence_ids": ", ".join(result["evidence_ids"]),
        "validation_status": result["validation_status"],
        "issues": ", ".join(result["issues"])
    })

improved_gate_df = pd.DataFrame(gate_results)

display(improved_gate_df)

print("\nVALIDATION SUMMARY")
print(
    improved_gate_df["validation_status"].value_counts()
)

print("\nImproved evidence-gate evaluation completed.")

In [ ]:
import inspect
import shutil
import importlib
from pathlib import Path

module_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/clinical_auditor.py"
)

# Preserve the current module
backup_path = module_path.with_suffix(
    ".pre_evidence_gate.bak"
)

shutil.copy2(module_path, backup_path)

source = module_path.read_text(encoding="utf-8")

# Add the tested evidence-validation function
definition = "def evidence_supported_gate("

if definition not in source:
    source += "\n\n" + inspect.getsource(
        evidence_supported_gate
    )
    print("Evidence gate added.")
else:
    print("Evidence gate already exists.")

# Check syntax and save
compile(source, str(module_path), "exec")
module_path.write_text(source, encoding="utf-8")

# Reload from the saved file
importlib.invalidate_caches()
importlib.reload(clinical_auditor)

# Test the saved function
test_result = clinical_auditor.evidence_supported_gate(
    prediction={
        "status": "present",
        "evidence_ids": ["S009"]
    },
    sentence_map=case_0811_sentence_maps_v2[
        "Structured Prompt"
    ],
    candidate_ids=["S009"]
)

assert (
    test_result["validation_status"]
    == "provisional_evidence_linked"
)

print("Validation:", test_result["validation_status"])
print("Evidence IDs:", test_result["evidence_ids"])
print("Saved module verification passed.")
print("Backup:", backup_path.name)

In [ ]:
import sys
import importlib
from pathlib import Path

module_folder = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Make the saved module accessible
if str(module_folder) not in sys.path:
    sys.path.insert(0, str(module_folder))

# Import the module into the current Colab session
import clinical_auditor
importlib.invalidate_caches()
clinical_auditor = importlib.reload(clinical_auditor)

# Verify the evidence gate was saved
assert callable(clinical_auditor.evidence_supported_gate)

# Test the saved function
result = clinical_auditor.evidence_supported_gate(
    prediction={
        "status": "present",
        "evidence_ids": ["S009"]
    },
    sentence_map=case_0811_sentence_maps_v2[
        "Structured Prompt"
    ],
    candidate_ids=["S009"]
)

print("Module imported successfully.")
print("Validation:", result["validation_status"])
print("Evidence IDs:", result["evidence_ids"])

assert result["validation_status"] == "provisional_evidence_linked"

print("\nSaved evidence gate verification passed.")

In [ ]:
import pandas as pd
import json
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Load completed pilot results directly from Drive
pilot_path = (
    research_dir / "CASE_0811_preservation_pilot_v1.json"
)

completed_predictions = json.loads(
    pilot_path.read_text(encoding="utf-8")
)

completed_pairs = {
    (item["fact_id"], item["method"])
    for item in completed_predictions
}

# Exclude completed comparisons
remaining_evaluation_df = case_0811_review_df[
    ~case_0811_review_df.apply(
        lambda row: (
            row["fact_id"],
            row["method"]
        ) in completed_pairs,
        axis=1
    )
].copy()

# Include available retrieval evidence
remaining_evaluation_df = remaining_evaluation_df.merge(
    case_0811_retrieval_df[
        ["fact_id", "method", "candidate_found", "evidence_ids"]
    ].rename(columns={
        "evidence_ids": "candidate_evidence_ids"
    }),
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

queue_path = (
    research_dir / "CASE_0811_remaining_evaluation_queue.csv"
)

remaining_evaluation_df.to_csv(queue_path, index=False)

print("Total comparisons:", len(case_0811_review_df))
print("Completed pilot comparisons:", len(completed_pairs))
print("Remaining comparisons:", len(remaining_evaluation_df))

print("\nRemaining comparisons by method:")
print(remaining_evaluation_df["method"].value_counts())

print("\nSaved:", queue_path)

assert len(remaining_evaluation_df) == 36
assert not remaining_evaluation_df.duplicated(
    ["fact_id", "method"]
).any()

print("\nEvaluation queue prepared successfully.")

In [ ]:
import json

EVIDENCE_FIRST_PROMPT = """
You are a clinical fact preservation evaluator.

Evaluate exactly ONE reference fact against a numbered
generated clinical note.

Classifications:
present: All essential clinical details are preserved.
partial: The core fact is preserved, but important details
         are missing or generalized.
missing: The clinical fact is not documented.
contradicted: The note explicitly conflicts with the fact.

Rules:
1. Examine the ENTIRE generated note.
2. Retrieved passages are candidates, not guaranteed evidence.
3. Accept clinically equivalent wording.
4. Preserve distinctions involving negation, anatomy,
   numerical values, dose, duration, and treatment intent.
5. Never infer undocumented clinical details.
6. Do not confuse unrelated findings.
7. For present, partial, or contradicted, cite at least
   one supporting numbered segment.
8. For missing, evidence_ids must be [].
9. Return only a JSON object with:
   status, evidence_ids, reason.

Reference fact:
{reference_fact}

Original source evidence:
{source_evidence}

Retrieved candidate passages:
{candidate_passages}

Complete numbered generated note:
{numbered_note}
"""

def evaluate_evidence_first(record, sentence_map):
    """
    Produce an evidence-linked LLM prediction.

    Returns an explicit validation outcome.
    Does not independently verify clinical accuracy.
    """

    candidate_ids = [
        sid.strip()
        for sid in str(
            record["candidate_evidence_ids"]
        ).split(",")
        if sid.strip() and sid.strip() != "nan"
    ]

    candidate_passages = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in candidate_ids
        if sid in sentence_map
    )

    numbered_note = "\n".join(
        f"{sid}: {sentence}"
        for sid, sentence in sentence_map.items()
    )

    prompt_text = EVIDENCE_FIRST_PROMPT.format(
        reference_fact=record["reference_fact"],
        source_evidence=record["source_evidence"],
        candidate_passages=(
            candidate_passages or "No literal candidates retrieved."
        ),
        numbered_note=numbered_note
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt_text}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    prediction = json.loads(
        response.choices[0].message.content
    )

    gate_result = clinical_auditor.evidence_supported_gate(
        prediction=prediction,
        sentence_map=sentence_map,
        candidate_ids=candidate_ids
    )

    return {
        "status": gate_result["prediction"],
        "evidence_ids": gate_result["evidence_ids"],
        "reason": prediction.get("reason", ""),
        "validation_status": gate_result["validation_status"],
        "validation_issues": gate_result["issues"]
    }


print("Evidence-first evaluator ready.")
print("Remaining comparisons:", len(remaining_evaluation_df))
print("No API requests made.")

In [ ]:
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

required_files = [
    "clinical_auditor.py",
    "CASE_0811_reference_facts_v1.csv",
    "CASE_0811_sentence_maps_v2.json",
    "CASE_0811_evidence_retrieval_v1.csv",
    "CASE_0811_preservation_review_v1.csv",
    "CASE_0811_remaining_evaluation_queue.csv"
]

print("RESEARCH FILE VERIFICATION\n")

for filename in required_files:
    exists = (research_dir / filename).exists()
    print(f"{filename}: {'FOUND' if exists else 'MISSING'}")

print("\nVerification completed.")

In [ ]:

from google.colab import drive
from pathlib import Path

# Reconnect Google Drive
drive.mount("/content/drive", force_remount=True)

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

print("\nGoogle Drive mounted.")
print("Research folder exists:", research_dir.exists())

if research_dir.exists():
    files = list(research_dir.iterdir())

    print("Files and folders found:", len(files))

    for item in sorted(files)[:20]:
        print(item.name)
else:
    print("Research folder not found at the expected location.")


In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Restore the saved clinical reference facts
revised_facts_df = pd.read_csv(
    research_dir / "CASE_0811_reference_facts_v1.csv"
)

# Restore the corrected sentence maps
case_0811_sentence_maps_v2 = json.loads(
    (
        research_dir / "CASE_0811_sentence_maps_v2.json"
    ).read_text(encoding="utf-8")
)

# Restore evidence retrieval
case_0811_retrieval_df = pd.read_csv(
    research_dir / "CASE_0811_evidence_retrieval_v1.csv"
).fillna({"evidence_ids": ""})

# Restore the review worksheet
case_0811_review_df = pd.read_csv(
    research_dir / "CASE_0811_preservation_review_v1.csv"
)

# Restore the remaining evaluation queue
remaining_evaluation_df = pd.read_csv(
    research_dir / "CASE_0811_remaining_evaluation_queue.csv"
).fillna({"candidate_evidence_ids": ""})

# Restore saved pilot results
pilot_results = json.loads(
    (
        research_dir / "CASE_0811_preservation_pilot_v1.json"
    ).read_text(encoding="utf-8")
)

# Restore the saved auditor module
if str(research_dir) not in sys.path:
    sys.path.insert(0, str(research_dir))

import clinical_auditor

assert callable(clinical_auditor.evidence_supported_gate)

print("Reference facts:", len(revised_facts_df))
print("Retrieval comparisons:", len(case_0811_retrieval_df))
print("Review records:", len(case_0811_review_df))
print("Remaining evaluations:", len(remaining_evaluation_df))
print("Saved pilot results:", len(pilot_results))
print(
    "Evidence gate available:",
    callable(clinical_auditor.evidence_supported_gate)
)

print("\nEvaluation environment restored successfully.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

pilot_path = (
    research_dir / "CASE_0811_evidence_first_pilot.json"
)

if pilot_path.exists():

    evidence_first_pilot_results = json.loads(
        pilot_path.read_text(encoding="utf-8")
    )

    print("Saved evidence-first pilot found.")

    display(
        pd.DataFrame(evidence_first_pilot_results)[
            [
                "fact_id",
                "method",
                "status",
                "evidence_ids",
                "validation_status",
                "validation_issues"
            ]
        ]
    )

    print(
        "\nRecovered predictions:",
        len(evidence_first_pilot_results)
    )

else:
    evidence_first_pilot_results = []

    print("No saved evidence-first pilot found.")
    print("No API requests were made.")

print("\nRecovery check completed.")

In [ ]:
import json
from openai import OpenAI
from google.colab import userdata

MODEL = "gpt-3.5-turbo"

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

EVIDENCE_FIRST_PROMPT = """
You are evaluating ONE reference clinical fact against a
numbered generated clinical note.

Classification:
present: Complete clinical meaning is preserved.
partial: Core fact preserved but important details omitted.
missing: Clinical fact not documented.
contradicted: Note explicitly conflicts with the fact.

Requirements:
1. Examine the complete generated note.
2. Use retrieved passages as candidate evidence.
3. Accept clinically equivalent wording.
4. Preserve negation, anatomy, numbers, and clinical context.
5. Never infer undocumented details.
6. Every present, partial, or contradicted classification
   must cite at least one supporting note segment.
7. Missing classifications must have no evidence IDs.
8. Return JSON with status, evidence_ids, reason.

Reference fact:
{reference_fact}

Original source evidence:
{source_evidence}

Retrieved candidate passages:
{candidate_passages}

Complete numbered generated note:
{numbered_note}
"""

def evaluate_evidence_first(record, sentence_map):

    candidate_ids = [
        sid.strip()
        for sid in str(
            record["candidate_evidence_ids"]
        ).split(",")
        if sid.strip() and sid.strip() != "nan"
    ]

    candidate_passages = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in candidate_ids
        if sid in sentence_map
    )

    numbered_note = "\n".join(
        f"{sid}: {text}"
        for sid, text in sentence_map.items()
    )

    prompt_text = EVIDENCE_FIRST_PROMPT.format(
        reference_fact=record["reference_fact"],
        source_evidence=record["source_evidence"],
        candidate_passages=candidate_passages or "None retrieved.",
        numbered_note=numbered_note
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt_text}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    prediction = json.loads(
        response.choices[0].message.content
    )

    validation = clinical_auditor.evidence_supported_gate(
        prediction,
        sentence_map,
        candidate_ids
    )

    return {
        "status": validation["prediction"],
        "evidence_ids": validation["evidence_ids"],
        "reason": prediction.get("reason", ""),
        "validation_status": validation["validation_status"],
        "validation_issues": validation["issues"]
    }

print("Model:", MODEL)
print("API client initialized:", client is not None)
print("Evidence-first evaluator ready.")
print("Remaining comparisons:", len(remaining_evaluation_df))
print("No API requests made.")

In [ ]:
import json
import pandas as pd

test_pairs = [
    ("F007", "Simple Baseline"),
    ("F003", "Simple Baseline"),
    ("F013", "Structured Prompt")
]

output_path = (
    research_dir / "CASE_0811_evidence_first_pilot.json"
)

# Recover any previously completed evaluations
if output_path.exists():
    test_results = json.loads(
        output_path.read_text(encoding="utf-8")
    )
else:
    test_results = []

for fact_id, method in test_pairs:

    if any(
        item["fact_id"] == fact_id
        and item["method"] == method
        for item in test_results
    ):
        print(f"{fact_id} / {method}: already saved")
        continue

    record = remaining_evaluation_df.loc[
        (remaining_evaluation_df["fact_id"] == fact_id)
        & (remaining_evaluation_df["method"] == method)
    ].iloc[0]

    result = evaluate_evidence_first(
        record,
        case_0811_sentence_maps_v2[method]
    )

    test_results.append({
        "fact_id": fact_id,
        "method": method,
        **result
    })

    # Save each successful response immediately
    output_path.write_text(
        json.dumps(
            test_results,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"{fact_id} / {method}:",
        result["status"],
        "|",
        result["validation_status"]
    )

print("\nPILOT RESULTS")
display(pd.DataFrame(test_results))

print("\nCompleted evaluations:", len(test_results))
print("Saved:", output_path)

In [ ]:
import pandas as pd

# Provisional assessments from the source and note excerpts
expected_pilot = {
    ("F007", "Simple Baseline"): "present",
    ("F003", "Simple Baseline"): "partial",
    ("F013", "Structured Prompt"): "missing"
}

pilot_check_rows = []

for item in test_results:

    key = (item["fact_id"], item["method"])
    expected = expected_pilot[key]

    pilot_check_rows.append({
        "fact_id": item["fact_id"],
        "method": item["method"],
        "llm_status": item["status"],
        "provisional_review": expected,
        "validation_status": item["validation_status"],
        "label_agreement": item["status"] == expected
    })

evidence_first_check_df = pd.DataFrame(pilot_check_rows)

display(evidence_first_check_df)

print(
    "\nProvisional label agreement:",
    int(evidence_first_check_df["label_agreement"].sum()),
    "/",
    len(evidence_first_check_df)
)

print("\nDisagreements requiring review:")
display(
    evidence_first_check_df[
        ~evidence_first_check_df["label_agreement"]
    ]
)

print("\nPilot comparison completed.")

In [ ]:
import re

def verify_explicit_negative_fact(
    concept,
    sentence_map
):
    """
    Verify whether a specifically named clinical
    concept is explicitly negated in the note.

    This is a conservative diagnostic check,
    not a complete clinical fact classifier.
    """

    evidence_ids = []
    contradictory_ids = []

    for sid, sentence in sentence_map.items():

        normalized_sentence = (
            clinical_auditor.normalize_clinical_terms(sentence)
        )
        normalized_concept = (
            clinical_auditor.normalize_clinical_terms(concept)
        )

        pattern = (
            r"(?<!\w)"
            + re.escape(normalized_concept)
            + r"(?!\w)"
        )

        if not re.search(pattern, normalized_sentence):
            continue

        polarity = clinical_auditor.detect_negation_scope_v2(
            sentence,
            concept
        )

        if polarity == "negated":
            evidence_ids.append(sid)

        elif polarity == "affirmed":
            contradictory_ids.append(sid)

    if evidence_ids and not contradictory_ids:
        status = "explicit_negation_preserved"
    elif contradictory_ids:
        status = "requires_review"
    else:
        status = "not_verified"

    return {
        "status": status,
        "evidence_ids": evidence_ids,
        "contradictory_ids": contradictory_ids
    }


# Test F007: no complications
result = verify_explicit_negative_fact(
    concept="complications",
    sentence_map=case_0811_sentence_maps_v2[
        "Simple Baseline"
    ]
)

print("Verification:", result["status"])
print("Evidence IDs:", result["evidence_ids"])
print("Contradictory IDs:", result["contradictory_ids"])

assert result["status"] == "explicit_negation_preserved"
assert result["evidence_ids"] == ["S003"]

print("\nNegative-fact verification test passed.")

In [ ]:
import pandas as pd
from pathlib import Path

# Retain the original evidence-first LLM predictions
hybrid_pilot_df = evidence_first_check_df.copy()

hybrid_pilot_df["deterministic_finding"] = "not_evaluated"
hybrid_pilot_df["deterministic_evidence_ids"] = ""

# Add the independently executed rule-driven result for F007
mask = (
    (hybrid_pilot_df["fact_id"] == "F007")
    & (hybrid_pilot_df["method"] == "Simple Baseline")
)

hybrid_pilot_df.loc[
    mask, "deterministic_finding"
] = result["status"]

hybrid_pilot_df.loc[
    mask, "deterministic_evidence_ids"
] = ", ".join(result["evidence_ids"])

# Flag disagreements for subsequent review
hybrid_pilot_df["requires_review"] = (
    ~hybrid_pilot_df["label_agreement"]
    | hybrid_pilot_df["validation_status"].eq("requires_review")
)

report_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_hybrid_pilot_report.csv"
)

hybrid_pilot_df.to_csv(report_path, index=False)

display(hybrid_pilot_df)

print("\nComparisons:", len(hybrid_pilot_df))
print(
    "Disagreements requiring review:",
    int(hybrid_pilot_df["requires_review"].sum())
)
print("Saved:", report_path)

In [ ]:
import json
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

pilot_files = [
    "CASE_0811_preservation_pilot_v1.json",
    "CASE_0811_evidence_first_pilot.json"
]

all_predictions = []

for filename in pilot_files:
    path = research_dir / filename

    if not path.exists():
        print("Missing pilot file:", filename)
        continue

    records = json.loads(path.read_text(encoding="utf-8"))

    for record in records:
        all_predictions.append({
            **record,
            "prediction_source": filename
        })

combined_predictions_df = pd.DataFrame(all_predictions)

# Verify uniqueness across the two pilots
duplicate_count = combined_predictions_df.duplicated(
    ["fact_id", "method"]
).sum()

assert duplicate_count == 0, (
    "Duplicate fact-method predictions detected."
)

completed_pairs = set(
    zip(
        combined_predictions_df["fact_id"],
        combined_predictions_df["method"]
    )
)

remaining_pairs = {
    (row["fact_id"], row["method"])
    for _, row in case_0811_review_df.iterrows()
} - completed_pairs

print("Saved predictions:", len(combined_predictions_df))
print("Duplicate predictions:", duplicate_count)
print("Remaining comparisons:", len(remaining_pairs))

display(
    combined_predictions_df[
        [
            "fact_id",
            "method",
            "status",
            "evidence_ids",
            "prediction_source"
        ]
    ]
)

output_path = (
    research_dir / "CASE_0811_combined_pilot_predictions.csv"
)

combined_predictions_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

In [ ]:
import pandas as pd

# Identify completed fact-method combinations
completed_pairs = set(
    zip(
        combined_predictions_df["fact_id"],
        combined_predictions_df["method"]
    )
)

# Select comparisons not yet evaluated
final_queue_df = case_0811_review_df.loc[
    ~case_0811_review_df.apply(
        lambda row: (
            row["fact_id"],
            row["method"]
        ) in completed_pairs,
        axis=1
    )
].copy()

# Add retrieved evidence IDs
retrieval_columns = case_0811_retrieval_df[
    ["fact_id", "method", "evidence_ids"]
].rename(
    columns={"evidence_ids": "candidate_evidence_ids"}
)

final_queue_df = final_queue_df.merge(
    retrieval_columns,
    on=["fact_id", "method"],
    how="left",
    validate="one_to_one"
)

final_queue_df["candidate_evidence_ids"] = (
    final_queue_df["candidate_evidence_ids"].fillna("")
)

# Save the queue
queue_path = (
    research_dir / "CASE_0811_final_evaluation_queue.csv"
)

final_queue_df.to_csv(queue_path, index=False)

print("Remaining comparisons:", len(final_queue_df))

print("\nComparisons by method:")
print(final_queue_df["method"].value_counts())

print("\nSaved:", queue_path)

assert len(final_queue_df) == 33

print("\nConsolidated evaluation queue ready.")

In [ ]:
import json
import pandas as pd
from pathlib import Path

output_path = (
    research_dir / "CASE_0811_remaining_predictions.json"
)

# Recover previously completed results
if output_path.exists():
    batch_results = json.loads(
        output_path.read_text(encoding="utf-8")
    )
else:
    batch_results = []

completed = {
    (item["fact_id"], item["method"])
    for item in batch_results
}

# Select up to five unevaluated comparisons
pending = final_queue_df.loc[
    ~final_queue_df.apply(
        lambda row: (
            row["fact_id"],
            row["method"]
        ) in completed,
        axis=1
    )
].head(5)

print("Comparisons in this batch:", len(pending))

for _, record in pending.iterrows():

    fact_id = record["fact_id"]
    method = record["method"]

    result = evaluate_evidence_first(
        record,
        case_0811_sentence_maps_v2[method]
    )

    batch_results.append({
        "fact_id": fact_id,
        "method": method,
        **result
    })

    # Save immediately after each completed request
    output_path.write_text(
        json.dumps(
            batch_results,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"{fact_id} / {method}:",
        result["status"],
        "|",
        result["validation_status"]
    )

print("\nBATCH SUMMARY")

display(
    pd.DataFrame(batch_results)[
        [
            "fact_id",
            "method",
            "status",
            "evidence_ids",
            "validation_status"
        ]
    ].tail(5)
)

print("\nTotal additional predictions saved:", len(batch_results))
print("Remaining from queue:", len(final_queue_df) - len(batch_results))
print("Saved:", output_path)

In [ ]:
import pandas as pd

inspection_rows = []

for item in batch_results[-5:]:

    method = item["method"]
    sentence_map = case_0811_sentence_maps_v2[method]

    cited_passages = "\n".join(
        f"{sid}: {sentence_map[sid]}"
        for sid in item["evidence_ids"]
    )

    reference_fact = revised_facts_df.loc[
        revised_facts_df["fact_id"] == item["fact_id"],
        "fact"
    ].iloc[0]

    inspection_rows.append({
        "fact_id": item["fact_id"],
        "method": method,
        "reference_fact": reference_fact,
        "llm_status": item["status"],
        "llm_reason": item["reason"],
        "cited_evidence": cited_passages
    })

batch_inspection_df = pd.DataFrame(inspection_rows)

pd.set_option("display.max_colwidth", 300)

display(batch_inspection_df)

print("\nFirst-batch inspection completed.")

In [ ]:
import re
import pandas as pd

# Explicit requirements for these five diagnostic comparisons
scope_requirements = {
    "F001": [
        r"\bmoderate gingivitis\b"
    ],
    "F002": [
        r"\bimpacted wisdom teeth\b"
    ],
    "F005": [
        r"\bwisdom teeth (?:extraction|will be removed)\b",
        r"\bdental rehabilitation\b"
    ]
}

rows = []

for item in batch_results[-5:]:

    fact_id = item["fact_id"]
    method = item["method"]

    note_text = " ".join(
        case_0811_sentence_maps_v2[method].values()
    )

    checks = {
        pattern: bool(
            re.search(pattern, note_text, re.IGNORECASE)
        )
        for pattern in scope_requirements[fact_id]
    }

    rows.append({
        "fact_id": fact_id,
        "method": method,
        "llm_status": item["status"],
        "required_components_found": sum(checks.values()),
        "required_components_total": len(checks),
        "all_required_components_found": all(checks.values())
    })

scope_check_df = pd.DataFrame(rows)

display(scope_check_df)

print(
    "\nAll required components detected:",
    int(scope_check_df["all_required_components_found"].sum()),
    "/",
    len(scope_check_df)
)

print("\nReference-scope verification completed.")

In [ ]:
import pandas as pd
from pathlib import Path

scope_review_rows = []

for item in batch_results[-5:]:

    fact_id = item["fact_id"]
    method = item["method"]

    # Provisional assessment after inspecting
    # the complete generated notes
    proposed_status = "present"

    if fact_id == "F005":
        evidence_ids = {
            "Simple Baseline": ["S005", "S011"],
            "Structured Prompt": ["S005", "S017"]
        }[method]
    else:
        evidence_ids = item["evidence_ids"]

    sentence_map = case_0811_sentence_maps_v2[method]

    assert all(sid in sentence_map for sid in evidence_ids)

    scope_review_rows.append({
        "fact_id": fact_id,
        "method": method,
        "llm_status": item["status"],
        "provisional_review_status": proposed_status,
        "review_evidence_ids": ", ".join(evidence_ids),
        "review_origin": "source_and_note_inspection",
        "requires_independent_review": True
    })

scope_review_df = pd.DataFrame(scope_review_rows)

output_path = Path(
    "/content/drive/MyDrive/"
    "Clinical_Auditor_Research/"
    "CASE_0811_scope_review_provisional.csv"
)

scope_review_df.to_csv(output_path, index=False)

display(scope_review_df)

print("\nRecords saved:", len(scope_review_df))
print("Original LLM predictions preserved.")
print("Saved:", output_path)

In [ ]:
import json
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Recover all completed prediction identifiers directly from Drive
completed_files = [
    "CASE_0811_preservation_pilot_v1.json",
    "CASE_0811_evidence_first_pilot.json",
    "CASE_0811_remaining_predictions.json"
]

completed_pairs = set()

for filename in completed_files:
    records = json.loads(
        (research_dir / filename).read_text(encoding="utf-8")
    )

    for record in records:
        completed_pairs.add(
            (record["fact_id"], record["method"])
        )

remaining_rows = []

for _, row in case_0811_review_df.iterrows():
    key = (row["fact_id"], row["method"])

    if key in completed_pairs:
        continue

    retrieval = case_0811_retrieval_df.loc[
        (case_0811_retrieval_df["fact_id"] == row["fact_id"])
        & (case_0811_retrieval_df["method"] == row["method"])
    ].iloc[0]

    remaining_rows.append({
        "case_id": row["case_id"],
        "fact_id": row["fact_id"],
        "method": row["method"],
        "reference_fact": row["reference_fact"],
        "source_evidence": row["source_evidence"],
        "candidate_evidence": row["candidate_evidence"],
        "full_numbered_note": row["full_numbered_note"],
        "candidate_found": retrieval["candidate_found"],
        "reviewed_status": "",
        "reviewed_evidence_ids": "",
        "review_notes": ""
    })

remaining_review_df = pd.DataFrame(remaining_rows)

assert len(remaining_review_df) == 28

output_path = (
    research_dir / "CASE_0811_remaining_review_28.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing review worksheet preserved; no overwrite performed."
    )

remaining_review_df.to_csv(output_path, index=False)

print("Remaining comparisons:", len(remaining_review_df))
print("Manual labels assigned:", 0)

print("\nBy method:")
print(remaining_review_df["method"].value_counts())

print("\nSaved:", output_path)
print("Review worksheet created successfully.")

In [ ]:
import pandas as pd

# Summarize the 28 remaining comparisons
review_overview = remaining_review_df[
    [
        "fact_id",
        "method",
        "reference_fact",
        "candidate_found"
    ]
].copy()

# Group the two methods into columns
review_matrix = review_overview.pivot(
    index=["fact_id", "reference_fact"],
    columns="method",
    values="candidate_found"
).reset_index()

review_matrix.columns.name = None

display(review_matrix)

print("\nUnique facts requiring review:",
      remaining_review_df["fact_id"].nunique())

print("Remaining comparisons:",
      len(remaining_review_df))

print("\nReview overview completed.")

In [ ]:
import pandas as pd
from pathlib import Path

# Provisional classifications from the previously displayed
# source facts and complete CASE_0811 generated notes.
# These are not independently adjudicated clinical labels.

provisional_labels = {
    "F006": ("present", "present"),
    "F008": ("present", "present"),
    "F009": ("present", "present"),
    "F010": ("present", "present"),
    "F011": ("missing", "missing"),
    "F012": ("missing", "missing"),
    "F013": ("missing", None),
    "F014": ("partial", "partial"),
    "F015": ("present", "present"),
    "F016": ("missing", "missing"),
    "F017": ("partial", None),
    "F018": ("present", "present"),
    "F019": ("partial", "partial"),
    "F020": ("partial", "missing"),
    "F021": ("present", "present")
}

methods = ["Simple Baseline", "Structured Prompt"]

label_lookup = {
    (fact_id, method): status
    for fact_id, statuses in provisional_labels.items()
    for method, status in zip(methods, statuses)
    if status is not None
}

remaining_review_df["reviewed_status"] = (
    remaining_review_df.apply(
        lambda row: label_lookup.get(
            (row["fact_id"], row["method"])
        ),
        axis=1
    )
)

# Record candidate evidence IDs without presenting them
# as independently verified supporting evidence.
remaining_review_df["reviewed_evidence_ids"] = (
    remaining_review_df["candidate_evidence"].apply(
        lambda text: ", ".join(
            line.split(":", 1)[0]
            for line in str(text).splitlines()
            if line.startswith("S") and ":" in line
        )
    )
)

remaining_review_df["review_notes"] = (
    "Provisional assessment; requires independent review. "
    "Candidate evidence IDs are not adjudicated."
)

assert remaining_review_df["reviewed_status"].notna().all()

output_path = (
    research_dir / "CASE_0811_remaining_review_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional review preserved."
    )

remaining_review_df.to_csv(output_path, index=False)

display(
    remaining_review_df[
        [
            "fact_id",
            "method",
            "reviewed_status",
            "reviewed_evidence_ids"
        ]
    ]
)

print("\nProvisional assessments:", len(remaining_review_df))
print("Unclassified comparisons:",
      remaining_review_df["reviewed_status"].isna().sum())
print("Saved:", output_path)

In [ ]:
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Load the 28 provisional assessments
remaining = pd.read_csv(
    research_dir / "CASE_0811_remaining_review_provisional.csv"
)

# Previously inspected pilot assessments
pilot_review_lookup = {
    ("F001", "Simple Baseline"): ("present", "S002"),
    ("F003", "Structured Prompt"): ("partial", "S010"),
    ("F007", "Structured Prompt"): ("missing", ""),
    ("F017", "Structured Prompt"): ("present", "S009"),
    ("F007", "Simple Baseline"): ("present", "S003"),
    ("F003", "Simple Baseline"): ("partial", "S006"),
    ("F013", "Structured Prompt"): ("missing", ""),
    ("F001", "Structured Prompt"): ("present", "S003"),
    ("F002", "Simple Baseline"): ("present", "S002, S011"),
    ("F002", "Structured Prompt"): ("present", "S002, S009"),
    ("F005", "Simple Baseline"): ("present", "S005, S011"),
    ("F005", "Structured Prompt"): ("present", "S005, S017")
}

pilot_rows = []

for (fact_id, method), (status, evidence) in pilot_review_lookup.items():
    pilot_rows.append({
        "fact_id": fact_id,
        "method": method,
        "reviewed_status": status,
        "reviewed_evidence_ids": evidence,
        "review_notes": "Provisional source-and-note inspection."
    })

pilot_review_df = pd.DataFrame(pilot_rows)

# Keep only the relevant columns from both sources
columns = [
    "fact_id",
    "method",
    "reviewed_status",
    "reviewed_evidence_ids",
    "review_notes"
]

combined_review_df = pd.concat(
    [remaining[columns], pilot_review_df[columns]],
    ignore_index=True
)

assert len(combined_review_df) == 40
assert not combined_review_df.duplicated(
    ["fact_id", "method"]
).any()

assert set(combined_review_df["reviewed_status"]) <= {
    "present", "partial", "missing", "contradicted"
}

# Add provenance explicitly
combined_review_df["annotation_status"] = (
    "provisional_not_independently_adjudicated"
)

output_path = (
    research_dir / "CASE_0811_combined_review_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing combined review preserved."
    )

combined_review_df.to_csv(output_path, index=False)

print("Combined comparisons:", len(combined_review_df))
print("Unique facts:", combined_review_df["fact_id"].nunique())
print("Duplicate comparisons:", combined_review_df.duplicated(
    ["fact_id", "method"]
).sum())

print("\nProvisional label distribution:")
display(
    pd.crosstab(
        combined_review_df["method"],
        combined_review_df["reviewed_status"]
    )
)

print("\nSaved:", output_path)
print("Independent adjudication: not yet performed.")

In [ ]:
import pandas as pd

status_weights = {
    "present": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "contradicted": 0.0
}

metric_rows = []

for method, group in combined_review_df.groupby("method"):

    total = len(group)
    counts = group["reviewed_status"].value_counts()

    present = int(counts.get("present", 0))
    partial = int(counts.get("partial", 0))
    missing = int(counts.get("missing", 0))
    contradicted = int(counts.get("contradicted", 0))

    weighted_score = (
        group["reviewed_status"]
        .map(status_weights)
        .mean()
    )

    metric_rows.append({
        "case_id": "CASE_0811",
        "method": method,
        "total_facts": total,
        "present": present,
        "partial": partial,
        "missing": missing,
        "contradicted": contradicted,
        "complete_preservation_rate": round(present / total, 4),
        "partial_preservation_rate": round(partial / total, 4),
        "omission_rate": round(missing / total, 4),
        "weighted_preservation_score": round(weighted_score, 4),
        "evaluation_status": "provisional"
    })

case_0811_metrics_df = pd.DataFrame(metric_rows)

display(case_0811_metrics_df)

output_path = (
    research_dir / "CASE_0811_provisional_metrics.csv"
)

case_0811_metrics_df.to_csv(output_path, index=False)

print("\nMetrics saved:", output_path)
print("Evaluation status: provisional")

In [ ]:
import pandas as pd

# Compare classifications for the same reference facts
fact_comparison_df = combined_review_df.pivot(
    index="fact_id",
    columns="method",
    values="reviewed_status"
).reset_index()

fact_comparison_df.columns.name = None

# Add reference fact descriptions
fact_comparison_df = fact_comparison_df.merge(
    revised_facts_df[["fact_id", "fact", "section"]],
    on="fact_id",
    validate="one_to_one"
)

# Identify disagreements between methods
different_facts_df = fact_comparison_df[
    fact_comparison_df["Simple Baseline"]
    != fact_comparison_df["Structured Prompt"]
].copy()

display(
    different_facts_df[
        [
            "fact_id",
            "section",
            "fact",
            "Simple Baseline",
            "Structured Prompt"
        ]
    ]
)

print("\nFacts with different classifications:",
      len(different_facts_df))

print(
    "Facts with identical classifications:",
    len(fact_comparison_df) - len(different_facts_df)
)

print("\nFact-level comparison completed.")

In [ ]:
import pandas as pd

# Preserve the fact-level comparison
case_0811_difference_report_df = different_facts_df.copy()

review_priorities = {
    "F007": (
        "Verify explicit documentation of no complications "
        "in both generated notes."
    ),
    "F017": (
        "Verify whether the four specific tooth numbers "
        "are preserved."
    ),
    "F020": (
        "Review incomplete preservation and the conflicting "
        "airway documentation in the original source."
    )
}

case_0811_difference_report_df["review_priority"] = (
    case_0811_difference_report_df["fact_id"]
    .map(review_priorities)
)

case_0811_difference_report_df["annotation_status"] = (
    "provisional_requires_independent_review"
)

output_path = (
    research_dir / "CASE_0811_method_difference_report.csv"
)

case_0811_difference_report_df.to_csv(
    output_path,
    index=False
)

display(
    case_0811_difference_report_df[
        [
            "fact_id",
            "Simple Baseline",
            "Structured Prompt",
            "review_priority"
        ]
    ]
)

print("\nDifferences documented:",
      len(case_0811_difference_report_df))

print("Saved:", output_path)
print("Original classifications unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

remaining_case_ids = [
    "CASE_1824",
    "CASE_2043",
    "CASE_2148",
    "CASE_2176"
]

case_rows = []

for case_id in remaining_case_ids:

    path = research_dir / f"{case_id}_generated_notes.csv"

    assert path.exists(), f"Missing generated notes: {case_id}"

    notes = pd.read_csv(path)

    assert len(notes) == 2
    assert set(notes["method"]) == {
        "Simple Baseline",
        "Structured Prompt"
    }
    assert notes["transcription"].notna().all()
    assert notes["generated_note"].notna().all()

    # Both methods must have the same original transcription
    assert notes["transcription"].nunique() == 1

    case_rows.append({
        "case_id": case_id,
        "methods": len(notes),
        "source_words": len(
            notes["transcription"].iloc[0].split()
        ),
        "baseline_words": len(
            notes.loc[
                notes["method"] == "Simple Baseline",
                "generated_note"
            ].iloc[0].split()
        ),
        "structured_words": len(
            notes.loc[
                notes["method"] == "Structured Prompt",
                "generated_note"
            ].iloc[0].split()
        ),
        "ready": True
    })

remaining_cases_summary_df = pd.DataFrame(case_rows)

display(remaining_cases_summary_df)

print("\nCases verified:", len(remaining_cases_summary_df))
print("Generated notes verified:", 2 * len(remaining_cases_summary_df))
print("All cases ready:", remaining_cases_summary_df["ready"].all())

In [ ]:
import json
import pandas as pd
from pathlib import Path

EXTRACTION_PROMPT_V2 = """
You are extracting clinical reference facts for a
documentation-preservation research study.

Use ONLY the original clinical transcription.

Extract a comprehensive checklist of distinct clinical facts.

Include, when documented:
- Presenting symptoms and duration
- Positive and negative clinical findings
- Medical and surgical history
- Medications, doses, routes, and frequency
- Allergies and reactions
- Examination findings
- Laboratory and diagnostic results
- Diagnoses and assessments
- Procedures and treatments
- Treatment plans, follow-up, and disposition

Rules:
1. Do not invent information.
2. Preserve negation, numbers, units, and laterality.
3. Include important negative findings.
4. Avoid duplicate facts.
5. Preserve all essential details of compound facts.
6. Use an EXACT continuous quotation from the transcription
   as source_evidence for every fact.
7. Do not summarize or paraphrase evidence quotations.
8. Do not use the generated clinical notes.
9. Return JSON with a "facts" array.
10. Each fact must have section, fact, source_evidence.

Transcription:
{transcription}
"""

extraction_summary = []

for case_id in remaining_case_ids:

    output_path = (
        research_dir / f"{case_id}_candidate_facts_v2.json"
    )

    if output_path.exists():
        facts = json.loads(
            output_path.read_text(encoding="utf-8")
        )
        print(case_id, ": existing facts loaded")

    else:
        notes = pd.read_csv(
            research_dir / f"{case_id}_generated_notes.csv"
        )

        transcription = notes["transcription"].iloc[0]

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": EXTRACTION_PROMPT_V2.format(
                    transcription=transcription
                )
            }],
            temperature=0,
            response_format={"type": "json_object"}
        )

        extracted = json.loads(
            response.choices[0].message.content
        )

        facts = extracted["facts"]

        # Save each case immediately
        output_path.write_text(
            json.dumps(facts, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

        print(case_id, ": extracted and saved")

    # Validate evidence quotations
    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )
    transcription = notes["transcription"].iloc[0]

    exact_matches = sum(
        isinstance(fact.get("source_evidence"), str)
        and bool(fact["source_evidence"].strip())
        and fact["source_evidence"] in transcription
        for fact in facts
    )

    extraction_summary.append({
        "case_id": case_id,
        "candidate_facts": len(facts),
        "exact_evidence_matches": exact_matches,
        "unmatched_evidence": len(facts) - exact_matches
    })

extraction_summary_df = pd.DataFrame(extraction_summary)

print("\nEXTRACTION SUMMARY")
display(extraction_summary_df)

print("\nCandidate extraction completed.")

In [ ]:
import json
import pandas as pd
import difflib

unmatched_rows = []

for case_id in remaining_case_ids:

    facts = json.loads(
        (
            research_dir / f"{case_id}_candidate_facts_v2.json"
        ).read_text(encoding="utf-8")
    )

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )

    source = notes["transcription"].iloc[0]

    # Split into source passages for approximate location only
    passages = [
        source[i:i + 350]
        for i in range(0, len(source), 350)
    ]

    for index, fact in enumerate(facts, start=1):

        evidence = str(fact.get("source_evidence", ""))

        if evidence and evidence in source:
            continue

        matches = difflib.get_close_matches(
            evidence,
            passages,
            n=1,
            cutoff=0
        )

        unmatched_rows.append({
            "case_id": case_id,
            "fact_id": f"F{index:03d}",
            "reference_fact": fact.get("fact", ""),
            "unmatched_evidence": evidence,
            "nearest_source_passage": (
                matches[0] if matches else ""
            )
        })

unmatched_evidence_df = pd.DataFrame(unmatched_rows)

pd.set_option("display.max_colwidth", 250)

display(unmatched_evidence_df)

print("\nUnmatched quotations:", len(unmatched_evidence_df))
print("Original files unchanged.")

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

# Distinctive phrases from the original transcriptions
evidence_anchors = {
    ("CASE_1824", "F005"): "No chest pain or shortness of breath",
    ("CASE_1824", "F006"): "Significant for hypertension and hyperlipidemia",
    ("CASE_1824", "F008"): "She had arcus cornealis",
    ("CASE_2148", "F002"): "On exam, he has a lacerated wound",
    ("CASE_2148", "F004"): "Osteomyelitis of the right fifth toe",
    ("CASE_2148", "F008"): "the third metatarsal bone is palpable",
    ("CASE_2176", "F005"): "She did complain of bladder spasms",
    ("CASE_2176", "F006"): "Her labs were within normal limits",
    ("CASE_2176", "F007"): "Her LFTs were slightly elevated"
}

repair_rows = []

for case_id in remaining_case_ids:

    facts = json.loads(
        (
            research_dir / f"{case_id}_candidate_facts_v2.json"
        ).read_text(encoding="utf-8")
    )

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )

    source = notes["transcription"].iloc[0]

    for index, fact in enumerate(facts, start=1):

        fact_id = f"F{index:03d}"
        original_evidence = fact.get("source_evidence", "")

        if (
            isinstance(original_evidence, str)
            and original_evidence
            and original_evidence in source
        ):
            continue

        anchor = evidence_anchors.get((case_id, fact_id))

        if anchor is None:
            continue

        match = re.search(
            re.escape(anchor),
            source,
            flags=re.IGNORECASE
        )

        if match:
            start = max(0, match.start() - 80)
            end = min(len(source), match.end() + 260)

            exact_passage = source[start:end]
            verified = exact_passage in source

        else:
            exact_passage = ""
            verified = False

        repair_rows.append({
            "case_id": case_id,
            "fact_id": fact_id,
            "anchor_found": match is not None,
            "exact_source_match": verified,
            "proposed_evidence": exact_passage
        })

evidence_repair_df = pd.DataFrame(repair_rows)

pd.set_option("display.max_colwidth", 180)

display(evidence_repair_df)

print("\nEvidence passages recovered:",
      int(evidence_repair_df["exact_source_match"].sum()),
      "/",
      len(evidence_repair_df))

print("Original candidate files unchanged.")

In [ ]:
import json
import re
import pandas as pd

review_rows = []

for case_id in remaining_case_ids:
    facts = json.loads(
        (
            research_dir / f"{case_id}_candidate_facts_v2.json"
        ).read_text(encoding="utf-8")
    )

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )
    source = notes["transcription"].iloc[0]

    for index, fact in enumerate(facts, start=1):
        fact_id = f"F{index:03d}"
        statement = str(fact.get("fact", ""))
        original_evidence = str(
            fact.get("source_evidence", "")
        )

        exact_match = (
            bool(original_evidence.strip())
            and original_evidence in source
        )

        repair = evidence_repair_df.loc[
            (evidence_repair_df["case_id"] == case_id)
            & (evidence_repair_df["fact_id"] == fact_id)
        ]

        proposed_evidence = (
            original_evidence
            if exact_match
            else (
                repair["proposed_evidence"].iloc[0]
                if not repair.empty
                else ""
            )
        )

        # Screening heuristic only, not a clinical judgment.
        compound_flag = (
            len(statement.split()) >= 25
            or len(re.findall(
                r"\b(?:and|but|also)\b|;",
                statement,
                flags=re.IGNORECASE
            )) >= 2
        )

        review_rows.append({
            "case_id": case_id,
            "fact_id": fact_id,
            "section": fact.get("section", ""),
            "reference_fact": statement,
            "original_evidence_exact": exact_match,
            "proposed_source_evidence": proposed_evidence,
            "compound_fact_screen": compound_flag,
            "source_transcription": source,
            "clinical_support_verified": False,
            "completeness_reviewed": False,
            "review_decision": ""
        })

reference_review_df = pd.DataFrame(review_rows)

output_path = (
    research_dir / "four_case_reference_quality_review.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing quality-review file preserved."
    )

reference_review_df.to_csv(output_path, index=False)

summary = reference_review_df.groupby("case_id").agg(
    facts=("fact_id", "count"),
    compound_flags=("compound_fact_screen", "sum"),
    original_exact_quotes=("original_evidence_exact", "sum")
).reset_index()

display(summary)

print("\nTotal candidate facts:", len(reference_review_df))
print("Potential compound facts:",
      int(reference_review_df["compound_fact_screen"].sum()))
print("Clinical support verified: 0")
print("Completeness reviews completed: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

compound_review_df = reference_review_df.loc[
    reference_review_df["compound_fact_screen"],
    [
        "case_id",
        "fact_id",
        "section",
        "reference_fact",
        "proposed_source_evidence"
    ]
].copy()

pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_rows", 30)

for case_id, group in compound_review_df.groupby("case_id"):
    print(f"\n{'=' * 65}")
    print(f"{case_id}: {len(group)} flagged facts")
    print("=" * 65)

    for _, row in group.iterrows():
        print(f"\n{row['fact_id']} | {row['section']}")
        print("FACT:", row["reference_fact"])
        print("SOURCE:", row["proposed_source_evidence"])

print("\nTotal flagged facts:", len(compound_review_df))
print("Original checklists unchanged.")

In [ ]:
import pandas as pd

split_proposals = {
    ("CASE_1824", "F008"): [
        "Blood pressure was 140/70 in the left arm while seated.",
        "Arcus cornealis was present.",
        "Pupils were equal, sclerae nonicteric, and conjunctivae pink.",
        "Chest was clear to auscultation.",
        "Cardiac examination documented a regular rhythm and tachycardia.",
        "Abdomen was soft and nontender.",
        "No appreciable peripheral edema was documented.",
        "Degenerative joint disease changes were present in the fingers."
    ],
    ("CASE_1824", "F010"): [
        "A routine sonogram was planned.",
        "Repeat urinalysis was planned to check for blood.",
        "Complement C3 and C4 tests were ordered.",
        "Cystoscopy with retrograde pyelogram was recommended if red cells persisted in urine."
    ],
    ("CASE_2043", "F004"): [
        "Toprol was listed as a current medication.",
        "Avalide was listed as a current medication."
    ],
    ("CASE_2043", "F016"): [
        "External genitalia had no lesions.",
        "BUS examination was normal.",
        "Vulva and vagina showed mild atrophy without lesions.",
        "Cervix and uterus were within normal limits.",
        "Ovaries were not readily palpable.",
        "No pelvic masses were appreciated."
    ],
    ("CASE_2148", "F002"): [
        "A lacerated wound was present on the plantar aspect of the right foot.",
        "The wound extended from the second to fifth metatarsal areas.",
        "The wound was deep and described as stage IV."
    ],
    ("CASE_2148", "F010"): [
        "Discontinue oral Levaquin, Zyvox, and intravenous Zosyn.",
        "Start intravenous Ancef 2 g every eight hours.",
        "Continue intravenous antibiotic therapy for six weeks.",
        "Continue wound care and initiate wound VAC placement."
    ],
    ("CASE_2176", "F004"): [
        "The patient underwent the documented procedure without difficulty.",
        "Pain was controlled with oral medication.",
        "The patient was afebrile.",
        "The patient tolerated a liquid diet."
    ],
    ("CASE_2176", "F005"): [
        "The patient reported bladder spasms during urination.",
        "The patient reported a history of chronic urinary tract infections.",
        "Urinalysis and urine culture were planned before discharge.",
        "A three-day presumptive ciprofloxacin prescription was provided."
    ],
    ("CASE_2176", "F006"): [
        "Omeprazole and Colace were included in discharge medications.",
        "Percocet was prescribed for pain.",
        "White blood cell count was elevated.",
        "The elevated white blood cell count was considered a possible leukemoid reaction."
    ],
    ("CASE_2176", "F008"): [
        "Clear liquids were recommended for 48 hours before returning to Medifast.",
        "The wound should be kept clean and dry.",
        "Lifting more than 10 pounds was prohibited for two weeks.",
        "Driving was restricted for one to two weeks.",
        "Driving required discontinuation of narcotic medication and ability to stop in an emergency.",
        "Strenuous activity was prohibited, but mobility was encouraged.",
        "Usual medications could be resumed according to medication reconciliation."
    ],
    ("CASE_2176", "F009"): [
        "Follow up on urinalysis and urine culture results.",
        "Follow up with Dr. X in two weeks.",
        "Call for shortness of breath, fever above 101.5, or chest pain.",
        "Call for intractable nausea, vomiting, or abdominal pain.",
        "Call for wound redness, swelling, or foul-smelling drainage."
    ]
}

proposal_rows = []

for (case_id, fact_id), statements in split_proposals.items():
    for i, statement in enumerate(statements, start=1):
        proposal_rows.append({
            "case_id": case_id,
            "original_fact_id": fact_id,
            "proposed_component_id": f"{fact_id}_C{i:02d}",
            "proposed_fact": statement,
            "review_status": "proposed_requires_source_verification"
        })

fact_split_proposals_df = pd.DataFrame(proposal_rows)

output_path = (
    research_dir / "four_case_fact_split_proposals.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing split proposal file preserved."
    )

fact_split_proposals_df.to_csv(output_path, index=False)

display(
    fact_split_proposals_df.groupby("case_id").agg(
        original_facts=("original_fact_id", "nunique"),
        proposed_components=("proposed_component_id", "count")
    ).reset_index()
)

print("\nTotal proposed components:",
      len(fact_split_proposals_df))
print("Original candidate facts changed: 0")
print("Saved:", output_path)

In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher

verification_rows = []

for _, row in fact_split_proposals_df.iterrows():
    case_id = row["case_id"]

    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    statement = row["proposed_fact"]

    # Search overlapping source windows for the closest passage.
    # Similarity is a review aid, not clinical verification.
    window_size = max(160, len(statement) * 2)
    step_size = 60

    windows = [
        source[i:i + window_size]
        for i in range(0, len(source), step_size)
    ]

    if not windows:
        windows = [source]

    best_passage = max(
        windows,
        key=lambda passage: SequenceMatcher(
            None,
            statement.lower(),
            passage.lower()
        ).ratio()
    )

    similarity = SequenceMatcher(
        None,
        statement.lower(),
        best_passage.lower()
    ).ratio()

    verification_rows.append({
        "case_id": case_id,
        "original_fact_id": row["original_fact_id"],
        "component_id": row["proposed_component_id"],
        "proposed_fact": statement,
        "candidate_source_passage": best_passage,
        "lexical_similarity": round(similarity, 3),
        "source_passage_exact": best_passage in source,
        "clinical_support_verified": False,
        "review_decision": ""
    })

component_verification_df = pd.DataFrame(verification_rows)

assert len(component_verification_df) == 51
assert component_verification_df["source_passage_exact"].all()

output_path = (
    research_dir / "four_case_component_verification.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing component verification file preserved."
    )

component_verification_df.to_csv(output_path, index=False)

display(
    component_verification_df.groupby("case_id").agg(
        components=("component_id", "count"),
        mean_lexical_similarity=("lexical_similarity", "mean"),
        clinical_support_verified=("clinical_support_verified", "sum")
    ).reset_index().round(3)
)

print("\nComponents screened:", len(component_verification_df))
print("Exact source passages:", int(
    component_verification_df["source_passage_exact"].sum()
))
print("Clinically verified components: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd
import re

review = component_verification_df.copy()

clinical_detail_pattern = (
    r"\b(?:no|not|without|negative|right|left|"
    r"intravenous|oral|mg|dose|every|weeks?|"
    r"tachycardia|osteomyelitis|fever|"
    r"culture|urinalysis|antibiotic)\b"
    r"|\d+"
)

review["clinical_detail_flag"] = (
    review["proposed_fact"].str.contains(
        clinical_detail_pattern,
        case=False,
        regex=True,
        na=False
    )
)

review["review_priority"] = review.apply(
    lambda row: (
        "HIGH"
        if row["clinical_detail_flag"]
        or row["lexical_similarity"] < 0.30
        else "STANDARD"
    ),
    axis=1
)

review["review_decision"] = ""
review["corrected_fact"] = ""
review["verified_source_quote"] = ""
review["reviewer_notes"] = ""

review = review.sort_values(
    ["review_priority", "case_id", "component_id"],
    ascending=[True, True, True]
)

output_path = (
    research_dir / "four_case_clinical_support_review.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing review worksheet preserved."
    )

review.to_csv(output_path, index=False)

display(
    pd.crosstab(
        review["case_id"],
        review["review_priority"]
    )
)

print("\nComponents requiring review:", len(review))
print("Clinical-support decisions completed:", 0)
print("Saved:", output_path)
print("Original reference facts unchanged.")

In [ ]:
import pandas as pd

high_priority_review_df = review.loc[
    review["review_priority"] == "HIGH"
].sort_values(["case_id", "component_id"])

for case_id, group in high_priority_review_df.groupby("case_id"):

    print(f"\n{'=' * 75}")
    print(f"{case_id}: {len(group)} high-priority components")
    print("=" * 75)

    for _, row in group.iterrows():

        print(f"\n[{row['component_id']}]")

        print("PROPOSED FACT:")
        print(row["proposed_fact"])

        print("\nCANDIDATE SOURCE PASSAGE:")
        print(row["candidate_source_passage"])

        print("\n" + "-" * 75)

print("\nHigh-priority components displayed:",
      len(high_priority_review_df))

print("Clinical verification decisions recorded: 0")

In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher

def source_sentences(text):
    # Preserve source wording and punctuation.
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text)
        if s.strip()
    ]

def clinical_tokens(text):
    stop = {
        "the", "was", "were", "with", "and", "for",
        "from", "that", "this", "had", "has",
        "patient", "documented", "recommended",
        "planned", "present", "would"
    }

    return {
        token
        for token in re.findall(r"[a-z0-9]+", text.lower())
        if len(token) > 2 and token not in stop
    }

retrieved_rows = []

for _, row in fact_split_proposals_df.iterrows():

    case_id = row["case_id"]
    statement = row["proposed_fact"]

    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    sentences = source_sentences(source)

    # Include individual sentences and adjacent pairs.
    candidates = sentences + [
        sentences[i] + " " + sentences[i + 1]
        for i in range(len(sentences) - 1)
    ]

    query_tokens = clinical_tokens(statement)

    def score(candidate):
        candidate_tokens = clinical_tokens(candidate)

        overlap = (
            len(query_tokens & candidate_tokens)
            / max(1, len(query_tokens))
        )

        lexical = SequenceMatcher(
            None,
            statement.lower(),
            candidate.lower()
        ).ratio()

        return 0.75 * overlap + 0.25 * lexical

    best = max(candidates, key=score)

    retrieved_rows.append({
        "case_id": case_id,
        "component_id": row["proposed_component_id"],
        "proposed_fact": statement,
        "candidate_source_quote": best,
        "retrieval_score": round(score(best), 3),
        "exact_source_substring": best in source,
        "clinical_support_verified": False
    })

sentence_evidence_df = pd.DataFrame(retrieved_rows)

output_path = (
    research_dir / "four_case_sentence_evidence_candidates.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing sentence evidence file preserved."
    )

sentence_evidence_df.to_csv(output_path, index=False)

display(
    sentence_evidence_df.groupby("case_id").agg(
        components=("component_id", "count"),
        mean_retrieval_score=("retrieval_score", "mean"),
        exact_quotes=("exact_source_substring", "sum")
    ).reset_index().round(3)
)

print("\nComponents processed:", len(sentence_evidence_df))
print("Clinically verified:", 0)
print("Saved:", output_path)

In [ ]:
priority_components = {
    "CASE_1824": [
        "F008_C01", "F008_C02", "F008_C05",
        "F010_C03", "F010_C04"
    ],
    "CASE_2043": [
        "F016_C03", "F016_C05", "F016_C06"
    ],
    "CASE_2148": [
        "F002_C01", "F002_C03",
        "F010_C01", "F010_C02",
        "F010_C03", "F010_C04"
    ],
    "CASE_2176": [
        "F004_C03", "F005_C03",
        "F006_C02", "F006_C03",
        "F008_C03", "F008_C04",
        "F009_C03"
    ]
}

selected_ids = {
    (case_id, component_id)
    for case_id, components in priority_components.items()
    for component_id in components
}

selected_evidence_df = sentence_evidence_df.loc[
    sentence_evidence_df.apply(
        lambda row: (
            row["case_id"],
            row["component_id"]
        ) in selected_ids,
        axis=1
    )
].sort_values(["case_id", "component_id"])

assert len(selected_evidence_df) == len(selected_ids)

for case_id, group in selected_evidence_df.groupby("case_id"):

    print(f"\n{'=' * 70}")
    print(case_id)
    print("=" * 70)

    for _, row in group.iterrows():
        print(f"\n[{row['component_id']}]")
        print("FACT:", row["proposed_fact"])
        print("SOURCE:", row["candidate_source_quote"])
        print("RETRIEVAL SCORE:", row["retrieval_score"])
        print("-" * 70)

print("\nEvidence pairs displayed:", len(selected_evidence_df))
print("Clinical-support labels assigned: 0")

In [ ]:
import pandas as pd

focused_review_df = selected_evidence_df.copy()

# Correct the current-versus-historical treatment evidence.
case_id = "CASE_2148"
component_id = "F010_C03"

source = reference_review_df.loc[
    reference_review_df["case_id"] == case_id,
    "source_transcription"
].iloc[0]

correct_quote = (
    "We will need to continue this treatment for 6 weeks "
    "for treatment of osteomyelitis and deep wound infection."
)

assert correct_quote in source, (
    "Corrected quotation not found in original transcription."
)

target = (
    (focused_review_df["case_id"] == case_id)
    & (focused_review_df["component_id"] == component_id)
)

assert target.sum() == 1

focused_review_df.loc[
    target, "candidate_source_quote"
] = correct_quote

# These are provisional source-support assessments,
# not independent clinical adjudications.
focused_review_df["provisional_support"] = "supported"
focused_review_df["review_origin"] = (
    "source_quotation_inspection"
)
focused_review_df["requires_independent_review"] = True

focused_review_df.loc[
    target, "review_note"
] = (
    "Replaced historical antibiotic-course evidence "
    "with the current treatment-plan quotation."
)

focused_review_df["review_note"] = (
    focused_review_df["review_note"].fillna("")
)

assert focused_review_df[
    "candidate_source_quote"
].notna().all()

# Check every quotation against its corresponding source.
for _, row in focused_review_df.iterrows():
    original_source = reference_review_df.loc[
        reference_review_df["case_id"] == row["case_id"],
        "source_transcription"
    ].iloc[0]

    assert row["candidate_source_quote"] in original_source

output_path = (
    research_dir
    / "four_case_focused_evidence_review_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional review file preserved."
    )

focused_review_df.to_csv(output_path, index=False)

print("Evidence pairs reviewed:", len(focused_review_df))
print("Incorrect treatment-episode matches corrected:", 1)
print("Provisional source-support assessments:",
      focused_review_df["provisional_support"].notna().sum())
print("Independent clinical adjudications:", 0)
print("Saved:", output_path)

In [ ]:
# Identify the 30 components not yet inspected
reviewed_keys = set(
    zip(
        focused_review_df["case_id"],
        focused_review_df["component_id"]
    )
)

remaining_component_df = sentence_evidence_df.loc[
    ~sentence_evidence_df.apply(
        lambda row: (
            row["case_id"],
            row["component_id"]
        ) in reviewed_keys,
        axis=1
    )
].sort_values(["case_id", "component_id"]).copy()

assert len(remaining_component_df) == 30

for case_id, group in remaining_component_df.groupby("case_id"):

    print(f"\n{'=' * 65}")
    print(f"{case_id}: {len(group)} remaining components")
    print("=" * 65)

    for _, row in group.iterrows():
        print(f"\n[{row['component_id']}]")
        print("FACT:", row["proposed_fact"])
        print("SOURCE:", row["candidate_source_quote"])
        print("-" * 65)

print("\nRemaining components displayed:",
      len(remaining_component_df))
print("Additional clinical-support decisions recorded: 0")

In [ ]:
import pandas as pd

# Work with copies; preserve earlier review files.
additional = remaining_component_df.copy()
additional["provisional_support"] = "supported"
additional["review_origin"] = "source_quotation_inspection"
additional["requires_independent_review"] = True
additional["review_note"] = ""

corrections = {
    ("CASE_1824", "F008_C03"): (
        "The pupils were equal. The sclerae were not icteric. "
        "The conjunctivae were pink."
    ),
    ("CASE_2176", "F009_C01"): (
        "We will follow up on both urinalysis and cultures."
    )
}

for (case_id, component_id), quote in corrections.items():

    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    assert quote in source, (
        f"Exact quotation not found: {case_id}, {component_id}"
    )

    target = (
        (additional["case_id"] == case_id)
        & (additional["component_id"] == component_id)
    )

    assert target.sum() == 1

    additional.loc[target, "candidate_source_quote"] = quote
    additional.loc[target, "review_note"] = (
        "Retrieval quotation corrected after source inspection."
    )

# Combine all inspected components.
columns = [
    "case_id",
    "component_id",
    "proposed_fact",
    "candidate_source_quote",
    "provisional_support",
    "review_origin",
    "requires_independent_review",
    "review_note"
]

complete_component_review_df = pd.concat(
    [
        focused_review_df[columns],
        additional[columns]
    ],
    ignore_index=True
).sort_values(["case_id", "component_id"])

assert len(complete_component_review_df) == 51
assert not complete_component_review_df.duplicated(
    ["case_id", "component_id"]
).any()

# Confirm every quotation is an exact source substring.
for _, row in complete_component_review_df.iterrows():

    source = reference_review_df.loc[
        reference_review_df["case_id"] == row["case_id"],
        "source_transcription"
    ].iloc[0]

    assert row["candidate_source_quote"] in source

output_path = (
    research_dir
    / "four_case_complete_component_review_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing consolidated review preserved."
    )

complete_component_review_df.to_csv(
    output_path,
    index=False
)

display(
    complete_component_review_df.groupby("case_id").agg(
        components=("component_id", "count"),
        provisional_supports=("provisional_support", "count")
    ).reset_index()
)

print("\nComponents consolidated:",
      len(complete_component_review_df))
print("Evidence corrections in this step:", 2)
print("Independent clinical adjudications:", 0)
print("Saved:", output_path)

In [ ]:
import pandas as pd

# Identify the original facts already replaced by proposed components.
split_keys = set(
    zip(
        fact_split_proposals_df["case_id"],
        fact_split_proposals_df["original_fact_id"]
    )
)

unsplit_facts_df = reference_review_df.loc[
    ~reference_review_df.apply(
        lambda row: (
            row["case_id"],
            row["fact_id"]
        ) in split_keys,
        axis=1
    )
].copy()

assert len(unsplit_facts_df) == 45

for case_id, group in unsplit_facts_df.groupby("case_id"):
    print(f"\n{'=' * 70}")
    print(f"{case_id}: {len(group)} unsplit facts")
    print("=" * 70)

    for _, row in group.iterrows():
        print(f"\n[{row['fact_id']}]")
        print("FACT:", row["reference_fact"])
        print("SOURCE:", row["proposed_source_evidence"])
        print(
            "ORIGINAL QUOTE EXACT:",
            row["original_evidence_exact"]
        )
        print("-" * 70)

print("\nUnsplit facts displayed:", len(unsplit_facts_df))
print("Clinical verification labels assigned: 0")
print("Original files unchanged.")

In [ ]:
import pandas as pd

quality_issues = {
    ("CASE_1824", "F005"): (
        "split_required",
        "Separate negative review-of-systems findings."
    ),
    ("CASE_1824", "F006"): (
        "split_required",
        "Separate medical conditions and historical procedures."
    ),
    ("CASE_1824", "F007"): (
        "split_required",
        "Separate medications; retain dose and frequency."
    ),
    ("CASE_1824", "F009"): (
        "split_required",
        "Separate urine microscopy, protein findings, "
        "and serum creatinine."
    ),
    ("CASE_2043", "F007"): (
        "exclude_provisionally",
        "No substantive vital-sign value is provided."
    ),
    ("CASE_2043", "F009"): (
        "granularity_review",
        "Multiple HEENT examination findings."
    ),
    ("CASE_2043", "F010"): (
        "granularity_review",
        "Multiple neck examination findings."
    ),
    ("CASE_2043", "F012"): (
        "granularity_review",
        "Multiple negative breast examination findings."
    ),
    ("CASE_2043", "F019"): (
        "granularity_review",
        "Multiple neurological examination findings."
    ),
    ("CASE_2148", "F003"): (
        "wording_review",
        "Retain that imaging was reported negative; "
        "do not imply osteomyelitis was ruled out."
    ),
    ("CASE_2148", "F004"): (
        "split_required",
        "Separate diabetes from historical fifth-toe osteomyelitis."
    ),
    ("CASE_2148", "F008"): (
        "wording_review",
        "Preserve that osteomyelitis was the clinician's "
        "clinical assessment."
    ),
    ("CASE_2176", "F002"): (
        "duplicate_review",
        "Same diagnosis as F001; consider one clinical fact "
        "with admission and discharge provenance."
    ),
    ("CASE_2176", "F003"): (
        "overlap_review",
        "Symptomatic cholelithiasis overlaps with F001; "
        "age and afebrile status remain distinct."
    ),
    ("CASE_2176", "F007"): (
        "split_required",
        "Separate AST/ALT values, possible explanation, "
        "and discharge readiness."
    )
}

quality_rows = []

for _, row in unsplit_facts_df.iterrows():
    key = (row["case_id"], row["fact_id"])

    decision, note = quality_issues.get(
        key,
        ("retain_provisionally", "No specific issue identified.")
    )

    quality_rows.append({
        "case_id": row["case_id"],
        "fact_id": row["fact_id"],
        "reference_fact": row["reference_fact"],
        "quality_decision": decision,
        "review_note": note,
        "clinical_support_verified": False,
        "requires_independent_review": True
    })

unsplit_quality_df = pd.DataFrame(quality_rows)

assert len(unsplit_quality_df) == 45

output_path = (
    research_dir / "four_case_unsplit_quality_decisions.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing quality decisions preserved."
    )

unsplit_quality_df.to_csv(output_path, index=False)

display(
    pd.crosstab(
        unsplit_quality_df["case_id"],
        unsplit_quality_df["quality_decision"]
    )
)

print("\nFacts reviewed:", len(unsplit_quality_df))
print(
    "Facts with identified issues:",
    (unsplit_quality_df["quality_decision"]
     != "retain_provisionally").sum()
)
print("Independent clinical verifications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

# Retain the 51 decomposed components.
components = complete_component_review_df[
    [
        "case_id",
        "component_id",
        "proposed_fact",
        "candidate_source_quote"
    ]
].copy()

components = components.rename(columns={
    "component_id": "reference_id",
    "proposed_fact": "reference_fact",
    "candidate_source_quote": "source_evidence"
})

components["reference_origin"] = "decomposed_component"

# Retain the 30 original facts without identified quality issues.
retained = unsplit_quality_df.loc[
    unsplit_quality_df["quality_decision"] == "retain_provisionally"
].merge(
    unsplit_facts_df[
        ["case_id", "fact_id", "proposed_source_evidence"]
    ],
    on=["case_id", "fact_id"],
    validate="one_to_one"
)

retained = retained.rename(columns={
    "fact_id": "reference_id",
    "proposed_source_evidence": "source_evidence"
})

retained["reference_origin"] = "original_candidate"

working_reference_df = pd.concat(
    [
        components[
            [
                "case_id", "reference_id",
                "reference_fact", "source_evidence",
                "reference_origin"
            ]
        ],
        retained[
            [
                "case_id", "reference_id",
                "reference_fact", "source_evidence",
                "reference_origin"
            ]
        ]
    ],
    ignore_index=True
)

working_reference_df["reference_status"] = (
    "provisional_not_independently_adjudicated"
)

resolution_queue_df = unsplit_quality_df.loc[
    unsplit_quality_df["quality_decision"]
    != "retain_provisionally"
].copy()

assert len(working_reference_df) == 81
assert len(resolution_queue_df) == 15

assert not working_reference_df.duplicated(
    ["case_id", "reference_id"]
).any()

# Validate source quotations against the original transcription.
for case_id, group in working_reference_df.groupby("case_id"):
    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    assert all(
        isinstance(quote, str)
        and bool(quote.strip())
        and quote in source
        for quote in group["source_evidence"]
    ), f"Invalid source quotation in {case_id}"

working_path = (
    research_dir / "four_case_working_reference_v1.csv"
)
queue_path = (
    research_dir / "four_case_reference_resolution_queue.csv"
)

if working_path.exists() or queue_path.exists():
    raise FileExistsError(
        "Existing working reference files preserved."
    )

working_reference_df.to_csv(working_path, index=False)
resolution_queue_df.to_csv(queue_path, index=False)

display(
    working_reference_df.groupby("case_id").agg(
        working_facts=("reference_id", "count")
    ).reset_index()
)

print("\nWorking reference facts:", len(working_reference_df))
print("Unresolved original facts:", len(resolution_queue_df))
print("Independent clinical adjudications: 0")
print("Saved:", working_path)
print("Saved:", queue_path)

In [ ]:
import pandas as pd

resolution_queue_df = pd.read_csv(
    research_dir / "four_case_reference_resolution_queue.csv"
)

pd.set_option("display.max_colwidth", 300)

display(
    resolution_queue_df[
        [
            "case_id",
            "fact_id",
            "reference_fact",
            "quality_decision",
            "review_note"
        ]
    ].sort_values(["case_id", "fact_id"])
)

print("\nUnresolved facts:", len(resolution_queue_df))
print("Original working reference checklist unchanged.")

In [ ]:
import pandas as pd

# Each tuple: (case, original fact, revised statement, source anchor)
revisions = [
    ("CASE_1824", "F005", "No chest pain or shortness of breath.", "No chest pain or shortness of breath"),
    ("CASE_1824", "F005", "No abdominal pain, nausea, or vomiting.", "She has no abdominal pain or nausea or vomiting."),
    ("CASE_1824", "F005", "No anemia was reported.", "She has no anemia."),
    ("CASE_1824", "F005", "No swelling was reported.", "She has noticed no swelling."),
    ("CASE_1824", "F005", "No history of seizures.", "She has no history of seizures."),

    ("CASE_1824", "F006", "History of hypertension and hyperlipidemia.", "Significant for hypertension and hyperlipidemia."),
    ("CASE_1824", "F006", "History of bilateral simple mastectomies approximately 35 years earlier.", "She has had bilateral simple mastectomies done 35 years ago."),
    ("CASE_1824", "F006", "History of partial lung resection for carcinoma.", "She has also had one-third of her lung removed for carcinoma"),
    ("CASE_1824", "F006", "History of hysterectomy.", "She also had hysterectomy in the past."),

    ("CASE_1824", "F007", "Dyazide once daily.", "Dyazide one a day."),
    ("CASE_1824", "F007", "Pravachol 80 mg daily in the evening.", "Pravachol 80 mg a day in the evening."),
    ("CASE_1824", "F007", "Vitamin E once daily.", "Vitamin E once a day."),
    ("CASE_1824", "F007", "One baby aspirin daily.", "One baby aspirin per day."),

    ("CASE_1824", "F009", "Urine microscopy showed 1–2 red blood cells per high-power field.", "1-2 red cells per high power fields."),
    ("CASE_1824", "F009", "No urine protein was documented.", "She had no protein."),
    ("CASE_1824", "F009", "Many squamous cells were seen in urine.", "She did have many squamous cells."),
    ("CASE_1824", "F009", "Creatinine was 1 mg percent.", "creatinine of 1 mg percent"),

    ("CASE_2043", "F009", "Pupils were equal, round, and reactive to light and accommodation.", "PERRLA."),
    ("CASE_2043", "F009", "Extraocular movements were intact.", "EOMI."),
    ("CASE_2043", "F009", "Tympanic membranes were clear bilaterally.", "TMs clear bilaterally."),
    ("CASE_2043", "F009", "Nose and throat were clear.", "Nose and throat clear."),
    ("CASE_2043", "F010", "Neck was supple without adenopathy or thyromegaly.", "Supple without adenopathy or thyromegaly."),
    ("CASE_2043", "F010", "Carotid pulses were palpably normal without bruit.", "Carotid pulses palpably normal without bruit."),
    ("CASE_2043", "F012", "No breast asymmetry or skin changes.", "No asymmetry, skin changes"),
    ("CASE_2043", "F012", "No dominant breast masses.", "dominant masses"),
    ("CASE_2043", "F012", "No nipple discharge.", "nipple discharge"),
    ("CASE_2043", "F012", "No axillary adenopathy.", "axillary adenopathy"),
    ("CASE_2043", "F019", "Cranial nerves II–XII were intact.", "Cranial nerves II-XII intact."),
    ("CASE_2043", "F019", "Strength was within normal limits.", "Strength, sensation, coordination, and reflexes all within normal limits."),
    ("CASE_2043", "F019", "Sensation was within normal limits.", "Strength, sensation, coordination, and reflexes all within normal limits."),
    ("CASE_2043", "F019", "Coordination was within normal limits.", "Strength, sensation, coordination, and reflexes all within normal limits."),
    ("CASE_2043", "F019", "Reflexes were within normal limits.", "Strength, sensation, coordination, and reflexes all within normal limits."),

    ("CASE_2148", "F003", "MRI and bone scan were reportedly negative for osteomyelitis.", "reported negative for osteomyelitis."),
    ("CASE_2148", "F004", "History of diabetes mellitus.", "Diabetes mellitus."),
    ("CASE_2148", "F004", "Right fifth-toe osteomyelitis had been treated with six weeks of intravenous antibiotics approximately five years earlier.", "Osteomyelitis of the right fifth toe, which was treated with intravenous antibiotic therapy for 6 weeks about 5 years back."),
    ("CASE_2148", "F008", "The clinician suspected osteomyelitis after palpating the third metatarsal bone.", "the third metatarsal bone is palpable, which leads to the clinical diagnosis of osteomyelitis."),

    ("CASE_2176", "F003", "The patient was 27 years old.", "27-year-old woman"),
    ("CASE_2176", "F003", "The patient was afebrile on presentation.", "She was afebrile."),
    ("CASE_2176", "F007", "AST was 51 and ALT was 83.", "They were 51 and 83 for the AST and ALT respectively."),
    ("CASE_2176", "F007", "Liver enzyme elevation was attributed by the clinician to cautery on the liver bed.", "I do believe this is related to the cautery used on the liver bed."),
    ("CASE_2176", "F007", "The clinician considered the patient ready for discharge.", "I feel that she looks good for discharge.")
]

new_rows = []

for case_id, fact_id, statement, anchor in revisions:
    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    assert anchor in source, (
        f"Source anchor not found: {case_id} {fact_id}: {anchor}"
    )

    existing_count = sum(
        (r["case_id"] == case_id and r["original_fact_id"] == fact_id)
        for r in new_rows
    )

    new_rows.append({
        "case_id": case_id,
        "original_fact_id": fact_id,
        "reference_id": f"{fact_id}_R{existing_count + 1:02d}",
        "reference_fact": statement,
        "source_evidence": anchor,
        "reference_origin": "resolution_proposal",
        "reference_status": "provisional_not_independently_adjudicated"
    })

resolved_proposals_df = pd.DataFrame(new_rows)

# The duplicate admission/discharge diagnosis is excluded
# from independent scoring, preserving F001 in the checklist.
excluded_df = resolution_queue_df.loc[
    (
        (resolution_queue_df["case_id"] == "CASE_2043")
        & (resolution_queue_df["fact_id"] == "F007")
    )
    | (
        (resolution_queue_df["case_id"] == "CASE_2176")
        & (resolution_queue_df["fact_id"] == "F002")
    )
].copy()

revised_reference_df = pd.concat(
    [
        working_reference_df,
        resolved_proposals_df[working_reference_df.columns]
    ],
    ignore_index=True
)

assert not revised_reference_df.duplicated(
    ["case_id", "reference_id"]
).any()

output_path = research_dir / "four_case_working_reference_v2.csv"

if output_path.exists():
    raise FileExistsError("Existing revised checklist preserved.")

revised_reference_df.to_csv(output_path, index=False)

display(
    revised_reference_df.groupby("case_id").agg(
        reference_facts=("reference_id", "count")
    ).reset_index()
)

print("\nNew proposed facts:", len(resolved_proposals_df))
print("Excluded duplicate/noninformative entries:", len(excluded_df))
print("Total provisional reference facts:", len(revised_reference_df))
print("Independent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

case_id = "CASE_2148"

notes = pd.read_csv(
    research_dir / f"{case_id}_generated_notes.csv"
)

source = notes["transcription"].iloc[0]

facts = revised_reference_df.loc[
    revised_reference_df["case_id"] == case_id,
    ["reference_id", "reference_fact"]
].sort_values("reference_id")

print("=" * 75)
print("ORIGINAL CLINICAL TRANSCRIPTION")
print("=" * 75)
print(source)

print("\n" + "=" * 75)
print("CURRENT PROVISIONAL REFERENCE FACTS")
print("=" * 75)

for _, row in facts.iterrows():
    print(f"{row['reference_id']}: {row['reference_fact']}")

print("\nSource words:", len(source.split()))
print("Reference facts:", len(facts))
print("Completeness independently verified: No")

In [ ]:
import pandas as pd

case_id = "CASE_2148"

source = pd.read_csv(
    research_dir / f"{case_id}_generated_notes.csv"
)["transcription"].iloc[0]

additions = [
    ("Patient was a 68-year-old male.",
     "The patient is a 68-year-old Caucasian male"),

    ("Wound cultures dated 06/25/2008 were reported positive for MSSA.",
     "Cultures taken from the wound on 06/25/2008, were reported positive for methicillin-sensitive Staphylococcus aureus (MSSA)."),

    ("Wound cultures dated 07/25/2008 were reported positive for MSSA.",
     "Wound cultures from 07/25/2008 showed methicillin-sensitive Staphylococcus aureus."),

    ("No laboratory or radiological data were available in the chart at consultation.",
     "No laboratory or radiological data is available at present in the chart."),

    ("PICC placement was planned to complete six weeks of intravenous antibiotics.",
     "We will get a PICC line placed to complete the 6-week course of intravenous antibiotic therapy."),

    ("Monday monitoring included CBC with differential, chemistry 7, LFTs, ESR, and CRP.",
     "CBC with differential, chemistry 7 panel, LFTs, ESR, and C-reactive protein levels every Monday"),

    ("Thursday monitoring included chemistry 7 and CBC.",
     "chemistry 7 panel and CBC every Thursday"),

    ("Wound healing would be monitored two to three times weekly.",
     "I will continue to monitor wound healing 2 to 3 times a week."),

    ("The long-term acute care facility wound care team would manage wound care.",
     "Wound care will be managed by the wound care team at the Long-Term Acute Care Facility."),

    ("Deep wound culture results from 07/29/2008, MRI, and bone scan were to be requested.",
     "We would also try to get the results of the deep wound cultures from 07/29/2008, MRI, and bone scan from Hospital."),

    ("The treatment plan was discussed with the patient and his daughter.",
     "The treatment plan was discussed in detail with the patient and his daughter"),

    ("The clinician recommended treating osteomyelitis on clinical grounds despite reportedly negative imaging.",
     "Even if the MRI and bone scan are negative, the treatment should be guided with diagnosis on clinical counts in my opinion."),

    ("Results of the debridement cultures were unavailable.",
     "Culture results from the debridement procedure are not available.")
]

new_rows = []

for index, (fact, quotation) in enumerate(additions, start=1):
    assert quotation in source, (
        f"Source evidence not found for additional fact {index}"
    )

    new_rows.append({
        "case_id": case_id,
        "reference_id": f"ADD{index:03d}",
        "reference_fact": fact,
        "source_evidence": quotation,
        "reference_origin": "completeness_review",
        "reference_status": "provisional_not_independently_adjudicated"
    })

expanded_reference_df = pd.concat(
    [revised_reference_df, pd.DataFrame(new_rows)],
    ignore_index=True
)

assert not expanded_reference_df.duplicated(
    ["case_id", "reference_id"]
).any()

output_path = (
    research_dir / "four_case_working_reference_v3.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing expanded checklist preserved."
    )

expanded_reference_df.to_csv(output_path, index=False)

display(
    expanded_reference_df.groupby("case_id").agg(
        reference_facts=("reference_id", "count")
    ).reset_index()
)

print("\nNew CASE_2148 facts:", len(new_rows))
print("Total provisional reference facts:", len(expanded_reference_df))
print("Independent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

cases_to_review = [
    "CASE_1824",
    "CASE_2043",
    "CASE_2176"
]

for case_id in cases_to_review:

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )

    source = notes["transcription"].iloc[0]

    facts = expanded_reference_df.loc[
        expanded_reference_df["case_id"] == case_id,
        ["reference_id", "reference_fact"]
    ].sort_values("reference_id")

    print("\n" + "=" * 85)
    print(f"{case_id} | ORIGINAL TRANSCRIPTION")
    print("=" * 85)
    print(source)

    print("\nCURRENT REFERENCE CHECKLIST")
    print("-" * 85)

    for _, row in facts.iterrows():
        print(
            f"{row['reference_id']}: "
            f"{row['reference_fact']}"
        )

    print(
        f"\nSource words: {len(source.split())}"
        f" | Reference facts: {len(facts)}"
    )

print("\nCompleteness inspection displayed for 3 cases.")
print("Files modified: 0")
print("API requests: 0")

In [ ]:
import pandas as pd

additions = [
    ("CASE_1824",
     "The patient reported longstanding decreased hearing.",
     "The patient has had decreased hearing for many years."),

    ("CASE_1824",
     "The patient did not use nonsteroidal anti-inflammatory agents.",
     "She does not use nonsteroidal agents."),

    ("CASE_1824",
     "No history of heart attack or stroke was documented.",
     "There is no history of heart attack or stroke."),

    ("CASE_1824",
     "The patient did not smoke.",
     "She does not smoke."),

    ("CASE_1824",
     "The clinician considered glomerular disease unlikely, but the microscopic hematuria remained unexplained.",
     "It seems unlikely that she has glomerular disease; however, we cannot explain the red cells in the urine."),

    ("CASE_2043",
     "The patient reported substantial stress involving her son.",
     "She has a lot of stress associated with a son that has some challenges."),

    ("CASE_2043",
     "A glucose test was planned.",
     "We will check a glucose, lipid profile and a Hemoccult test also and a mammogram."),

    ("CASE_2043",
     "A Hemoccult test was planned.",
     "We will check a glucose, lipid profile and a Hemoccult test also and a mammogram."),

    ("CASE_2043",
     "The remainder of the review of systems was negative.",
     "Her review of systems is otherwise negative."),

    ("CASE_2176",
     "No consultations were documented during the hospitalization.",
     "CONSULTS:, None."),

    ("CASE_2176",
     "Bilirubin laboratory results were within normal limits.",
     "Her labs in terms of her bilirubin were within normal limits."),

    ("CASE_2176",
     "The patient was permitted to shower after discharge.",
     "she may shower."),

    ("CASE_2176",
     "The clinician considered a urinary tract infection a possible contributor to the elevated white blood cell count.",
     "and if she does have UTI, may also be related.")
]

added_rows = []

for index, (case_id, fact, quote) in enumerate(
    additions, start=1
):
    source = reference_review_df.loc[
        reference_review_df["case_id"] == case_id,
        "source_transcription"
    ].iloc[0]

    assert quote in source, (
        f"Source quotation not found: {case_id}, ADD{index:03d}"
    )

    added_rows.append({
        "case_id": case_id,
        "reference_id": f"COMP{index:03d}",
        "reference_fact": fact,
        "source_evidence": quote,
        "reference_origin": "completeness_review",
        "reference_status":
            "provisional_not_independently_adjudicated"
    })

four_case_reference_v4_df = pd.concat(
    [
        expanded_reference_df,
        pd.DataFrame(added_rows)
    ],
    ignore_index=True
)

assert len(four_case_reference_v4_df) == 148
assert not four_case_reference_v4_df.duplicated(
    ["case_id", "reference_id"]
).any()

output_path = (
    research_dir / "four_case_working_reference_v4.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing checklist preserved; no overwrite performed."
    )

four_case_reference_v4_df.to_csv(
    output_path,
    index=False
)

display(
    four_case_reference_v4_df.groupby("case_id").agg(
        reference_facts=("reference_id", "count")
    ).reset_index()
)

print("\nNew proposed facts:", len(added_rows))
print("Total provisional reference facts:",
      len(four_case_reference_v4_df))
print("Independent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher

def normalize_fact(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

duplicate_rows = []

for case_id, group in four_case_reference_v4_df.groupby("case_id"):
    records = group[
        ["reference_id", "reference_fact"]
    ].to_dict("records")

    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            first = records[i]
            second = records[j]

            a = normalize_fact(first["reference_fact"])
            b = normalize_fact(second["reference_fact"])

            similarity = SequenceMatcher(
                None, a, b
            ).ratio()

            if similarity >= 0.70 or a == b:
                duplicate_rows.append({
                    "case_id": case_id,
                    "fact_id_1": first["reference_id"],
                    "fact_id_2": second["reference_id"],
                    "fact_1": first["reference_fact"],
                    "fact_2": second["reference_fact"],
                    "similarity": round(similarity, 3),
                    "review_decision": ""
                })

duplicate_review_df = pd.DataFrame(
    duplicate_rows,
    columns=[
        "case_id", "fact_id_1", "fact_id_2",
        "fact_1", "fact_2", "similarity",
        "review_decision"
    ]
).sort_values(
    ["case_id", "similarity"],
    ascending=[True, False]
)

output_path = (
    research_dir / "four_case_duplicate_review.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing duplication review preserved."
    )

duplicate_review_df.to_csv(output_path, index=False)

display(duplicate_review_df)

print("\nPotential duplicate pairs:", len(duplicate_review_df))
print("Reference facts modified: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

# Preserve all distinct clinical findings.
duplicate_decisions_df = duplicate_review_df.copy()

duplicate_decisions_df["review_decision"] = (
    "retain_distinct_clinical_concepts"
)

# Flag the two microbiology dates without reconciling them.
culture_mask = (
    (duplicate_decisions_df["case_id"] == "CASE_2148")
    & (
        (duplicate_decisions_df["fact_id_1"] == "ADD002")
        | (duplicate_decisions_df["fact_id_2"] == "ADD002")
    )
)

duplicate_decisions_df.loc[
    culture_mask, "review_decision"
] = "retain_both_dates_verify_provenance"

# Recognize the overlapping afebrile documentation.
afebrile_mask = (
    (duplicate_decisions_df["case_id"] == "CASE_2176")
    & (
        (
            (duplicate_decisions_df["fact_id_1"] == "F003_R02")
            & (duplicate_decisions_df["fact_id_2"] == "F004_C03")
        )
        | (
            (duplicate_decisions_df["fact_id_1"] == "F004_C03")
            & (duplicate_decisions_df["fact_id_2"] == "F003_R02")
        )
    )
)

assert culture_mask.sum() == 1
assert afebrile_mask.sum() == 1

duplicate_decisions_df.loc[
    afebrile_mask, "review_decision"
] = "retain_temporally_distinct_review_overlap"

# Create a revised checklist without changing original files.
four_case_reference_v5_df = four_case_reference_v4_df.copy()

target = (
    (four_case_reference_v5_df["case_id"] == "CASE_2176")
    & (
        four_case_reference_v5_df["reference_id"]
        == "F004_C03"
    )
)

assert target.sum() == 1

four_case_reference_v5_df.loc[
    target, "reference_fact"
] = "The patient remained afebrile during the hospital course."

# Keep the original exact source quotation unchanged.
output_decisions = (
    research_dir / "four_case_duplicate_decisions.csv"
)

output_reference = (
    research_dir / "four_case_working_reference_v5.csv"
)

if output_decisions.exists() or output_reference.exists():
    raise FileExistsError(
        "Existing review outputs preserved."
    )

duplicate_decisions_df.to_csv(
    output_decisions, index=False
)

four_case_reference_v5_df.to_csv(
    output_reference, index=False
)

print("Screened pairs resolved:", len(duplicate_decisions_df))
print("Reference statements clarified:", 1)
print("Reference facts excluded:", 0)
print("Total provisional facts:", len(four_case_reference_v5_df))

print("\nDecision distribution:")
print(duplicate_decisions_df["review_decision"].value_counts())

print("\nSaved:", output_decisions)
print("Saved:", output_reference)

In [ ]:
import pandas as pd

evaluation_rows = []

for case_id in remaining_case_ids:

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )

    facts = four_case_reference_v5_df.loc[
        four_case_reference_v5_df["case_id"] == case_id
    ]

    for _, fact in facts.iterrows():
        for _, note in notes.iterrows():

            evaluation_rows.append({
                "case_id": case_id,
                "reference_id": fact["reference_id"],
                "method": note["method"],
                "reference_fact": fact["reference_fact"],
                "source_evidence": fact["source_evidence"],
                "generated_note": note["generated_note"],
                "reference_status": fact["reference_status"],
                "preservation_status": "",
                "supporting_note_evidence": "",
                "review_notes": "",
                "requires_independent_review": True
            })

four_case_evaluation_df = pd.DataFrame(evaluation_rows)

assert len(four_case_evaluation_df) == 296

assert not four_case_evaluation_df.duplicated(
    ["case_id", "reference_id", "method"]
).any()

assert four_case_evaluation_df["generated_note"].notna().all()

output_path = (
    research_dir / "four_case_preservation_evaluation_v1.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing evaluation worksheet preserved."
    )

four_case_evaluation_df.to_csv(output_path, index=False)

display(
    four_case_evaluation_df.groupby(
        ["case_id", "method"]
    ).size().unstack(fill_value=0)
)

print("\nTotal comparisons:", len(four_case_evaluation_df))
print("Preservation labels assigned: 0")
print("Independent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import re
import pandas as pd

def normalize_tokens(text):
    stopwords = {
        "the", "and", "with", "was", "were", "has",
        "had", "for", "from", "that", "this", "are",
        "patient", "documented", "reported", "would",
        "will", "been", "during", "into", "after"
    }

    return {
        token
        for token in re.findall(
            r"[a-z0-9]+", str(text).lower()
        )
        if len(token) > 2 and token not in stopwords
    }

def split_note(text):
    # Preserve the generated wording.
    sentences = re.split(
        r"(?<=[.!?])\s+|\n+",
        str(text)
    )
    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

retrieval_rows = []

for _, row in four_case_evaluation_df.iterrows():

    fact_tokens = normalize_tokens(row["reference_fact"])
    note_sentences = split_note(row["generated_note"])

    scored = []

    for index, sentence in enumerate(
        note_sentences, start=1
    ):
        evidence_tokens = normalize_tokens(sentence)

        overlap = (
            len(fact_tokens & evidence_tokens)
            / max(1, len(fact_tokens))
        )

        scored.append({
            "sentence_id": f"S{index:03d}",
            "sentence": sentence,
            "overlap_score": round(overlap, 3)
        })

    scored.sort(
        key=lambda item: item["overlap_score"],
        reverse=True
    )

    top_candidates = scored[:3]

    retrieval_rows.append({
        "case_id": row["case_id"],
        "reference_id": row["reference_id"],
        "method": row["method"],
        "reference_fact": row["reference_fact"],
        "candidate_evidence_ids": ", ".join(
            item["sentence_id"]
            for item in top_candidates
            if item["overlap_score"] > 0
        ),
        "candidate_evidence": "\n".join(
            f"{item['sentence_id']}: {item['sentence']}"
            for item in top_candidates
            if item["overlap_score"] > 0
        ),
        "top_overlap_score": (
            top_candidates[0]["overlap_score"]
            if top_candidates else 0.0
        ),
        "preservation_status": "",
        "requires_independent_review": True
    })

four_case_retrieval_df = pd.DataFrame(retrieval_rows)

assert len(four_case_retrieval_df) == 296

output_path = (
    research_dir / "four_case_evidence_retrieval_v1.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing evidence retrieval file preserved."
    )

four_case_retrieval_df.to_csv(
    output_path, index=False
)

display(
    four_case_retrieval_df.groupby(
        ["case_id", "method"]
    ).agg(
        comparisons=("reference_id", "count"),
        candidate_matches=(
            "candidate_evidence_ids",
            lambda values: values.ne("").sum()
        ),
        mean_top_overlap=("top_overlap_score", "mean")
    ).reset_index().round(3)
)

print("\nTotal comparisons:", len(four_case_retrieval_df))
print("Preservation classifications assigned: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

review_columns = [
    "case_id",
    "reference_id",
    "method",
    "reference_fact",
    "source_evidence",
    "generated_note"
]

preservation_review_df = four_case_evaluation_df[
    review_columns
].merge(
    four_case_retrieval_df[
        [
            "case_id",
            "reference_id",
            "method",
            "candidate_evidence_ids",
            "candidate_evidence",
            "top_overlap_score"
        ]
    ],
    on=["case_id", "reference_id", "method"],
    how="left",
    validate="one_to_one"
)

preservation_review_df["reviewed_status"] = ""
preservation_review_df["reviewed_evidence_ids"] = ""
preservation_review_df["review_notes"] = ""
preservation_review_df["annotation_status"] = (
    "awaiting_provisional_review"
)
preservation_review_df["requires_independent_review"] = True

assert len(preservation_review_df) == 296
assert preservation_review_df["candidate_evidence"].notna().all()

output_path = (
    research_dir / "four_case_preservation_review_v1.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing preservation review worksheet preserved."
    )

preservation_review_df.to_csv(output_path, index=False)

display(
    preservation_review_df.groupby(
        ["case_id", "method"]
    ).agg(
        comparisons=("reference_id", "count"),
        mean_overlap=("top_overlap_score", "mean")
    ).reset_index().round(3)
)

print("\nReview comparisons:", len(preservation_review_df))
print("Preservation labels assigned: 0")
print("Independent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# Reconnect Google Drive
drive.mount("/content/drive")

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

# Restore the latest reference checklist
four_case_reference_v5_df = pd.read_csv(
    research_dir / "four_case_working_reference_v5.csv"
)

# Restore the full preservation review worksheet
preservation_review_df = pd.read_csv(
    research_dir / "four_case_preservation_review_v1.csv"
)

# Restore the four evaluation case IDs
remaining_case_ids = [
    "CASE_1824",
    "CASE_2043",
    "CASE_2148",
    "CASE_2176"
]

# Verify that all previously generated notes still exist
note_files = [
    research_dir / f"{case_id}_generated_notes.csv"
    for case_id in remaining_case_ids
]

assert all(path.exists() for path in note_files), (
    "One or more generated note files are missing from Drive."
)

print("Google Drive connected successfully.")
print("Reference facts restored:", len(four_case_reference_v5_df))
print("Review comparisons restored:", len(preservation_review_df))
print("Generated note files found:", len(note_files))
print("Ready to continue Step 268.")

In [ ]:
import pandas as pd

for case_id in remaining_case_ids:

    notes = pd.read_csv(
        research_dir / f"{case_id}_generated_notes.csv"
    )

    print("\n" + "=" * 75)
    print(case_id)
    print("=" * 75)

    for method in ["Simple Baseline", "Structured Prompt"]:

        note = notes.loc[
            notes["method"] == method,
            "generated_note"
        ].iloc[0]

        print(f"\n--- {method} ---")
        print(note)
        print("\nWord count:", len(note.split()))

print("\nGenerated notes displayed: 8")
print("API requests: 0")
print("Files modified: 0")

In [ ]:
import json
import pandas as pd

rubric = {
    "present": (
        "All essential components explicitly preserved, "
        "including negation, uncertainty, quantities, and timing."
    ),
    "partial": (
        "Meaningful clinical information retained, "
        "but essential specificity or components omitted."
    ),
    "missing": (
        "No meaningful preservation of the reference fact."
    ),
    "contradicted": (
        "Generated documentation explicitly conflicts "
        "with the reference fact."
    )
}

rubric_path = (
    research_dir / "four_case_preservation_rubric_v1.json"
)

if rubric_path.exists():
    raise FileExistsError(
        "Existing preservation rubric preserved."
    )

rubric_path.write_text(
    json.dumps(rubric, indent=2),
    encoding="utf-8"
)

# Restore the review worksheet from Drive.
preservation_review_df = pd.read_csv(
    research_dir / "four_case_preservation_review_v1.csv"
).fillna({
    "candidate_evidence_ids": "",
    "candidate_evidence": "",
    "reviewed_status": "",
    "reviewed_evidence_ids": "",
    "review_notes": ""
})

assert len(preservation_review_df) == 296

print("Rubric categories:", ", ".join(rubric))
print("Comparisons ready:", len(preservation_review_df))
print("Preservation labels assigned:",
      preservation_review_df["reviewed_status"].ne("").sum())
print("Saved:", rubric_path)

In [ ]:
import pandas as pd

case_1824_review_df = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_1824"
].copy()

assert len(case_1824_review_df) == 76

# Display both methods side by side for each reference fact.
comparison_matrix = case_1824_review_df.pivot(
    index=["reference_id", "reference_fact"],
    columns="method",
    values="candidate_evidence"
).reset_index()

comparison_matrix.columns.name = None

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 50)

display(comparison_matrix)

print("\nReference facts:", len(comparison_matrix))
print("Fact-note comparisons:", len(case_1824_review_df))
print("Preservation classifications assigned: 0")
print("API requests: 0")

In [ ]:
import pandas as pd

# Provisional whole-note assessments.
# P = present, R = partial, M = missing.
# These are not independently adjudicated labels.

labels = {
    "Simple Baseline": {
        "COMP001": "M", "COMP002": "M",
        "COMP003": "M", "COMP004": "M",
        "COMP005": "M",
        "F001": "P", "F002": "R",
        "F003": "R", "F004": "P",
        "F005_R01": "M", "F005_R02": "M",
        "F005_R03": "M", "F005_R04": "M",
        "F005_R05": "M",
        "F006_R01": "P", "F006_R02": "R",
        "F006_R03": "P", "F006_R04": "P",
        "F007_R01": "R", "F007_R02": "R",
        "F007_R03": "R", "F007_R04": "R",
        "F008_C01": "M", "F008_C02": "P",
        "F008_C03": "M", "F008_C04": "M",
        "F008_C05": "R", "F008_C06": "M",
        "F008_C07": "M", "F008_C08": "P",
        "F009_R01": "P", "F009_R02": "P",
        "F009_R03": "M", "F009_R04": "M",
        "F010_C01": "P", "F010_C02": "P",
        "F010_C03": "P", "F010_C04": "P"
    },
    "Structured Prompt": {
        "COMP001": "P", "COMP002": "M",
        "COMP003": "M", "COMP004": "P",
        "COMP005": "P",
        "F001": "P", "F002": "R",
        "F003": "P", "F004": "P",
        "F005_R01": "P", "F005_R02": "R",
        "F005_R03": "M", "F005_R04": "M",
        "F005_R05": "M",
        "F006_R01": "P", "F006_R02": "R",
        "F006_R03": "R", "F006_R04": "P",
        "F007_R01": "M", "F007_R02": "M",
        "F007_R03": "M", "F007_R04": "M",
        "F008_C01": "M", "F008_C02": "M",
        "F008_C03": "M", "F008_C04": "P",
        "F008_C05": "P", "F008_C06": "R",
        "F008_C07": "P", "F008_C08": "P",
        "F009_R01": "P", "F009_R02": "P",
        "F009_R03": "P", "F009_R04": "P",
        "F010_C01": "P", "F010_C02": "P",
        "F010_C03": "P", "F010_C04": "P"
    }
}

status_map = {
    "P": "present",
    "R": "partial",
    "M": "missing"
}

case_review = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_1824"
].copy()

expected_ids = set(case_review["reference_id"])

for method, method_labels in labels.items():
    assert set(method_labels) == expected_ids, (
        f"Missing or extra fact IDs for {method}"
    )

case_review["reviewed_status"] = case_review.apply(
    lambda row: status_map[
        labels[row["method"]][row["reference_id"]]
    ],
    axis=1
)

case_review["annotation_status"] = (
    "provisional_whole_note_review"
)
case_review["requires_independent_review"] = True
case_review["review_notes"] = (
    "Provisional comparison against complete note; "
    "clinical adjudication and evidence verification pending."
)

assert len(case_review) == 76
assert case_review["reviewed_status"].notna().all()

output_path = (
    research_dir / "CASE_1824_preservation_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional assessment preserved."
    )

case_review.to_csv(output_path, index=False)

print("Comparisons classified:", len(case_review))

print("\nProvisional classification distribution:")
display(
    pd.crosstab(
        case_review["method"],
        case_review["reviewed_status"]
    )
)

print("\nIndependent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

case_2043_review_df = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2043"
].copy()

assert len(case_2043_review_df) == 94

comparison_matrix_2043 = case_2043_review_df.pivot(
    index=["reference_id", "reference_fact"],
    columns="method",
    values="candidate_evidence"
).reset_index()

comparison_matrix_2043.columns.name = None

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 60)

display(comparison_matrix_2043)

print("\nReference facts:", len(comparison_matrix_2043))
print("Fact-note comparisons:", len(case_2043_review_df))
print("Preservation classifications assigned: 0")
print("API requests: 0")

In [ ]:
import pandas as pd

case_review = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2043"
].copy()

# Every unlisted reference fact is provisionally missing.
present = {
    "Simple Baseline": [
        "F001", "F002", "F004_C01", "F004_C02",
        "F006", "F017", "F022", "F025",
        "F026", "F027"
    ],
    "Structured Prompt": [
        "F001", "F002", "F003", "F004_C01",
        "F004_C02", "F006", "F008", "F017",
        "F022", "F023", "F025", "F026",
        "F027"
    ]
}

partial = {
    "Simple Baseline": [
        "F005", "F008", "F013", "F014",
        "F015", "F018", "F020", "F021",
        "F024"
    ],
    "Structured Prompt": [
        "F005", "F013", "F014", "F015",
        "F018", "F020", "F021", "F024"
    ]
}

expected_ids = set(case_review["reference_id"])

labels = {}

for method in ["Simple Baseline", "Structured Prompt"]:
    present_ids = set(present[method])
    partial_ids = set(partial[method])

    assert present_ids <= expected_ids
    assert partial_ids <= expected_ids
    assert not (present_ids & partial_ids)

    labels[method] = {
        fact_id: (
            "present" if fact_id in present_ids
            else "partial" if fact_id in partial_ids
            else "missing"
        )
        for fact_id in expected_ids
    }

case_review["reviewed_status"] = case_review.apply(
    lambda row: labels[row["method"]][row["reference_id"]],
    axis=1
)

case_review["annotation_status"] = (
    "provisional_whole_note_review"
)
case_review["requires_independent_review"] = True
case_review["review_notes"] = (
    "Provisional whole-note assessment. General normal "
    "examination statements do not establish preservation "
    "of specific findings. Independent review pending."
)

assert len(case_review) == 94
assert case_review["reviewed_status"].notna().all()
assert not case_review.duplicated(
    ["case_id", "reference_id", "method"]
).any()

output_path = (
    research_dir / "CASE_2043_preservation_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional assessment preserved."
    )

case_review.to_csv(output_path, index=False)

print("Comparisons classified:", len(case_review))

print("\nProvisional classification distribution:")
display(
    pd.crosstab(
        case_review["method"],
        case_review["reviewed_status"]
    )
)

print("\nIndependent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

case_2148_review_df = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2148"
].copy()

assert len(case_2148_review_df) == 58

comparison_matrix_2148 = case_2148_review_df.pivot(
    index=["reference_id", "reference_fact"],
    columns="method",
    values="candidate_evidence"
).reset_index()

comparison_matrix_2148.columns.name = None

pd.set_option("display.max_colwidth", 350)
pd.set_option("display.max_rows", 50)

display(comparison_matrix_2148)

print("\nReference facts:", len(comparison_matrix_2148))
print("Fact-note comparisons:", len(case_2148_review_df))
print("Preservation classifications assigned: 0")
print("API requests: 0")

In [ ]:
import pandas as pd

case_review = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2148"
].copy()

# P = present, R = partial, M = missing.
# Provisional whole-note assessments only.

labels = {
    "Simple Baseline": {
        "ADD001": "P",
        "ADD002": "R",
        "ADD003": "R",
        "ADD004": "M",
        "ADD005": "M",
        "ADD006": "M",
        "ADD007": "M",
        "ADD008": "M",
        "ADD009": "M",
        "ADD010": "R",
        "ADD011": "R",
        "ADD012": "P",
        "ADD013": "M",
        "F001": "R",
        "F002_C01": "R",
        "F002_C02": "M",
        "F002_C03": "M",
        "F003_R01": "P",
        "F004_R01": "P",
        "F004_R02": "M",
        "F005": "M",
        "F006": "M",
        "F007": "M",
        "F008_R01": "R",
        "F009": "M",
        "F010_C01": "R",
        "F010_C02": "R",
        "F010_C03": "P",
        "F010_C04": "P"
    },
    "Structured Prompt": {
        "ADD001": "P",
        "ADD002": "R",
        "ADD003": "R",
        "ADD004": "M",
        "ADD005": "M",
        "ADD006": "M",
        "ADD007": "M",
        "ADD008": "R",
        "ADD009": "R",
        "ADD010": "M",
        "ADD011": "R",
        "ADD012": "P",
        "ADD013": "M",
        "F001": "R",
        "F002_C01": "P",
        "F002_C02": "P",
        "F002_C03": "P",
        "F003_R01": "P",
        "F004_R01": "P",
        "F004_R02": "R",
        "F005": "M",
        "F006": "M",
        "F007": "R",
        "F008_R01": "R",
        "F009": "R",
        "F010_C01": "R",
        "F010_C02": "R",
        "F010_C03": "P",
        "F010_C04": "P"
    }
}

status_map = {
    "P": "present",
    "R": "partial",
    "M": "missing"
}

expected_ids = set(case_review["reference_id"])

for method, method_labels in labels.items():
    assert set(method_labels) == expected_ids, (
        f"Missing or extra reference IDs for {method}"
    )

case_review["reviewed_status"] = case_review.apply(
    lambda row: status_map[
        labels[row["method"]][row["reference_id"]]
    ],
    axis=1
)

case_review["annotation_status"] = (
    "provisional_whole_note_review"
)
case_review["requires_independent_review"] = True
case_review["review_notes"] = (
    "Provisional whole-note assessment; "
    "specific dose, timing, cultures, and treatment "
    "context require independent verification."
)

assert len(case_review) == 58
assert case_review["reviewed_status"].notna().all()

output_path = (
    research_dir / "CASE_2148_preservation_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional assessment preserved."
    )

case_review.to_csv(output_path, index=False)

print("Comparisons classified:", len(case_review))

print("\nProvisional classification distribution:")
display(
    pd.crosstab(
        case_review["method"],
        case_review["reviewed_status"]
    )
)

print("\nIndependent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

case_2176_review_df = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2176"
].copy()

assert len(case_2176_review_df) == 68

comparison_matrix_2176 = case_2176_review_df.pivot(
    index=["reference_id", "reference_fact"],
    columns="method",
    values="candidate_evidence"
).reset_index()

comparison_matrix_2176.columns.name = None

pd.set_option("display.max_colwidth", 350)
pd.set_option("display.max_rows", 50)

display(comparison_matrix_2176)

print("\nReference facts:", len(comparison_matrix_2176))
print("Fact-note comparisons:", len(case_2176_review_df))
print("Preservation classifications assigned: 0")
print("API requests: 0")

In [ ]:
import pandas as pd

case_review = preservation_review_df.loc[
    preservation_review_df["case_id"] == "CASE_2176"
].copy()

# Provisional assessments after whole-note inspection.
# Unlisted reference facts receive "missing".

present = {
    "Simple Baseline": [
        "F001", "F003_R01", "F004_C01",
        "F004_C02", "F004_C03", "F004_C04",
        "F005_C02", "F005_C03", "F006_C01",
        "F006_C02", "F006_C03", "F006_C04",
        "F009_C02"
    ],
    "Structured Prompt": [
        "F001", "F003_R01", "F004_C01",
        "F004_C03", "F004_C04", "F005_C01",
        "F005_C02", "F006_C01", "F006_C02",
        "F006_C03", "F007_R03", "F009_C01",
        "F009_C02"
    ]
}

partial = {
    "Simple Baseline": [
        "F003_R02", "F005_C04", "F007_R02",
        "F007_R03", "F008_C01", "F008_C03",
        "F008_C04"
    ],
    "Structured Prompt": [
        "F003_R02", "F004_C02", "F005_C04",
        "F007_R01", "F007_R02"
    ]
}

expected_ids = set(case_review["reference_id"])

for method in ["Simple Baseline", "Structured Prompt"]:
    assert set(present[method]) <= expected_ids
    assert set(partial[method]) <= expected_ids
    assert not (set(present[method]) & set(partial[method]))

def classify(row):
    method = row["method"]
    reference_id = row["reference_id"]

    if reference_id in present[method]:
        return "present"
    if reference_id in partial[method]:
        return "partial"
    return "missing"

case_review["reviewed_status"] = case_review.apply(
    classify, axis=1
)

case_review["annotation_status"] = (
    "provisional_whole_note_review"
)
case_review["requires_independent_review"] = True
case_review["review_notes"] = (
    "Provisional assessment using the complete generated "
    "note. Specific discharge instructions, medication "
    "details, and clinical context require independent review."
)

assert len(case_review) == 68
assert case_review["reviewed_status"].notna().all()
assert not case_review.duplicated(
    ["case_id", "reference_id", "method"]
).any()

output_path = (
    research_dir / "CASE_2176_preservation_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing provisional assessment preserved."
    )

case_review.to_csv(output_path, index=False)

print("Comparisons classified:", len(case_review))

print("\nProvisional classification distribution:")
display(
    pd.crosstab(
        case_review["method"],
        case_review["reviewed_status"]
    )
)

print("\nIndependent clinical adjudications: 0")
print("Saved:", output_path)

In [ ]:
import pandas as pd

case_ids = [
    "CASE_1824",
    "CASE_2043",
    "CASE_2148",
    "CASE_2176"
]

# Load completed provisional assessments
all_reviews = []

for case_id in case_ids:
    path = research_dir / f"{case_id}_preservation_provisional.csv"
    df = pd.read_csv(path)
    assert len(df) > 0, f"Empty review file: {case_id}"
    all_reviews.append(df)

four_case_results_df = pd.concat(
    all_reviews, ignore_index=True
)

# Validate completeness and uniqueness
assert len(four_case_results_df) == 296
assert not four_case_results_df.duplicated(
    ["case_id", "reference_id", "method"]
).any()

assert four_case_results_df["reviewed_status"].notna().all()

expected_labels = {
    "present", "partial", "missing", "contradicted"
}

assert set(four_case_results_df["reviewed_status"]) <= expected_labels

# Exploratory weights, not clinically validated
weights = {
    "present": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "contradicted": 0.0
}

metric_rows = []

for (case_id, method), group in four_case_results_df.groupby(
    ["case_id", "method"]
):
    total = len(group)
    counts = group["reviewed_status"].value_counts()

    present = int(counts.get("present", 0))
    partial = int(counts.get("partial", 0))
    missing = int(counts.get("missing", 0))
    contradicted = int(counts.get("contradicted", 0))

    metric_rows.append({
        "case_id": case_id,
        "method": method,
        "total_facts": total,
        "present": present,
        "partial": partial,
        "missing": missing,
        "contradicted": contradicted,
        "complete_preservation_rate": round(present / total, 4),
        "omission_rate": round(missing / total, 4),
        "weighted_preservation_score": round(
            group["reviewed_status"].map(weights).mean(), 4
        ),
        "evaluation_status": "provisional"
    })

four_case_metrics_df = pd.DataFrame(metric_rows)

reviews_path = (
    research_dir / "four_case_preservation_results_provisional.csv"
)
metrics_path = (
    research_dir / "four_case_preservation_metrics_provisional.csv"
)

if reviews_path.exists() or metrics_path.exists():
    raise FileExistsError(
        "Existing consolidated results preserved."
    )

four_case_results_df.to_csv(reviews_path, index=False)
four_case_metrics_df.to_csv(metrics_path, index=False)

display(
    four_case_metrics_df[
        [
            "case_id",
            "method",
            "present",
            "partial",
            "missing",
            "complete_preservation_rate",
            "omission_rate",
            "weighted_preservation_score"
        ]
    ]
)

print("\nTotal comparisons consolidated:",
      len(four_case_results_df))
print("Cases evaluated:", four_case_results_df["case_id"].nunique())
print("Independent clinical adjudications: 0")
print("Saved:", reviews_path)
print("Saved:", metrics_path)

In [ ]:
import pandas as pd

# Load the previously reviewed CASE_0811 assessments
case_0811 = pd.read_csv(
    research_dir / "CASE_0811_combined_review_provisional.csv"
)

# Retain only the shared comparison fields
columns = ["case_id", "reference_id", "method", "reviewed_status"]

other_cases = four_case_results_df[columns].copy()

# CASE_0811 uses fact_id instead of reference_id
case_0811 = case_0811.rename(
    columns={"fact_id": "reference_id"}
)
case_0811["case_id"] = "CASE_0811"

combined = pd.concat(
    [other_cases, case_0811[columns]],
    ignore_index=True
)

assert len(combined) == 336
assert combined["case_id"].nunique() == 5
assert not combined.duplicated(
    ["case_id", "reference_id", "method"]
).any()

weights = {
    "present": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "contradicted": 0.0
}

summary_rows = []

for method, group in combined.groupby("method"):
    counts = group["reviewed_status"].value_counts()

    summary_rows.append({
        "method": method,
        "total_fact_comparisons": len(group),
        "present": int(counts.get("present", 0)),
        "partial": int(counts.get("partial", 0)),
        "missing": int(counts.get("missing", 0)),
        "contradicted": int(counts.get("contradicted", 0)),
        "complete_preservation_rate": round(
            (group["reviewed_status"] == "present").mean(), 4
        ),
        "omission_rate": round(
            (group["reviewed_status"] == "missing").mean(), 4
        ),
        "weighted_preservation_score": round(
            group["reviewed_status"].map(weights).mean(), 4
        ),
        "evaluation_status": "provisional"
    })

five_case_summary_df = pd.DataFrame(summary_rows)

summary_path = (
    research_dir / "five_case_pilot_summary_provisional.csv"
)
details_path = (
    research_dir / "five_case_pilot_comparisons_provisional.csv"
)

if summary_path.exists() or details_path.exists():
    raise FileExistsError(
        "Existing five-case results preserved."
    )

five_case_summary_df.to_csv(summary_path, index=False)
combined.to_csv(details_path, index=False)

display(five_case_summary_df)

print("\nValidation cases:", combined["case_id"].nunique())
print("Total comparisons:", len(combined))
print("Independent clinical adjudications: 0")
print("Saved:", summary_path)
print("Saved:", details_path)

In [ ]:
import pandas as pd

# Restore the consolidated comparisons if needed.
combined = pd.read_csv(
    research_dir / "five_case_pilot_comparisons_provisional.csv"
)

weights = {
    "present": 1.0,
    "partial": 0.5,
    "missing": 0.0,
    "contradicted": 0.0
}

combined["preservation_weight"] = (
    combined["reviewed_status"].map(weights)
)

assert combined["preservation_weight"].notna().all()
assert len(combined) == 336

# Calculate metrics separately for each case and method.
case_metrics = (
    combined.groupby(["case_id", "method"])
    .agg(
        fact_count=("reference_id", "count"),
        complete_preservation=(
            "reviewed_status",
            lambda s: (s == "present").mean()
        ),
        omission_rate=(
            "reviewed_status",
            lambda s: (s == "missing").mean()
        ),
        weighted_preservation=(
            "preservation_weight", "mean"
        )
    )
    .reset_index()
)

# Every case receives equal weight in these averages.
balanced_metrics = (
    case_metrics.groupby("method")
    .agg(
        cases=("case_id", "nunique"),
        mean_complete_preservation=(
            "complete_preservation", "mean"
        ),
        mean_omission_rate=("omission_rate", "mean"),
        mean_weighted_preservation=(
            "weighted_preservation", "mean"
        )
    )
    .reset_index()
)

display(balanced_metrics.round(4))

# Show paired differences by case.
paired = case_metrics.pivot(
    index="case_id",
    columns="method",
    values="weighted_preservation"
)

paired["structured_minus_baseline"] = (
    paired["Structured Prompt"]
    - paired["Simple Baseline"]
)

print("\nPaired weighted-score differences:")
display(paired.round(4))

output_path = (
    research_dir / "five_case_balanced_metrics_provisional.csv"
)

if output_path.exists():
    raise FileExistsError(
        "Existing balanced metrics preserved."
    )

balanced_metrics.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("Independent clinical adjudications: 0")

In [ ]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import pandas as pd

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

final_dir = research_dir / "Final_Pilot_Results"
final_dir.mkdir(parents=True, exist_ok=True)

# Load saved results
summary = pd.read_csv(
    research_dir / "five_case_pilot_summary_provisional.csv"
)

balanced = pd.read_csv(
    research_dir / "five_case_balanced_metrics_provisional.csv"
)

comparisons = pd.read_csv(
    research_dir / "five_case_pilot_comparisons_provisional.csv"
)

four_case_metrics = pd.read_csv(
    research_dir / "four_case_preservation_metrics_provisional.csv"
)

case_0811_metrics = pd.read_csv(
    research_dir / "CASE_0811_provisional_metrics.csv"
)

case_metrics = pd.concat(
    [case_0811_metrics, four_case_metrics],
    ignore_index=True
)

assert len(comparisons) == 336
assert comparisons["case_id"].nunique() == 5
assert len(summary) == 2
assert len(balanced) == 2
assert len(case_metrics) == 10

methods = ["Simple Baseline", "Structured Prompt"]

pooled = summary.set_index("method").loc[methods]
equal_case = balanced.set_index("method").loc[methods]

def pct(value):
    return f"{100 * value:.2f}%"

def pp_difference(metric):
    return (
        100 * (
            equal_case.loc["Structured Prompt", metric]
            - equal_case.loc["Simple Baseline", metric]
        )
    )

# Prepare a concise results section for the paper
report = f"""# Clinical Auditor Agent
## Five-Case Pilot Experiment: Provisional Results

### Study scope

Five selected validation cases were evaluated using two
clinical-note generation methods: Simple Baseline and
Structured Prompt.

There were 168 reference-fact comparisons per method,
for 336 comparisons overall.

CASE_0014 was used for development and is excluded
from this five-case pilot.

### Pooled results

| Metric | Simple Baseline | Structured Prompt |
|---|---:|---:|
| Complete preservation | {pct(pooled.loc["Simple Baseline", "complete_preservation_rate"])} | {pct(pooled.loc["Structured Prompt", "complete_preservation_rate"])} |
| Omission rate | {pct(pooled.loc["Simple Baseline", "omission_rate"])} | {pct(pooled.loc["Structured Prompt", "omission_rate"])} |
| Weighted preservation | {pct(pooled.loc["Simple Baseline", "weighted_preservation_score"])} | {pct(pooled.loc["Structured Prompt", "weighted_preservation_score"])} |

### Case-balanced results

| Metric | Simple Baseline | Structured Prompt |
|---|---:|---:|
| Complete preservation | {pct(equal_case.loc["Simple Baseline", "mean_complete_preservation"])} | {pct(equal_case.loc["Structured Prompt", "mean_complete_preservation"])} |
| Omission rate | {pct(equal_case.loc["Simple Baseline", "mean_omission_rate"])} | {pct(equal_case.loc["Structured Prompt", "mean_omission_rate"])} |
| Weighted preservation | {pct(equal_case.loc["Simple Baseline", "mean_weighted_preservation"])} | {pct(equal_case.loc["Structured Prompt", "mean_weighted_preservation"])} |

### Principal findings

The Structured Prompt achieved a
{pp_difference("mean_complete_preservation"):+.2f}
percentage-point difference in case-balanced complete
preservation and a
{pp_difference("mean_weighted_preservation"):+.2f}
percentage-point difference in case-balanced weighted
preservation relative to the Simple Baseline.

The corresponding case-balanced omission-rate difference
was {pp_difference("mean_omission_rate"):+.2f}
percentage points.

The Structured Prompt had a higher weighted preservation
score in three of the five evaluated cases.

### Evaluation methodology

Clinical reference facts were extracted and refined
through source-evidence inspection and decomposition.

Generated notes were provisionally classified as:
present, partial, missing, or contradicted.

The exploratory weighted preservation metric assigned:
present = 1, partial = 0.5, missing = 0,
and contradicted = 0.

Pooled scores weight each reference fact equally.
Case-balanced scores give each selected case equal weight.

### Limitations

1. All preservation labels are provisional.
   No independent clinical adjudication was performed.

2. The pilot includes only five selected validation cases.
   The sample is too small for broad clinical conclusions.

3. Reference-fact extraction and completeness review
   were not independently validated.

4. Some reference facts may overlap or carry different
   levels of clinical importance.

5. The exploratory partial-credit weighting is not a
   clinically validated scoring system.

6. This experiment measures preservation in generated
   documentation, not the detection accuracy of the
   Clinical Auditor Agent.

7. The implemented prototype is a rule-driven
   proof of concept. The complete proposed architecture,
   including BiLSTM-CRF and semantic bipartite matching,
   has not been implemented or validated.

8. These preliminary findings do not establish
   clinical safety, statistical significance,
   or generalizability.

### Conclusion

In this small exploratory pilot, structured prompting
showed higher average documentation-preservation scores
than simple prompting. The observed differences warrant
evaluation with larger datasets, independently reviewed
reference annotations, and separate validation of
the Clinical Auditor's detection performance.

These results should be described as provisional
proof-of-concept findings.
"""

report_path = final_dir / "Pilot_Results_Report.md"
report_path.write_text(report, encoding="utf-8")

# Save clean summary tables
summary.to_csv(
    final_dir / "Pooled_Results.csv", index=False
)
balanced.to_csv(
    final_dir / "Case_Balanced_Results.csv", index=False
)
case_metrics.to_csv(
    final_dir / "Case_Level_Results.csv", index=False
)

# Include the existing evaluation evidence and source artifacts
source_files = [
    research_dir / "five_case_pilot_comparisons_provisional.csv",
    research_dir / "four_case_working_reference_v5.csv",
    research_dir / "CASE_0811_combined_review_provisional.csv",
    research_dir / "four_case_preservation_results_provisional.csv",
    research_dir / "four_case_preservation_rubric_v1.json",
]

source_files += [
    research_dir / f"{case_id}_generated_notes.csv"
    for case_id in [
        "CASE_0811", "CASE_1824",
        "CASE_2043", "CASE_2148", "CASE_2176"
    ]
]

# Some generation files may have different storage layouts.
included_sources = [p for p in source_files if p.is_file()]
missing_sources = [p.name for p in source_files if not p.is_file()]

archive_path = final_dir / "Clinical_Auditor_Pilot_Package.zip"

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for file_path in sorted(final_dir.iterdir()):
        if file_path.is_file() and file_path != archive_path:
            archive.write(file_path, arcname=file_path.name)

    for file_path in included_sources:
        archive.write(
            file_path,
            arcname=f"Supporting_Data/{file_path.name}"
        )

print("FINAL RESEARCH PACKAGE")
print("=" * 55)
print("Validation cases: 5")
print("Fact-note comparisons: 336")
print("Generated-note methods: 2")
print("Independent adjudications: 0")

print("\nSaved report:", report_path)
print("Saved archive:", archive_path)

print("\nSupporting files included:", len(included_sources))
print("Optional files not found:", missing_sources)

print("\nFINAL SUMMARY")
display(balanced.round(4))

In [ ]:
from pathlib import Path

research_dir = Path(
    "/content/drive/MyDrive/Clinical_Auditor_Research"
)

files_to_check = [
    "clinical_auditor.py",
    "Final_Pilot_Results/Pilot_Results_Report.md",
    "Final_Pilot_Results/Pooled_Results.csv",
    "Final_Pilot_Results/Case_Balanced_Results.csv",
    "Final_Pilot_Results/Case_Level_Results.csv",
]

print("GITHUB UPLOAD FILE CHECK")
print("=" * 50)

for filename in files_to_check:
    path = research_dir / filename
    if path.is_file():
        print(f"FOUND:   {filename}")
    else:
        print(f"MISSING: {filename}")

print("\nAvailable notebooks:")
notebooks = list(research_dir.rglob("*.ipynb"))

if notebooks:
    for notebook in notebooks[:20]:
        print(notebook.relative_to(research_dir))
else:
    print("No notebooks found in the research Drive folder.")
    print("Colab notebooks may be stored elsewhere in MyDrive.")

print("\nNo files uploaded to GitHub.")
print("No API requests made.")

In [ ]:

from pathlib import Path
import re

drive = Path("/content/drive/MyDrive")
research_dir = drive / "Clinical_Auditor_Research"

print("COLAB NOTEBOOKS")
print("=" * 55)

notebooks = sorted(drive.rglob("*.ipynb"))

if notebooks:
    for notebook in notebooks:
        print(notebook.relative_to(drive))
else:
    print("No notebooks found in MyDrive.")
    print("We can export them directly from Colab.")

print("\nSOURCE FILE CREDENTIAL CHECK")
print("=" * 55)

source_path = research_dir / "clinical_auditor.py"
source = source_path.read_text(
    encoding="utf-8", errors="replace"
)

patterns = {
    "OpenAI-style API key": r"sk-(?:proj-)?[A-Za-z0-9_-]{20,}",
    "Google API key": r"AIza[0-9A-Za-z_-]{30,}",
    "GitHub token": r"(?:ghp_|gho_|ghu_|ghs_|ghr_|github_pat_)[A-Za-z0-9_]+",
    "Credential assignment": (
        r"(?im)^\s*(?:api_key|access_token|secret_key|password)"
        r"\s*=\s*[\"'][^\"']+[\"']"
    ),
}

for label, pattern in patterns.items():
    count = len(re.findall(pattern, source))
    print(f"{label}: {'REVIEW REQUIRED' if count else 'Not detected'}"
          f" ({count})")

print("\nNo file contents or credentials displayed.")
print("No files modified or uploaded.")

In [ ]:

from pathlib import Path
import json
import re

notebook_dir = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

notebook_names = [
    "01_Dataset_Preprocessing.ipynb",
    "02_Baseline_Experiment.ipynb"
]

patterns = {
    "Possible OpenAI key": (
        r"sk-(?:proj-)?[A-Za-z0-9_-]{20,}"
    ),
    "Possible Google API key": (
        r"AIza[0-9A-Za-z_-]{30,}"
    ),
    "Possible GitHub token": (
        r"(?:ghp_|gho_|ghu_|ghs_|ghr_|github_pat_)"
        r"[A-Za-z0-9_]+"
    ),
    "Possible hardcoded credential": (
        r"(?im)^\s*(?:api_key|access_token|"
        r"secret_key|password)\s*=\s*"
        r"[\"'][^\"']+[\"']"
    )
}

print("NOTEBOOK PUBLICATION CHECK")
print("=" * 55)

for name in notebook_names:
    path = notebook_dir / name
    notebook = json.loads(path.read_text(encoding="utf-8"))

    cells = notebook.get("cells", [])
    code_cells = [
        cell for cell in cells
        if cell.get("cell_type") == "code"
    ]

    # Scan the entire notebook, including saved outputs.
    raw_text = path.read_text(encoding="utf-8")

    print(f"\n{name}")
    print("Code cells:", len(code_cells))
    print(
        "Cells containing saved output:",
        sum(bool(cell.get("outputs")) for cell in code_cells)
    )

    for label, pattern in patterns.items():
        count = len(re.findall(pattern, raw_text))
        print(
            f"{label}: "
            f"{'REVIEW REQUIRED' if count else 'Not detected'}"
            f" ({count})"
        )

    source_text = "\n".join(
        "".join(cell.get("source", []))
        for cell in cells
    )

    print(
        "Uses Google Drive:",
        "/content/drive/" in source_text
    )
    print(
        "Contains OpenAI API references:",
        bool(re.search(
            r"openai|OPENAI_API_KEY",
            source_text,
            re.IGNORECASE
        ))
    )

print("\nNo notebook contents displayed.")
print("No files modified or uploaded.")